In [2]:
from pathlib import Path
import os, sys, time, json, subprocess, importlib.util

RUN_NAME = "cure_brca_cbio_clam1024_v7_1"
BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK_DIR = BASE_DIR / RUN_NAME
RESUME_FROM = ""  # exact attached saved-output folder; leave blank for a fresh run
# EXPLICIT SOURCE CHANGE: this is NOT the anonymous Kaggle table.
MOLECULAR_SOURCE = "cbioportal_brca"   # "attached" keeps the previous strict CSV/mapping route
ALLOW_MOLECULAR_DOWNLOADS = True
CBIO_DUPLICATE_GENE_POLICY = "exclude_conflicting"  # explicit QC; "error" retains strict failure
CBIOPORTAL_LOCAL_ARCHIVE = ""         # optional local original study .tar.gz
CBIOPORTAL_LOCAL_DIR = ""             # optional folder containing its extracted required tables
MAX_MOLECULAR_DOWNLOAD_GIB = 8.0      # transfer/scratch guard, NOT an asserted archive size
MAX_MOLECULAR_EXTRACTED_GIB = 2.0     # only required tables are extracted
OMICS_CSV = ""                       # used ONLY in "attached" mode
ID_COLUMN = None  # automatic TCGA-barcode detection, never row-position matching
ID_MAPPING_CSV = ""  # optional real mapping with source_id,patient_id
CLINICAL_CSV = ""  # optional CSV: patient_id,time_days,event; event=1 means death
LOCAL_SVS_MANIFEST = ""  # optional CSV: patient_id,path (raw .svs, not ResNet .pt)
EXTERNAL_SPLIT_CSV = ""  # optional CSV: patient_id,fold,split
ALLOW_GDC_DOWNLOADS = True

RUN_TRAINING = True
REQUIRE_GPU = True
GPU_INDEX = 0  # uses ONE GPU; selecting T4 x2 does not enable data parallelism
FOLDS_TO_RUN = None  # None runs all five; [0] explicitly requests only fold zero
INSPECT_FOLD = 0
PATCH_POOL_PER_PATIENT = 1024
CLAM_REF = "53e2409d4a8189c682c173382964a85f114f923c"  # revision already reported as loaded in your Kaggle error log
CLAM_SOURCE_DIR = "" # optional attached official CLAM source directory; blank downloads upstream
CLAM_SEGMENTATION_OVERRIDES = {} # e.g. upstream preset keys; inspect masks before changing
SHORT_BAG_POLICY = "exclude" # "exclude" enforces exactly 1,024 unique patches; "keep" permits fewer
PATCHES_PER_PATIENT_STEP = 1024
ENCODER_MICROBATCH_SIZE = 4
PATCH_SAMPLING = "coverage_cycle"
PATCH_STORAGE_FORMAT = "jpeg"  # explicit lossy RGB cache; use "png" for lossless
JPEG_QUALITY = 95  # no chroma subsampling
PATIENT_BATCH_SIZE = 8
VALIDATION_PATCHES_PER_PATIENT = 1024
EVALUATION_CHUNK_SIZE = 4
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
LEARNING_RATE = 1e-4
WSI_AUGMENTATION = "four_views"
OMICS_AUGMENTATION = "none"  # "legacy_four_views" is an unvalidated molecular ablation
UNKNOWN_MPP_OVERRIDE = None  # only set a DOCUMENTED microns-per-pixel value
RUN_REAL_DATA_GRADIENT_CHECK = True
SHOW_REAL_INPUT_IMAGES = True
PRINT_FULL_MODEL_SUMMARY = False

SESSION_HOURS = 10.0  # planned pause, not a promise the full job finishes in one session
MAX_SAVED_OUTPUT_GIB = 16.0
MAX_TOTAL_DOWNLOAD_GIB = 1500.0  # total transfer limit, not simultaneous disk usage
SESSION_STARTED_UNIX = time.time()
DEADLINE_UNIX = SESSION_STARTED_UNIX + SESSION_HOURS*3600
PIPELINE_ACTIVE = True

if not RUN_NAME or '/' in RUN_NAME or RUN_NAME in ('.','..'):
    raise ValueError('RUN_NAME must be a directory name, not a path.')
if 'keras' in sys.modules or 'tensorflow' in sys.modules:
    raise RuntimeError('Restart the notebook session, then run from cell 1 so GPU/backend selection takes effect.')
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
print('Output directory:', WORK_DIR)
print('Requested patient batch:', PATIENT_BATCH_SIZE)
print('Requested patches per patient/update:', PATCHES_PER_PATIENT_STEP)
print('Whole-bag input rows per update:', PATIENT_BATCH_SIZE * PATCHES_PER_PATIENT_STEP)
print('Encoder rows at a time:', ENCODER_MICROBATCH_SIZE)
print('Patch cache format:', PATCH_STORAGE_FORMAT, '| quality:', JPEG_QUALITY)
print("Explicit molecular source:", MOLECULAR_SOURCE)


Output directory: /kaggle/working/cure_brca_cbio_clam1024_v7_1
Requested patient batch: 2
Requested patches per patient/update: 1024
Whole-bag input rows per update: 2048
Encoder rows at a time: 4
Patch cache format: jpeg | quality: 95
Explicit molecular source: cbioportal_brca


## 2. Check the native runtime before downloading slides
This keeps Kaggle's TensorFlow installation rather than reinstalling its CUDA stack in the running kernel.
It installs only a missing OpenSlide Python binding/native wheel. A missing TensorFlow GPU stops a training run.
For a separate local machine, install TensorFlow using its official platform instructions before opening this notebook.

In [3]:
if importlib.util.find_spec('tensorflow') is None:
    raise RuntimeError('TensorFlow is missing. Use Kaggle GPU or a correctly installed TensorFlow environment; no backend fallback.')
if importlib.util.find_spec('openslide') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openslide-python>=1.4,<2', 'openslide-bin>=4.0,<5'])
try:
    import openslide
except (ImportError, OSError) as error:
    raise RuntimeError('OpenSlide/native library import failed. Repair the environment and restart; do not replace SVS with fake patches.') from error

import tensorflow as tf
import keras
from keras import layers as L, ops
from packaging.version import Version
if Version(tf.__version__) < Version('2.16') or Version(keras.__version__) < Version('3.0'):
    raise RuntimeError('This notebook requires TensorFlow >=2.16 and Keras 3.')
if keras.backend.backend() != 'tensorflow':
    raise RuntimeError('Production training must use the TensorFlow backend.')
physical_gpus = tf.config.list_physical_devices('GPU')
if RUN_TRAINING and REQUIRE_GPU and not physical_gpus:
    raise RuntimeError('Enable a Kaggle GPU accelerator before Run All. GPU training was not started.')
for device in physical_gpus:
    tf.config.experimental.set_memory_growth(device, True)
keras.mixed_precision.set_global_policy('float32')
keras.utils.set_random_seed(2026)
if physical_gpus:
    with tf.device('/GPU:0'):
        x = tf.Variable([1.0, 2.0])
        with tf.GradientTape() as tape:
            scalar = tf.reduce_sum(x*x)
        gradient = tape.gradient(scalar, x)
        tf.debugging.assert_near(gradient, [2.0,4.0])
    print('GPU arithmetic/gradient check device:', scalar.device)
print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPU devices:', physical_gpus)
print('OpenSlide:', openslide.__version__)
for module, requirement in (("h5py","h5py>=3.10"),("tqdm","tqdm>=4.66")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install",requirement])
print("CLAM preprocessing will run on the CPU; CURE training uses one TensorFlow GPU.")


GPU arithmetic/gradient check device: /job:localhost/replica:0/task:0/device:GPU:0
TensorFlow: 2.20.0 | Keras: 3.13.2
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
OpenSlide: 1.4.6
CLAM preprocessing will run on the CPU; CURE training uses one TensorFlow GPU.


I0000 00:00:1790857787.723705      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


In [4]:
# Shared imports. There is no custom package to download or import.
import copy, gc, hashlib, math, platform, re, shutil, tempfile, traceback
from typing import Any
import numpy as np
import pandas as pd
import cv2
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from PIL import Image
from sklearn.model_selection import StratifiedKFold, train_test_split
from IPython.display import display

ORDER = ('rna', 'cnv', 'mutation')
GDC = 'https://api.gdc.cancer.gov'
register = keras.saving.register_keras_serializable

## 3. File integrity, real patient IDs and resumable execution helpers
TCGA identifiers are normalized to the patient barcode. Anonymous numeric rows are not patient IDs.
Only a planned `SessionPause` is handled as resumable progress; other errors are saved and raised.

In [5]:
def atomic_json(path: str | Path, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', dir=path.parent, delete=False, suffix='.tmp') as f:
        json.dump(obj, f, indent=2, sort_keys=True, allow_nan=False, default=lambda x: x.item() if isinstance(x, np.generic) else str(x))
        temp = f.name
    os.replace(temp, path)


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    frame.to_csv(temp, index=False)
    os.replace(temp, path)


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(2 ** 20), b''):
            h.update(b)
    return h.hexdigest()


def fingerprint(obj: Any) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()


def stable_seed(*args: Any) -> int:
    return int(fingerprint(list(args))[:8], 16)


def patient_id(value: Any) -> str:
    """Accept a real TCGA barcode; NEVER infer a barcode from a row number."""
    s = str(value).strip().upper().replace('.', '-').replace('_', '-')
    m = re.fullmatch('(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})(?:-[A-Z0-9-]+)?', s)
    if not m:
        raise ValueError(f'Not a TCGA barcode: {value!r}. Supply an authentic patient-ID mapping.')
    return m.group(1)


def sample_type(value: Any) -> str | None:
    s = str(value).strip().upper().replace('.', '-').replace('_', '-')
    parts = s.split('-')
    return parts[3][:2] if len(parts) > 3 and re.match('^\\d{2}', parts[3]) else None


def numeric_time(value: Any) -> float | None:
    try:
        x = float(value)
    except (ValueError, TypeError):
        return None
    return x if np.isfinite(x) and x > 0 else None


def require_files(paths: list[str | Path]) -> None:
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

In [6]:
class SessionPause(RuntimeError):
    """Cleanly finish this notebook version so Kaggle can save its output."""


def check_deadline(deadline_unix=None, reserve_seconds=300):
    if deadline_unix is not None and time.time() + reserve_seconds >= deadline_unix:
        raise SessionPause('Session time budget reached. Save this output and resume in a new Kaggle notebook using the saved output.')


def tree_bytes(root):
    return sum((p.stat().st_size for p in Path(root).rglob('*') if p.is_file()))


def assert_storage(root, max_output_bytes, extra_bytes=0):
    """A conservative budget, not an assertion about the user's current Kaggle quota."""
    root = Path(root)
    used = tree_bytes(root)
    if used + extra_bytes > max_output_bytes:
        raise OSError(f'Output budget exceeded: {(used + extra_bytes) / 2 ** 30:.2f} GiB > {max_output_bytes / 2 ** 30:.2f} GiB. Do not silently change the patch budget. Use a new experiment with a smaller pool or a larger supported storage workflow.')
    if shutil.disk_usage(root).free < extra_bytes + 2 ** 29:
        raise OSError('Insufficient free disk for the next operation plus a 512 MiB margin.')
    return used


def status(root, stage, state, **extra):
    report = dict(stage=stage, status=state, updated_at_unix=time.time(), **extra)
    atomic_json(Path(root) / 'RUN_STATUS.json', report)
    print(f'[{stage}] {state}', flush=True)
    return report


def restore_saved_run(previous, destination):
    """Copy a prior *saved* output into the same canonical writable directory.

    An existing writable run is never overwritten. Kaggle does not automatically
    make yesterday's unsaved scratch files available to a new version.
    """
    previous, destination = (Path(previous).resolve(), Path(destination).resolve())
    marker = previous / 'RUN_CONTRACT.json'
    if not marker.is_file():
        raise ValueError(f'{previous} is not a saved Kaggle CURE run: RUN_CONTRACT.json missing.')
    contract = json.loads(marker.read_text())
    if str(destination) != contract['work_dir']:
        raise ValueError(f"Restore to the original path {contract['work_dir']}; cached manifests use it.")
    if previous == destination or previous in destination.parents or destination in previous.parents:
        raise ValueError('Resume input and writable destination must be separate directories.')
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError('Writable run already exists. Set RESUME_FROM="" to continue it without copying.')
    required = tree_bytes(previous)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(destination.parent).free < required + 2 ** 30:
        raise OSError('Not enough free space to restore the saved run plus a 1 GiB margin.')
    temp = destination.with_name(destination.name + '.restoring')
    if temp.exists():
        raise FileExistsError(f'Partial restore found at {temp}; inspect/remove it before retrying.')
    print(f'Restoring {required / 2 ** 30:.2f} GiB from saved Kaggle output...', flush=True)
    shutil.copytree(previous, temp)
    if destination.exists():
        destination.rmdir()
    os.replace(temp, destination)


class SlideGeometryError(ValueError):
    """Missing/invalid physical image scale, not an arbitrary programming exception."""
    pass

## 4. Molecular input modes
`attached` retains the original ID-checked reader. It never assigns patient IDs by row position.
`cbioportal_brca` uses the explicitly selected cBioPortal study's own RNA, CNA, MAF and clinical files.
The new adapter is below. The two routes do not mix measurements or identities.


In [7]:
# Molecular-table intake: preserve actual identifiers; never invent patient pairing.
class OmicsInputError(ValueError):
    """The molecular table is not yet safe to pair with TCGA images."""


def read_table(path: str | Path) -> pd.DataFrame:
    # Clinical/other legacy table reader is unchanged in this revision.
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path, sep=None, engine='python', comment='#', compression='infer')


def read_omics_frame(path):
    """Read CSV/TSV/semicolon tables without losing an unnamed R row-label field.

    Preserves source IDs as strings (including leading zeros). Only leading
    comment lines are skipped. No cell content following '#' is discarded.
    Headerless and inconsistent-width files are errors, not inferred ID maps.
    """
    if Path(path).suffix.lower() == '.npz':
        return read_cbio_prepared_frame(path)
    import csv, gzip, bz2, lzma, itertools, zipfile, io, contextlib
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    opener = {'.gz': gzip.open, '.bz2': bz2.open, '.xz': lzma.open}.get(path.suffix.lower(), open)
    try:
        with contextlib.ExitStack() as stack:
            if path.suffix.lower() == '.zip':
                archive = stack.enter_context(zipfile.ZipFile(path))
                names = [n for n in archive.namelist() if not n.endswith('/')]
                if len(names) != 1 or not names[0].lower().endswith(('.csv', '.tsv', '.txt')):
                    raise OmicsInputError('A molecular ZIP must contain exactly one CSV/TSV/TXT file.')
                stream = stack.enter_context(io.TextIOWrapper(archive.open(names[0]), encoding='utf-8-sig', newline=''))
            else:
                stream = stack.enter_context(opener(path, 'rt', encoding='utf-8-sig', newline=''))
            first = None
            for line in stream:
                if line.strip() and not line.lstrip().startswith('#'):
                    first = line
                    break
            if first is None:
                raise OmicsInputError(f'Empty molecular file: {path}')
            # Pick only a delimiter that actually separates the header fields.
            widths = {d: len(next(csv.reader([first], delimiter=d))) for d in (',', '\t', ';')}
            width = max(widths.values())
            winners = [d for d, w in widths.items() if w == width]
            if width < 2 or len(winners) != 1:
                raise OmicsInputError(f'Cannot determine a unique CSV/TSV delimiter in {path.name}.')
            delimiter = winners[0]
            reader = csv.reader(itertools.chain([first], stream), delimiter=delimiter, strict=True)
            header = [s.strip().lstrip('\ufeff') for s in next(reader)]
            rows = [row for row in reader if row and any(s.strip() for s in row)]
    except (UnicodeError, csv.Error, zipfile.BadZipFile) as error:
        raise OmicsInputError(f'Molecular text parsing failed for {path.name}: {error}') from error
    if not rows:
        raise OmicsInputError(f'No patient/feature records in {path.name}.')
    row_widths = {len(row) for row in rows}
    recovered_row_label = False
    if row_widths == {len(header) + 1}:
        if all(not row[-1].strip() for row in rows):
            raise OmicsInputError('Extra blank field in every data row (possible trailing delimiter). '
                                  'Correct the CSV rather than inferring shifted feature names.')
        # R's write.table(..., row.names=TRUE, col.names=TRUE) convention.
        # This only preserves an EXISTING first field; its identity is validated below.
        header = ['__row_label__'] + header
        recovered_row_label = True
    elif row_widths != {len(header)}:
        raise OmicsInputError(
            f'Inconsistent field counts in {path.name}: header={len(header)}, rows={sorted(row_widths)}. '
            'No columns or rows were truncated.')
    header = [name if name else f'__unnamed_{i}__' for i, name in enumerate(header)]
    if len(header) != len(set(header)):
        raise OmicsInputError(f'Duplicate column names in {path.name}; resolve duplicate identifiers/features first.')
    frame = pd.DataFrame(rows, columns=header, dtype=object)
    frame.attrs['reader'] = dict(delimiter=repr(delimiter), source_shape=list(frame.shape),
                                recovered_existing_row_label=recovered_row_label)
    return frame


def canonical_tcga_barcode(value):
    if pd.isna(value):
        raise OmicsInputError('Missing TCGA barcode.')
    s = str(value).strip().upper().replace('.', '-').replace('_', '-')
    # Patient or TCGA sample/aliquot/slide barcode. Do not extract a substring
    # from an arbitrary sentence, filename or numeric sample identifier.
    if not re.fullmatch(r'TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}(?:-\d{2}[A-Z0-9]*(?:-[A-Z0-9]+)*)?', s):
        raise OmicsInputError(f'Invalid TCGA ID value {str(value)!r}; no barcode is invented.')
    return s


def _barcode_profile(values):
    values = list(values)
    valid, invalid = [], []
    for value in values:
        try:
            valid.append(canonical_tcga_barcode(value))
        except OmicsInputError:
            valid.append(None)
            if len(invalid) < 3:
                invalid.append(str(value))
    return dict(total=len(values), valid=sum(x is not None for x in valid),
                invalid_examples=invalid), valid


def inspect_omics_frame(frame, patterns):
    """Compact evidence written before any CLAM or WSI download is attempted."""
    report = dict(shape=list(frame.shape), reader=frame.attrs.get('reader', {}),
                  first_columns=list(frame.columns[:8]), last_columns=list(frame.columns[-8:]),
                  feature_counts={k: sum(bool(re.search(p, str(c))) for c in frame.columns)
                                  for k, p in patterns.items()}, id_columns=[])
    for col in frame.columns:
        values = frame[col]
        # Numeric molecular columns cannot contain TCGA barcodes; only report likely IDs.
        if any('TCGA' in str(v).upper() for v in values):
            p, _ = _barcode_profile(values)
            report['id_columns'].append(dict(column=str(col), **p))
    report['column_header_ids'] = _barcode_profile(frame.columns)[0]
    report['non_omics_columns'] = [c for c in frame.columns if not any(
        re.search(p, str(c)) for p in patterns.values()) and not str(c).startswith('pp_')][:30]
    return report


def _choose_barcode_column(frame, explicit=None):
    if explicit and explicit not in frame:
        raise OmicsInputError(f'ID column {explicit!r} missing. Available: {list(frame.columns[:25])}')
    candidates, partial = {}, []
    for col in frame.columns:
        values = frame[col]
        if col == explicit or any('TCGA' in str(v).upper() for v in values):
            profile, barcodes = _barcode_profile(values)
            if profile['valid'] == len(frame) and len(frame):
                candidates[col] = barcodes
            else:
                partial.append(dict(column=str(col), **profile))
                if col == explicit:
                    raise OmicsInputError(f'Explicit TCGA ID column is invalid: {partial[-1]}')
    if not candidates:
        raise OmicsInputError(
            'Cannot identify a complete TCGA ID column. '
            f'ID-like columns: {partial}. No anonymous row numbers are matched to slides. '
            'Supply a barcode-bearing molecular file or its authentic source_id,patient_id mapping.')
    # All complete identity columns must agree at patient level, even when the user
    # explicitly selects a column. Conflicting metadata cannot be silently ignored.
    names = list(candidates)
    patient_vectors = {c: [patient_id(x) for x in vals] for c, vals in candidates.items()}
    for col in names[1:]:
        if patient_vectors[col] != patient_vectors[names[0]]:
            raise OmicsInputError(f'Conflicting TCGA ID columns {names}; correct the source metadata.')
    for j in range(len(frame)):
        samples = {sample_type(candidates[c][j]) for c in names} - {None}
        if len(samples) > 1:
            raise OmicsInputError(f'Conflicting sample types across TCGA ID columns at row {j}.')
    # Prefer a sample-level identifier so normal-tissue rows cannot be accidentally
    # accepted after choosing a shorter patient identifier. Explicit overrides that
    # would discard known sample types do not supersede this check.
    rank = lambda c: (sum(sample_type(x) is not None for x in candidates[c]),
                      sum(len(x) for x in candidates[c]), c == explicit,
                      c in ('sample_id','SAMPLE_ID','patient_id','case_id','PATIENT_ID'))
    chosen = max(names, key=rank)
    return chosen, pd.Series(candidates[chosen], index=frame.index), partial


def _id_column(df: pd.DataFrame, explicit: str | None) -> str:
    return _choose_barcode_column(df, explicit)[0]


def _apply_authentic_id_map(raw_keys, mapping_path):
    mapping = read_omics_frame(mapping_path)
    if not {'source_id','patient_id'} <= set(mapping):
        raise OmicsInputError('ID mapping must have source_id,patient_id columns.')
    keys = mapping['source_id'].map(lambda x: str(x).strip())
    if keys.eq('').any() or keys.duplicated().any():
        raise OmicsInputError('ID mapping contains blank or duplicate source_id entries.')
    values = mapping['patient_id'].map(canonical_tcga_barcode)
    lookup = dict(zip(keys, values))
    raw_keys = raw_keys.map(lambda x: str(x).strip())
    missing = sorted(set(raw_keys) - set(lookup))
    if missing:
        raise OmicsInputError(f'ID mapping does not cover every molecular row. Missing keys: {missing[:5]}')
    return raw_keys.map(lookup)


def orient_omics_frame(frame, patterns, id_column=None, mapping_path=None):
    """Standard sample rows or explicitly evidenced transposed feature rows."""
    frame = frame.copy()
    # Supports a caller-provided DataFrame with an actual TCGA index as well.
    profile, ids = _barcode_profile(frame.index)
    if len(frame) and profile['valid'] == len(frame):
        if '__tcga_index__' in frame:
            raise OmicsInputError('Reserved __tcga_index__ column already exists.')
        frame.insert(0, '__tcga_index__', ids)
    has_feature_columns = all(any(re.search(p, str(c)) for c in frame.columns) for p in patterns.values())
    if has_feature_columns:
        if mapping_path:
            if not id_column or id_column not in frame:
                raise OmicsInputError('An id_column naming the EXISTING row key is required with id_mapping_csv.')
            raw_ids = _apply_authentic_id_map(frame[id_column], mapping_path)
            # A map must not conflict with actual barcodes already present.
            for c in frame.columns:
                p, actual = _barcode_profile(frame[c]) if any('TCGA' in str(v).upper() for v in frame[c]) else ({'valid':0}, [])
                if p['valid'] == len(frame) and len(frame):
                    if [patient_id(x) for x in actual] != [patient_id(x) for x in raw_ids]:
                        raise OmicsInputError('Supplied mapping conflicts with existing TCGA identities.')
            return frame, raw_ids, dict(layout='patients_by_features', id_column=id_column,
                                       mapping_sha256=sha256_file(mapping_path))
        chosen, raw_ids, partial = _choose_barcode_column(frame, id_column)
        return frame, raw_ids, dict(layout='patients_by_features', id_column=chosen,
                                   other_incomplete_id_like_columns=partial)

    # In a transposed table, one field holds molecular FEATURE NAMES and the
    # remaining column headers must be actual TCGA barcodes. No row-order join.
    feature_columns = []
    for c in frame.columns:
        v = frame[c].map(lambda x: str(x).strip())
        if all(v.map(lambda x: bool(re.search(p, x))).any() for p in patterns.values()):
            feature_columns.append(c)
    if len(feature_columns) != 1:
        raise OmicsInputError(
            'Cannot establish molecular-table orientation with all RNA/CNV/mutation groups. '
            f'Feature-name columns found: {feature_columns}. Inspect OMICS_PREFLIGHT.json; '
            'unknown features are not assigned to modalities by position.')
    if id_column or mapping_path:
        raise OmicsInputError('Transposed input requires actual TCGA column headers in automatic mode. '
                              'For anonymous headers, first supply an explicitly converted mapped table.')
    feature_col = feature_columns[0]
    patient_cols = [c for c in frame if c != feature_col]
    header_profile, barcodes = _barcode_profile(patient_cols)
    if not patient_cols or header_profile['valid'] != len(patient_cols):
        raise OmicsInputError(f'Transposed table has unrecognized sample headers: {header_profile}. '
                              'Cannot identify TCGA IDs; supply the authentic table/mapping.')
    features = frame[feature_col].map(lambda x: str(x).strip())
    if features.eq('').any() or features.duplicated().any():
        raise OmicsInputError('Transposed molecular feature names are blank or duplicated.')
    values = frame[patient_cols].T.copy()
    values.columns = features.tolist()
    values = values.reset_index(drop=True)
    if '__tcga_header__' in values:
        raise OmicsInputError('Reserved __tcga_header__ is a molecular column name.')
    values.insert(0, '__tcga_header__', barcodes)
    return values, values['__tcga_header__'], dict(layout='features_by_patients_transposed',
              id_column='__tcga_header__', feature_name_column=feature_col)


def load_omics(path, patterns: dict[str, str], id_column=None, mapping_path=None):
    """Validated sample-wise matrices; sources never matched by row position."""
    if Path(path).suffix.lower() == '.npz':
        return load_cbio_prepared_npz(path, patterns, id_column, mapping_path)
    for kind in ORDER:
        if kind not in patterns:
            raise OmicsInputError(f'Missing feature pattern for {kind}')
        try:
            re.compile(patterns[kind])
        except re.error as error:
            raise OmicsInputError(f'Invalid pattern for {kind}: {error}') from error
    raw = read_omics_frame(path)
    evidence = inspect_omics_frame(raw, patterns)
    df, raw_ids, identity = orient_omics_frame(raw, patterns, id_column, mapping_path)
    original_rows = len(df)
    raw_ids = raw_ids.map(canonical_tcga_barcode)
    eligible = raw_ids.map(lambda x: sample_type(x) in (None, '01'))
    excluded_samples = raw_ids.loc[~eligible].tolist()
    df, raw_ids = df.loc[eligible].copy(), raw_ids.loc[eligible]
    if df.empty:
        raise OmicsInputError('No primary-tumor/patient molecular rows remain after sample-type filtering.')
    pids = raw_ids.map(patient_id)
    blocks, used = {}, set()
    missing_tokens = {'','na','n/a','nan','null','none','not available','[not available]'}
    for kind in ORDER:
        cols = sorted(str(c) for c in df.columns if re.search(patterns[kind], str(c)))
        if not cols:
            raise OmicsInputError(f'No {kind} columns match {patterns[kind]!r}.')
        if used.intersection(cols):
            raise OmicsInputError('Feature patterns overlap between modalities.')
        used.update(cols)
        def numeric(column):
            clean = column.map(lambda x: str(x).strip() if pd.notna(x) else '')
            clean = clean.mask(clean.str.lower().isin(missing_tokens), np.nan)
            try:
                return pd.to_numeric(clean, errors='raise')
            except (ValueError, TypeError) as error:
                raise OmicsInputError(f'Non-numeric molecular values in feature {column.name!r}: {error}') from error
        block = df[cols].apply(numeric).replace([np.inf, -np.inf], np.nan)
        block.index = pids.values
        block.index.name = 'patient_id'
        blocks[kind] = block
    joined = pd.concat(blocks.values(), axis=1)
    duplicate_counts = joined.index.value_counts()
    for pid, group in joined.groupby(level=0, sort=False):
        if len(group) > 1:
            a = group.to_numpy(dtype='float64')
            if not np.array_equal(a, np.broadcast_to(a[0], a.shape), equal_nan=True):
                raise OmicsInputError(f'Conflicting molecular rows for {pid}; choose a documented '
                                       'primary-tumor sample/aliquot, not an arbitrary first row.')
    blocks = {k: v.loc[~v.index.duplicated(keep='first')].sort_index() for k, v in blocks.items()}
    before = set(blocks['rna'].index)
    complete = set.intersection(*(set(v.index[v.notna().any(axis=1)]) for v in blocks.values()))
    if not complete:
        raise OmicsInputError('No patients have measured features in all three requested molecular modalities.')
    blocks = {k: v.loc[sorted(complete)] for k, v in blocks.items()}
    audit = dict(input_rows=original_rows, non_primary_rows_excluded=len(excluded_samples),
                 excluded_non_primary_barcodes=excluded_samples,
                 identical_duplicate_rows_collapsed=int((duplicate_counts-1).clip(lower=0).sum()),
                 unique_patients_before_modality_qc=len(before),
                 patients_excluded_wholly_missing_modality=sorted(before-complete),
                 unique_patients=len(blocks['rna']), features={k: len(v.columns) for k,v in blocks.items()},
                 source_sha256=sha256_file(path), id_column=identity['id_column'],
                 omitted_columns=[str(c) for c in df.columns if c not in used],
                 channel_order=list(ORDER), source_inspection=evidence, identity=identity,
                 warning='Syntax-valid barcodes still require the later TCGA-BRCA GDC intersection. '
                         'No extra patients are created by resampling. Upstream cohort-wide preprocessing cannot be undone.')
    return blocks, audit


In [8]:
class OmicsSpatializer:
    """Fit medians/selection/means/scales on TRAIN patients only. No outcome features."""

    def __init__(self, size=128, max_features=None, interpolation=None):
        self.size = int(size)
        self.max_features = max_features
        self.interpolation = interpolation or dict(rna='linear', cnv='linear', mutation='nearest')
        self.state: dict = {}

    def fit(self, blocks, train_ids):
        if not train_ids or len(set(train_ids)) != len(train_ids):
            raise ValueError('Training IDs must be nonempty and unique.')
        self.train_ids = list(train_ids)
        for kind in ORDER:
            data = blocks[kind].loc[train_ids]
            good = data.notna().any(axis=0)
            data = data.loc[:, good]
            if data.empty:
                raise ValueError(f'{kind}: no measured features in the training fold.')
            med = data.median(axis=0)
            x = data.fillna(med)
            variance = x.var(axis=0, ddof=0)
            cols = variance[variance > 1e-12].index.tolist()
            if not cols:
                raise ValueError(f'{kind}: every training feature is constant or missing.')
            if self.max_features and len(cols) > self.max_features:
                cols = variance.loc[cols].sort_values(ascending=False, kind='stable').index[:self.max_features].tolist()
            cols = sorted(cols)
            x = x[cols].to_numpy(dtype=np.float64)
            self.state[kind] = dict(columns=cols, median=med[cols].tolist(), mean=x.mean(axis=0).tolist(), scale=np.maximum(x.std(axis=0), 1e-06).tolist())
        return self

    def transform(self, blocks, ids):
        if not self.state:
            raise RuntimeError('Call fit on training patients or load saved state before transform.')
        maps = []
        target = np.linspace(0, 1, self.size ** 2)
        for kind in ORDER:
            st = self.state[kind]
            x = blocks[kind].loc[ids, st['columns']].to_numpy(dtype=np.float64)
            med = np.asarray(st['median'])
            x = np.where(np.isfinite(x), x, med)
            x = (x - st['mean']) / st['scale']
            x = np.clip(x, -8, 8)
            source = np.linspace(0, 1, x.shape[1])
            method = self.interpolation[kind]
            if method == 'linear':
                out = np.stack([np.interp(target, source, row) for row in x])
            elif method == 'nearest':
                ix = np.rint(target * (x.shape[1] - 1)).astype(int)
                out = x[:, ix]
            else:
                raise ValueError(f'Unsupported interpolation: {method}')
            maps.append(out.reshape(len(ids), self.size, self.size))
        result = np.stack(maps, axis=-1).astype('float32')
        if not np.isfinite(result).all():
            raise FloatingPointError('Non-finite omics map.')
        return result

    def save(self, path):
        atomic_json(path, dict(size=self.size, max_features=self.max_features, interpolation=self.interpolation, train_ids=self.train_ids, state=self.state))

    @classmethod
    def load(cls, path):
        d = json.loads(Path(path).read_text())
        obj = cls(d['size'], d['max_features'], d['interpolation'])
        obj.state, obj.train_ids = (d['state'], d['train_ids'])
        return obj

In [9]:
def _omics_candidate_paths(cfg):
    """Only the explicitly selected source or the two files in the requested dataset.

    Does not search unrelated datasets or silently download a substitute cohort.
    A locked run's saved molecular snapshot takes precedence on resume.
    """
    explicit = cfg.get('omics_path')
    if explicit:
        p = Path(explicit).expanduser().resolve()
        if not p.is_file():
            raise FileNotFoundError(p)
        return [p]
    work = Path(cfg['work_dir'])
    saved = sorted(p for p in (work/'sources').glob('omics*') if p.is_file())
    if saved:
        if len(saved) != 1:
            raise OmicsInputError('Multiple saved molecular sources exist. Specify OMICS_CSV; no guessing.')
        return saved
    roots = cfg.get('omics_search_roots', ['/kaggle/input'])
    found = set()
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for name in ('data.csv','brca_data_w_subtypes.csv'):
            for path in root.rglob(name):
                if 'brca-multiomics-tcga' in str(path).lower():
                    found.add(path.resolve())
    if not found:
        raise FileNotFoundError(
            'Attach samdemharter/brca-multiomics-tcga with Add Input, or set OMICS_CSV '
            'to an explicit barcode-bearing CSV/TSV. No source was substituted.')
    return sorted(found, key=lambda p: (p.name != 'data.csv', str(p)))


def molecular_blocks_equal(first, second):
    """Exact equality of the required molecular feature matrices, independent of row order."""
    for kind in ORDER:
        a, b = first[kind], second[kind]
        if not a.index.equals(b.index) or not a.columns.equals(b.columns):
            return False
        if not np.array_equal(a.to_numpy(dtype='float64'), b.to_numpy(dtype='float64'), equal_nan=True):
            return False
    return True


def prepare_omics_input(cfg):
    """Validate identity before cloning CLAM, downloading SVS, or locking a source.

    Saves a structured report even when none of the attached sources is usable.
    Automatically accepts a companion file ONLY when that file itself contains
    valid identities and all three molecular blocks. It does NOT map rows of
    anonymous data.csv to rows of the companion file.
    """
    cfg = copy.deepcopy(cfg)
    work = Path(cfg['work_dir']); work.mkdir(parents=True, exist_ok=True)
    report = dict(status='checking', production_data_verified=False, candidates=[],
                  policy='existing TCGA IDs or explicit authentic map only; no row-order pairing',
                  target_project='TCGA-BRCA', target_project_membership='checked later against GDC')
    path = work/'OMICS_PREFLIGHT.json'
    usable = []
    try:
        candidates = _omics_candidate_paths(cfg)
        for candidate in candidates:
            item = dict(path=str(candidate), sha256=sha256_file(candidate), status='checking')
            try:
                frame = read_omics_frame(candidate)
                item['inspection'] = inspect_omics_frame(frame, cfg['feature_patterns'])
                del frame
                blocks, audit = load_omics(candidate, cfg['feature_patterns'],
                                           cfg.get('id_column'), cfg.get('id_mapping_csv'))
                item.update(status='identity_and_modalities_valid', patients=audit['unique_patients'],
                            features=audit['features'], identity=audit['identity'],
                            fingerprint= fingerprint({k: dict(index=v.index.tolist(), columns=v.columns.tolist(),
                                array_sha256=hashlib.sha256(v.to_numpy(dtype='float64').tobytes()).hexdigest())
                                for k,v in blocks.items()}))
                usable.append((candidate, blocks, audit))
            except OmicsInputError as error:
                item.update(status='not_usable_for_pairing', reason=str(error))
            report['candidates'].append(item)
            print(f"Molecular source {candidate.name}: {item['status']}")
            if 'reason' in item:
                print('  ', item['reason'])
        if not usable:
            report['status'] = 'blocked_missing_or_invalid_identity'
            raise OmicsInputError(
                'No attached molecular table passed TCGA ID and modality validation. '
                f'Read {path}. Existing IDs in unnamed columns or transposed headers are supported. '
                'If IDs were removed from the data, upload the actual source CSV and an authentic mapping; '
                'adding synthetic TCGA IDs or matching row numbers would corrupt WSI-omics pairing.')
        first_path, first_blocks, first_audit = usable[0]
        for candidate, blocks, audit in usable[1:]:
            if not molecular_blocks_equal(first_blocks, blocks):
                report['status'] = 'blocked_multiple_different_sources'
                raise OmicsInputError(
                    'Multiple barcode-bearing molecular files contain different patients/features/values. '
                    'Set OMICS_CSV to the intended source explicitly; no cohort was chosen automatically.')
        report.update(status='passed_identity_and_modalities', selected_path=str(first_path),
                      selected_sha256=sha256_file(first_path), actual_paired_cohort_not_yet_known=True,
                      equivalent_usable_paths=[str(p) for p,_,_ in usable],
                      selected_audit=first_audit,
                      note='This validates existing identifiers, layout and feature blocks, not real-SVS '
                           'decoding, TensorFlow execution, or model performance.')
        cfg['omics_path'] = str(first_path)
        cfg['omics_preflight_sha256'] = report['selected_sha256']
        print('Selected molecular input:', first_path)
        print('Layout:', first_audit['identity']['layout'], '| actual patient records:', first_audit['unique_patients'])
        print('Source feature counts:', first_audit['features'])
        return cfg
    except (OmicsInputError, FileNotFoundError) as error:
        if report['status'] == 'checking':
            report['status'] = 'blocked'
        report['error'] = str(error)
        raise
    finally:
        atomic_json(path, report)
        print('Molecular preflight report:', path)


### 4.1 Explicit cBioPortal TCGA-BRCA source adapter
These are ordinary inline functions, not an external package. The default source is
`https://datahub.assets.cbioportal.org/brca_tcga_pan_can_atlas_2018.tar.gz`.
Only the declared study and required files are accepted. Network errors do not trigger a dataset substitution.
A local copy of that study archive/folder is also supported. Keep upstream source notices and data terms.


In [10]:
# Explicit alternative data source. Does NOT recover IDs for anonymous Kaggle rows.
CBIO_STUDY = 'brca_tcga_pan_can_atlas_2018'
CBIO_ARCHIVE_URL = 'https://datahub.assets.cbioportal.org/' + CBIO_STUDY + '.tar.gz'
CBIO_REQUIRED_FILES = (
    'meta_study.txt', 'meta_mrna_seq_v2_rsem.txt', 'meta_cna.txt', 'meta_mutations.txt',
    'data_clinical_sample.txt', 'data_clinical_patient.txt',
    'data_mrna_seq_v2_rsem.txt', 'data_cna.txt', 'data_mutations.txt',
    'case_lists/cases_sequenced.txt',
)




In [11]:
def cbio_meta(path):
    """Read the publisher's key:value metadata, preserving values after ':' ."""
    out = {}
    for line in Path(path).read_text(encoding='utf-8-sig').splitlines():
        if not line.strip() or line.lstrip().startswith('#'):
            continue
        key, sep, value = line.partition(':')
        if not sep or not key.strip() or key.strip() in out:
            raise OmicsInputError(f'Malformed or duplicate metadata entry in {path}: {line[:100]}')
        out[key.strip()] = value.strip()
    return out


In [12]:
def cbio_read_tsv(path, **kwargs):
    """cBioPortal staging TSV; leading '#' records are metadata, not observations."""
    path = Path(path)
    with path.open('rb') as stream:
        prefix = stream.read(100)
    if prefix.startswith(b'version https://git-lfs.github.com/spec/'):
        raise OmicsInputError(f'{path.name} is a Git-LFS pointer, not data. Use the study archive.')
    return pd.read_csv(path, sep='\t', comment='#', low_memory=False, **kwargs)


In [13]:
def cbio_fetch_archive(url, target, max_bytes, deadline_unix=None):
    """Download only the explicitly selected source; resume with a server validator.

    No data source is substituted on an HTTP/authentication/integrity failure.
    Incomplete downloads use .part; only a complete gzip file is renamed.
    """
    from urllib.parse import urlparse
    url = str(url)
    if urlparse(url).scheme != 'https':
        raise ValueError('The public study download requires HTTPS.')
    target = Path(target); target.parent.mkdir(parents=True, exist_ok=True)
    done = target.with_name(target.name + '.download.json')
    if target.exists():
        if not done.is_file():
            raise IOError('Existing downloaded archive has no integrity record; use an explicit LOCAL_ARCHIVE instead.')
        meta = json.loads(done.read_text())
        if meta['url'] != url or meta['sha256'] != sha256_file(target):
            raise IOError('Archive source or bytes differ from the completed download record.')
        if target.stat().st_size > max_bytes:
            raise OSError('Archive exceeds the configured source-transfer budget.')
        return target
    part = target.with_name(target.name + '.part')
    record = target.with_name(target.name + '.part.json')
    prior = json.loads(record.read_text()) if record.exists() else {}
    offset = part.stat().st_size if part.exists() else 0
    validator = prior.get('etag') or prior.get('last_modified')
    headers = {'Accept-Encoding': 'identity'}
    if offset and prior.get('url') == url and validator:
        headers.update({'Range': f'bytes={offset}-', 'If-Range': validator})
    else:
        offset = 0
    check_deadline(deadline_unix)
    with session().get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        if response.status_code == 206:
            match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', response.headers.get('Content-Range', ''))
            if not match or int(match[1]) != offset:
                raise IOError('Server returned an inconsistent partial-download range.')
            total = int(match[3]); mode = 'ab'
        elif response.status_code == 200:
            offset = 0; mode = 'wb'
            length = response.headers.get('Content-Length')
            total = int(length) if length else None
        else:
            raise IOError(f'Unexpected archive HTTP status {response.status_code}')
        if total is not None and total > max_bytes:
            raise OSError('Archive exceeds MAX_MOLECULAR_DOWNLOAD_GIB; no patients were truncated.')
        needed = (total-offset) if total is not None else min(max_bytes, 2**30)
        if shutil.disk_usage(target.parent).free < needed + 2**29:
            raise OSError('Insufficient scratch storage for the molecular archive.')
        atomic_json(record, dict(url=url, etag=response.headers.get('ETag'),
                                last_modified=response.headers.get('Last-Modified'), total_bytes=total))
        read = offset
        with part.open(mode) as stream:
            for block in response.iter_content(2**20):
                check_deadline(deadline_unix)
                if not block:
                    continue
                read += len(block)
                if read > max_bytes:
                    raise OSError('Molecular download exceeded its explicit transfer budget.')
                stream.write(block)
        if total is not None and read != total:
            raise IOError(f'Incomplete archive transfer: {read} of {total} bytes.')
    with part.open('rb') as stream:
        if stream.read(2) != b'\x1f\x8b':
            raise IOError('Download is not gzip; an HTML login/error response is not patient data.')
    os.replace(part, target)
    atomic_json(done, dict(url=url, bytes=target.stat().st_size, sha256=sha256_file(target)))
    if record.exists():
        record.unlink()
    return target


In [14]:
def cbio_extract_required(archive_path, destination, max_bytes, deadline_unix=None):
    """Extract an allowlist of regular files, never arbitrary archive paths/symlinks."""
    import tarfile
    from pathlib import PurePosixPath
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    found = set(); written = 0
    with tarfile.open(archive_path, mode='r:gz') as archive:
        for member in archive:
            check_deadline(deadline_unix)
            name = member.name[2:] if member.name.startswith('./') else member.name
            parts = PurePosixPath(name).parts
            if PurePosixPath(name).is_absolute() or '..' in parts or '\\' in name:
                raise IOError('Unsafe archive path rejected.')
            if parts and parts[0] == CBIO_STUDY:
                parts = parts[1:]
            relative = '/'.join(parts)
            if relative not in CBIO_REQUIRED_FILES:
                continue
            if not member.isfile() or relative in found:
                raise IOError(f'Nonregular or duplicate required archive member: {relative}')
            written += member.size
            if written > max_bytes:
                raise OSError('Required source tables exceed the configured molecular storage allowance.')
            if shutil.disk_usage(destination).free < member.size + 2**29:
                raise OSError('Insufficient free storage to extract the next source table.')
            target = destination / relative; target.parent.mkdir(parents=True, exist_ok=True)
            temp = target.with_name(target.name + '.tmp')
            source = archive.extractfile(member)
            if source is None:
                raise IOError(f'Cannot read {relative}')
            copied = 0
            with source, temp.open('wb') as out:
                while True:
                    check_deadline(deadline_unix)
                    block = source.read(2**20)
                    if not block:
                        break
                    out.write(block); copied += len(block)
            if copied != member.size:
                raise IOError(f'Truncated archive member: {relative}')
            os.replace(temp, target); found.add(relative)
    missing = set(CBIO_REQUIRED_FILES) - found
    if missing:
        raise OmicsInputError(f'The selected study archive lacks required files: {sorted(missing)}')
    return {name: sha256_file(destination/name) for name in CBIO_REQUIRED_FILES}


In [15]:
def cbio_validate_study(folder):
    """Verify the requested study and assay meanings before reading measurements."""
    folder = Path(folder)
    missing = [name for name in CBIO_REQUIRED_FILES if not (folder/name).is_file()]
    if missing:
        raise FileNotFoundError(f'Incomplete cBioPortal study folder: {missing}')
    meta = cbio_meta(folder/'meta_study.txt')
    if meta.get('cancer_study_identifier') != CBIO_STUDY:
        raise OmicsInputError('The selected archive is not the requested TCGA-BRCA PanCancer Atlas study.')
    expected = {
        'meta_mrna_seq_v2_rsem.txt': ('MRNA_EXPRESSION', 'CONTINUOUS', 'data_mrna_seq_v2_rsem.txt'),
        'meta_cna.txt': ('COPY_NUMBER_ALTERATION', 'DISCRETE', 'data_cna.txt'),
        'meta_mutations.txt': ('MUTATION_EXTENDED', 'MAF', 'data_mutations.txt'),
    }
    for filename, (assay, dtype, datafile) in expected.items():
        info = cbio_meta(folder/filename)
        if (info.get('cancer_study_identifier'), info.get('genetic_alteration_type'),
            info.get('datatype'), info.get('data_filename')) != (CBIO_STUDY, assay, dtype, datafile):
            raise OmicsInputError(f'Unexpected assay metadata in {filename}; refusing to mislabel modalities.')
    return meta


In [16]:
def cbio_sample_map(folder):
    frame = cbio_read_tsv(Path(folder)/'data_clinical_sample.txt', dtype=str)
    if not {'PATIENT_ID', 'SAMPLE_ID'} <= set(frame):
        raise OmicsInputError('Publisher clinical sample table lacks PATIENT_ID and SAMPLE_ID.')
    rows = frame[['PATIENT_ID', 'SAMPLE_ID']].copy()
    rows['PATIENT_ID'] = rows.PATIENT_ID.map(canonical_tcga_barcode)
    rows['SAMPLE_ID'] = rows.SAMPLE_ID.map(canonical_tcga_barcode)
    if any(len(p) != 12 for p in rows.PATIENT_ID):
        raise OmicsInputError('PATIENT_ID must name a participant, not a sample/aliquot.')
    if any(patient_id(s) != p for p,s in zip(rows.PATIENT_ID, rows.SAMPLE_ID)):
        raise OmicsInputError('Publisher sample-to-patient mapping contradicts a TCGA barcode.')
    if rows.groupby('SAMPLE_ID').PATIENT_ID.nunique().gt(1).any():
        raise OmicsInputError('A sample is mapped to multiple patients.')
    if any(sample_type(s) is None for s in rows.SAMPLE_ID):
        raise OmicsInputError('SAMPLE_ID does not contain a TCGA sample type.')
    rows = rows.drop_duplicates().sort_values(['PATIENT_ID','SAMPLE_ID'])
    return dict(zip(rows.SAMPLE_ID, rows.PATIENT_ID))


In [17]:
def cbio_resolve_sample(value, mapping):
    """Use explicit mapping keys; an aliquot may extend ONE publisher sample barcode."""
    barcode = canonical_tcga_barcode(value)
    if barcode in mapping:
        return barcode
    # TCGA sample-level IDs in this release may end at type (01) or include vial (01A).
    parts = barcode.split('-')
    if len(parts) < 4:
        return None
    plausible = [s for s in mapping if barcode.startswith(s) and patient_id(s) == patient_id(barcode)]
    if len(plausible) > 1:
        # Prefer the most-specific actual publisher key, never a guessed row match.
        longest = max(map(len, plausible)); plausible = [s for s in plausible if len(s) == longest]
    return plausible[0] if len(plausible) == 1 else None


In [18]:
def cbio_gene_key(entrez, symbol):
    """Prefer stable Entrez ID; use an explicit symbol only when no ID is supplied."""
    text = '' if pd.isna(entrez) else str(entrez).strip()
    if text and text.lower() not in ('na','nan','0','0.0','-1'):
        try:
            val = float(text)
        except ValueError as error:
            raise OmicsInputError(f'Invalid Entrez identifier {text!r}') from error
        if not np.isfinite(val) or val < 1 or not val.is_integer():
            raise OmicsInputError(f'Invalid Entrez identifier {text!r}')
        return f'ENTREZ_{int(val)}'
    symbol = '' if pd.isna(symbol) else str(symbol).strip()
    return 'SYMBOL_' + symbol if symbol and symbol.lower() not in ('na','nan','?') else None


### 4.2 Duplicate genes: an explicit, audited exclusion rule

The previous reader rejected any repeated canonical key with different measurements. The replacement
below distinguishes exact redundancy from unresolved conflicting records. With `exclude_conflicting`,
all rows for an ambiguous gene key are omitted, not averaged or selected using variance. Identical copies
are collapsed once. This changes features, not the direct patient-ID selection rule. Both duplicate groups
and their original primary-sample values are saved before source-table cleanup.


In [19]:
def cbio_expression_matrix(path, mapping, kind,
                           duplicate_gene_policy='exclude_conflicting', audit_dir=None):
    """Read a cBioPortal gene-by-sample matrix without inventing merged values.

    Duplicate rule (explicit source-quality-control choice, not biological repair):
    * identical parsed float64 vectors, including matching missingness: keep one;
    * conflicting vectors: exclude the entire canonical key, or fail in 'error' mode.
    Sample columns and their publisher identities are never deduplicated here.
    No patient outcomes, training statistics, averages, or variance ranking are used.
    """
    import csv
    path = Path(path)
    if kind not in ('rna', 'cnv'):
        raise ValueError("kind must be 'rna' or 'cnv'.")
    if duplicate_gene_policy not in ('exclude_conflicting', 'error'):
        raise ValueError("duplicate_gene_policy must be 'exclude_conflicting' or 'error'.")

    # Detect repeated sample/header labels before pandas can rename them.
    with path.open(encoding='utf-8-sig', newline='') as stream:
        line = next((s for s in stream if s.strip() and not s.lstrip().startswith('#')), None)
    if line is None:
        raise OmicsInputError(f'{path.name} is empty or contains only comments.')
    if line.startswith('version https://git-lfs.github.com/spec/'):
        raise OmicsInputError(f'{path.name} is a Git-LFS pointer, not the molecular data.')
    header = next(csv.reader([line], delimiter='\t'))
    if len(header) != len(set(header)):
        raise OmicsInputError(f'Duplicate header in {path.name}')
    gene_cols = [c for c in ('Hugo_Symbol', 'Entrez_Gene_Id') if c in header]
    if not gene_cols:
        raise OmicsInputError(f'{path.name} lacks documented gene identifiers.')
    sample_cols = [c for c in header if c not in gene_cols]
    resolved = {c: cbio_resolve_sample(c, mapping) for c in sample_cols}
    primary = [c for c in sample_cols
               if resolved[c] is not None and sample_type(resolved[c]) == '01']
    if not primary:
        raise OmicsInputError(f'No primary-tumor sample columns in {path.name}')
    if len({resolved[c] for c in primary}) != len(primary):
        raise OmicsInputError('Multiple molecular columns map to one sample. Resolve aliquots explicitly.')

    # Compare in float64 before the existing float32 model-storage conversion.
    # Differences must not be hidden by an early float32 rounding operation.
    frame = cbio_read_tsv(path, usecols=gene_cols + primary,
                         dtype={**{c: str for c in gene_cols},
                                **{c: np.float64 for c in primary}})
    missing = pd.Series([None] * len(frame), index=frame.index, dtype=object)
    keys = [cbio_gene_key(e, s) for e, s in zip(
        frame.get('Entrez_Gene_Id', missing), frame.get('Hugo_Symbol', missing))]
    symbols = frame.get('Hugo_Symbol', missing).fillna('').astype(str).str.strip().tolist()
    key_array = np.asarray(keys, dtype=object)
    usable = np.asarray([k is not None for k in keys], dtype=bool)
    numeric = frame[primary].to_numpy(dtype=np.float64, copy=False)
    values = numeric[usable]
    if np.isinf(values).any():
        raise OmicsInputError(f'Infinite {kind} measurements are not allowed.')
    if kind == 'cnv' and not np.isin(values[np.isfinite(values)], [-2, -1, 0, 1, 2]).all():
        raise OmicsInputError('Expected discrete GISTIC calls -2,-1,0,1,2.')

    groups = {}
    for row, key in enumerate(keys):
        if key is not None:
            groups.setdefault(key, []).append(row)
    keep_rows, duplicate_rows, issues, excluded_keys = [], [], [], []
    annotations = {}
    actions = {}
    collapsed = 0
    for key, positions in groups.items():
        block = numeric[positions]
        equal = (block == block[0]) | (np.isnan(block) & np.isnan(block[0]))
        same = bool(equal.all())
        aliases = sorted({symbols[i] for i in positions if symbols[i]})
        if len(positions) == 1:
            keep_rows.append(positions[0])
            annotations[key] = ';'.join(aliases)
            continue
        duplicate_rows.extend(positions)
        if same:
            action = 'collapsed_identical'
            collapsed += len(positions) - 1
            keep_rows.append(positions[0])
            annotations[key] = ';'.join(aliases)
        else:
            action = ('excluded_conflicting' if duplicate_gene_policy == 'exclude_conflicting'
                      else 'rejected_conflicting')
            excluded_keys.append(key)
        for i in positions:
            actions[i] = action
        differing_columns = np.flatnonzero(~equal.all(axis=0)).tolist()
        examples = []
        for j in differing_columns[:5]:
            examples.append(dict(sample_id=resolved[primary[j]], source_column=primary[j],
                values_by_source_row=[None if np.isnan(numeric[i, j])
                                      else float(numeric[i, j]) for i in positions]))
        issues.append(dict(gene_key=key, action=action, source_data_rows_1based=[i + 1 for i in positions],
            hugo_symbols=aliases, source_row_count=len(positions),
            conflicting_sample_columns=len(differing_columns), examples=examples))

    # Audit files are produced even for strict mode, before the error is raised.
    audit = dict(source_shape=[len(frame), len(sample_cols)], primary_samples=len(primary),
        genes=len(keep_rows), skipped_no_gene_id=int((~usable).sum()),
        identical_duplicate_genes_collapsed=collapsed,
        conflicting_duplicate_gene_keys=excluded_keys,
        conflicting_duplicate_gene_keys_excluded=(len(excluded_keys)
            if duplicate_gene_policy == 'exclude_conflicting' else 0),
        conflicting_duplicate_source_rows=sum(x['source_row_count'] for x in issues
                                              if x['action'] != 'collapsed_identical'),
        duplicate_gene_policy=duplicate_gene_policy,
        duplicate_gene_details=issues,
        duplicate_row_numbering='1-based parsed data rows, excluding header and comments',
        equality_rule='exact parsed float64 equality; matching NaN locations compare equal',
        source_sha256=sha256_file(path),
        unmatched_sample_columns=[c for c in sample_cols if resolved[c] is None],
        transform=('published continuous expression values unchanged' if kind == 'rna'
                   else 'GISTIC calls unchanged'),
        numeric_storage='float32 after duplicate QC; no aggregation or relogging',
        interpretation='Conflicting gene keys are excluded, not resolved biologically. '
                       'This changes the feature universe; no patient rows are directly removed.')
    if audit_dir is not None:
        out = Path(audit_dir)
        out.mkdir(parents=True, exist_ok=True)
        rows = sorted(duplicate_rows)
        evidence = frame.iloc[rows].copy()
        evidence.insert(0, '__source_data_row_1based__', [i + 1 for i in rows])
        evidence.insert(1, '__canonical_gene_key__', [keys[i] for i in rows])
        evidence.insert(2, '__qc_action__', [actions[i] for i in rows])
        target = out / f'source_{kind}_duplicate_rows.tsv.gz'
        temporary = target.with_name(target.name + '.tmp')
        evidence.to_csv(temporary, sep='\t', index=False,
                        compression={'method': 'gzip', 'mtime': 0})
        os.replace(temporary, target)
        audit['duplicate_rows_file'] = target.name
        audit['duplicate_rows_sha256'] = sha256_file(target)
        atomic_json(out / f'source_{kind}_gene_qc.json', audit)
    print(f'[{kind} gene QC] {len(frame)} source rows -> {len(keep_rows)} unique retained keys; '
          f'{collapsed} identical extra rows collapsed; '
          f'{len(excluded_keys)} conflicting keys found; policy={duplicate_gene_policy}', flush=True)
    if excluded_keys and duplicate_gene_policy == 'error':
        raise OmicsInputError(f'Conflicting duplicate {kind} gene {excluded_keys[0]}; '
            f'{len(excluded_keys)} conflicting key(s). See the gene-QC audit; no arbitrary averaging.')
    if not keep_rows:
        raise OmicsInputError(f'No {kind} gene rows remain after documented duplicate-gene QC.')

    # Preserve sample values; only drop keys ruled out above and redundant identical rows.
    kept_values = numeric[keep_rows]
    finite = kept_values[np.isfinite(kept_values)]
    if len(finite) and np.abs(finite).max() > np.finfo(np.float32).max:
        raise OmicsInputError(f'{kind} values exceed finite float32 range.')
    kept = pd.DataFrame(kept_values.astype(np.float32).T,
                        index=[resolved[c] for c in primary],
                        columns=[keys[i] for i in keep_rows])
    kept = kept.sort_index().sort_index(axis=1)
    return kept, audit, annotations


In [20]:
def cbio_sequenced_samples(folder, mapping):
    info = cbio_meta(Path(folder)/'case_lists/cases_sequenced.txt')
    if info.get('cancer_study_identifier') != CBIO_STUDY or not info.get('case_list_ids'):
        raise OmicsInputError('Missing actual sequenced-sample list; absence of MAF rows alone is not an assay.')
    result = set()
    for raw in info['case_list_ids'].split():
        sample = cbio_resolve_sample(raw,mapping)
        if sample is None:
            raise OmicsInputError(f'Sequenced sample is absent from clinical sample map: {raw}')
        if sample_type(sample) == '01':
            result.add(sample)
    if not result:
        raise OmicsInputError('No primary tumors in the measured-mutation sample list.')
    return result


In [21]:
def cbio_mutation_matrix(path, mapping, selected_samples, genes):
    """1 = a mutation record in the published MAF; 0 = no such record in an assayed sample.

    Uses all classifications in this release, not an undisclosed mutation filter.
    Zero does not assert perfect per-locus assay coverage or clinical wild type.
    """
    header = cbio_read_tsv(path,nrows=0).columns.tolist()
    if 'Tumor_Sample_Barcode' not in header or not {'Hugo_Symbol','Entrez_Gene_Id'}.intersection(header):
        raise OmicsInputError('Mutation file is not the documented MAF schema.')
    use = [c for c in ('Tumor_Sample_Barcode','Entrez_Gene_Id','Hugo_Symbol','Variant_Classification') if c in header]
    smap = {s:i for i,s in enumerate(selected_samples)}; gmap = {g:j for j,g in enumerate(genes)}
    data = np.zeros((len(selected_samples),len(genes)),dtype=np.float32)
    aliases = {}; rows_total=rows_used=outside=0; classifications={}
    # Iterator accepts read_csv chunksize; low_memory=False remains supported.
    for frame in cbio_read_tsv(path,usecols=use,dtype=str,chunksize=10000):
        rows_total += len(frame)
        for value in frame.Tumor_Sample_Barcode.dropna().unique():
            if value not in aliases:
                aliases[value] = cbio_resolve_sample(value,mapping)
        mapped = frame.Tumor_Sample_Barcode.map(aliases)
        frame = frame.loc[mapped.isin(smap)].copy()
        frame['__sample__'] = mapped.loc[frame.index]
        for item in frame.to_dict('records'):
            key = cbio_gene_key(item.get('Entrez_Gene_Id'),item.get('Hugo_Symbol'))
            if key not in gmap:
                outside += 1; continue
            data[smap[item['__sample__']],gmap[key]]=1.0; rows_used += 1
            label = str(item.get('Variant_Classification','not_provided'))
            classifications[label]=classifications.get(label,0)+1
    return pd.DataFrame(data,index=selected_samples,columns=genes), dict(
        source_rows=rows_total, rows_in_selected_samples_and_gene_universe=rows_used,
        rows_outside_shared_gene_universe=outside, classification_counts=classifications,
        encoding='binary presence of any published MAF record within the RNA/CNV-shared gene universe',
        zero_means='no record in the released MAF for this assayed tumor sample; not proven per-gene wild type')


In [22]:
def cbio_survival(folder):
    frame = cbio_read_tsv(Path(folder)/'data_clinical_patient.txt',dtype=str)
    if not {'PATIENT_ID','OS_MONTHS','OS_STATUS'} <= set(frame):
        raise OmicsInputError('Patient file lacks PATIENT_ID, OS_MONTHS, OS_STATUS; no endpoint is fabricated.')
    rows=[]; rejected=[]
    for row in frame.to_dict('records'):
        pid = canonical_tcga_barcode(row['PATIENT_ID'])
        if len(pid)!=12:
            raise OmicsInputError('Clinical patient identifier is not participant-level.')
        raw_status=str(row['OS_STATUS']).strip().upper()
        event={'0:LIVING':0,'1:DECEASED':1}.get(raw_status)
        months=numeric_time(row['OS_MONTHS'])
        if event is None or months is None:
            rejected.append(dict(patient_id=pid,reason='missing_or_invalid_published_overall_survival'))
            continue
        rows.append(dict(patient_id=pid,time_days=months*365.25/12,event=event))
    if not rows:
        raise OmicsInputError('No valid published overall-survival observations.')
    # Conflict validation is from the supplied pipeline; do not silently merge endpoints.
    return validate_clinical(pd.DataFrame(rows)), rejected


In [23]:
def cbio_prepare_arrays(folder, output_dir, duplicate_patient_policy='exclude',
                        duplicate_gene_policy='exclude_conflicting'):
    """Create a source-identified numeric input, NOT an identity map for Kaggle rows."""
    folder=Path(folder); out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    study=cbio_validate_study(folder); mapping=cbio_sample_map(folder)
    rna, ra, annotations = cbio_expression_matrix(folder/'data_mrna_seq_v2_rsem.txt',mapping,'rna',
        duplicate_gene_policy=duplicate_gene_policy,audit_dir=out)
    cnv, ca, cn_symbols = cbio_expression_matrix(folder/'data_cna.txt',mapping,'cnv',
        duplicate_gene_policy=duplicate_gene_policy,audit_dir=out)
    sequenced=cbio_sequenced_samples(folder,mapping)
    shared=sorted(set(rna.index)&set(cnv.index)&sequenced)
    genes=sorted(set(rna.columns)&set(cnv.columns))
    # No suffixed pseudo-genes: retain the existing canonical RNA/CNV/MAF intersection.
    gene_summary = pd.DataFrame([
        dict(modality=kind, policy=a['duplicate_gene_policy'], source_rows=a['source_shape'][0],
             unique_retained_gene_keys=a['genes'],
             identical_extra_rows_collapsed=a['identical_duplicate_genes_collapsed'],
             conflicting_keys_excluded=a['conflicting_duplicate_gene_keys_excluded'],
             primary_sample_columns=a['primary_samples'], shared_gene_keys=len(genes))
        for kind, a in [('rna', ra), ('cnv', ca)]])
    atomic_csv(gene_summary, out/'source_gene_qc_summary.csv')
    if not shared or not genes:
        raise OmicsInputError('No jointly assayed primary sample/gene intersection in the selected source.')
    selected=[]; exclusions=[]
    for pid in sorted({mapping[s] for s in shared}):
        own=sorted(s for s in shared if mapping[s]==pid)
        if len(own)>1:
            if duplicate_patient_policy=='exclude':
                exclusions.append(dict(patient_id=pid,reason='multiple_jointly_assayed_primary_samples',samples=';'.join(own)))
                continue
            raise OmicsInputError(f'Multiple jointly assayed primary samples for {pid}; no arbitrary sample is selected.')
        selected.extend(own)
    if not selected:
        raise OmicsInputError('No unambiguous patients after joint sample-level matching.')
    # Whole-modality missingness must not be mistaken for a measured patient.
    valid=[s for s in selected if rna.loc[s,genes].notna().any() and cnv.loc[s,genes].notna().any()]
    for s in set(selected)-set(valid):
        exclusions.append(dict(patient_id=mapping[s],reason='wholly_missing_rna_or_cnv',samples=s))
    selected=sorted(valid,key=lambda s:mapping[s])
    if not selected:
        raise OmicsInputError('All jointly assayed candidates have a wholly missing molecular modality.')
    mut, ma=cbio_mutation_matrix(folder/'data_mutations.txt',mapping,selected,genes)
    pids=[mapping[s] for s in selected]
    payload=dict(format_version=np.asarray('cure_cbioportal_arrays_v1'),
                 study_id=np.asarray(CBIO_STUDY), patient_id=np.asarray(pids,dtype=str),
                 sample_id=np.asarray(selected,dtype=str))
    for kind,prefix,frame in [('rna','rs_',rna),('cnv','cn_',cnv),('mutation','mu_',mut)]:
        payload[kind]=frame.loc[selected,genes].to_numpy(dtype='float32')
        payload[kind+'_columns']=np.asarray([prefix+g for g in genes],dtype=str)
    target=out/'omics_cbioportal_brca.npz'; tmp=target.with_name(target.name+'.tmp')
    with tmp.open('wb') as f:
        np.savez_compressed(f,**payload)
    os.replace(tmp,target)
    clinical, clinical_exclusions=cbio_survival(folder)
    atomic_csv(clinical,out/'clinical_cbioportal_brca.csv')
    atomic_csv(pd.DataFrame(exclusions,columns=['patient_id','reason','samples']),out/'source_sample_exclusions.csv')
    atomic_csv(pd.DataFrame(clinical_exclusions,columns=['patient_id','reason']),out/'source_survival_exclusions.csv')
    atomic_csv(pd.DataFrame({'patient_id':pids,'sample_id':selected}),out/'source_sample_patient_map.csv')
    atomic_csv(pd.DataFrame({'gene_key':genes,'symbol':[annotations.get(g,cn_symbols.get(g,'')) for g in genes]}),
               out/'source_gene_annotations.csv')
    report=dict(study_id=CBIO_STUDY, study=study, molecular_patients=len(pids),
                original_matrices=dict(rna=ra,cnv=ca), mutation=ma,
                shared_gene_universe=len(genes), original_measurement_dimensions={k:list(payload[k].shape) for k in ORDER},
                source_is_NOT_the_anonymous_Kaggle_CSV=True, source_sample_join='exact publisher sample map before patient collapse',
                duplicate_patient_policy=duplicate_patient_policy, excluded_patients=exclusions,
                duplicate_gene_policy=duplicate_gene_policy,
                array_preparation_version='gene_qc_v7_1',
                gene_qc_reports=['source_rna_gene_qc.json','source_cnv_gene_qc.json','source_gene_qc_summary.csv'],
                gene_qc_note='Conflicting canonical gene keys are excluded from the shared RNA/CNV/MAF universe; '
                            'identical duplicate vectors are collapsed. No gene values are averaged. '
                            'Existing sample/outcome/WSI eligibility rules still apply.',
                clinical_patients=len(clinical), clinical_endpoint='Overall survival, months since initial diagnosis; days = months*365.25/12',
                learned_preprocessing='none at source conversion; imputation/variance filtering/scaling remain train-fold-only',
                upstream_preprocessing='published RSEM batch normalization/GISTIC/MAF; cannot undo source processing',
                mutation_classification_filter='all records in published MAF, restricted to shared gene universe',
                source_file_hashes={n:sha256_file(folder/n) for n in CBIO_REQUIRED_FILES})
    atomic_json(out/'CBIOPORTAL_SOURCE_REPORT.json',report)
    return target, out/'clinical_cbioportal_brca.csv', report


In [24]:
def load_cbio_prepared_npz(path, patterns, id_column=None, mapping_path=None):
    """Load only the adapter's non-pickle numeric schema and validate every axis."""
    if mapping_path or id_column not in (None,'patient_id','sample_id'):
        raise OmicsInputError('Prepared cBioPortal arrays already contain publisher identities; no replacement map is accepted.')
    with np.load(path,allow_pickle=False) as saved:
        needed={'format_version','study_id','patient_id','sample_id'}|set(ORDER)|{k+'_columns' for k in ORDER}
        if not needed <= set(saved.files) or str(saved['format_version'])!='cure_cbioportal_arrays_v1' or str(saved['study_id'])!=CBIO_STUDY:
            raise OmicsInputError('Unknown or incomplete prepared molecular NPZ schema.')
        pids=[canonical_tcga_barcode(p) for p in saved['patient_id'].tolist()]
        samples=[canonical_tcga_barcode(s) for s in saved['sample_id'].tolist()]
        if not pids or len(pids)!=len(set(pids)) or len(samples)!=len(pids):
            raise OmicsInputError('Prepared patient/sample axes are empty, duplicated or inconsistent.')
        if any(len(p)!=12 or patient_id(s)!=p or sample_type(s)!='01' for p,s in zip(pids,samples)):
            raise OmicsInputError('Prepared TCGA sample/patient correspondence failed.')
        blocks={}
        for kind in ORDER:
            cols=saved[kind+'_columns'].astype(str).tolist(); x=saved[kind]
            if not cols or len(cols)!=len(set(cols)) or x.shape!=(len(pids),len(cols)):
                raise OmicsInputError(f'Invalid {kind} feature/sample dimensions.')
            if kind not in patterns or not all(re.search(patterns[kind],c) for c in cols):
                raise OmicsInputError(f'Prepared {kind} names do not match the requested feature definition.')
            if x.dtype.kind not in 'fiu' or np.isinf(x).any() or not np.isfinite(x).any(axis=1).all():
                raise OmicsInputError(f'Invalid {kind} numeric matrix or unmeasured patient.')
            blocks[kind]=pd.DataFrame(x,index=pd.Index(pids,name='patient_id'),columns=cols).sort_index().sort_index(axis=1)
    audit=dict(input_rows=len(pids),unique_patients=len(pids),id_column='patient_id',
               identity=dict(layout='publisher_sample_matched_numeric_arrays',sample_ids_preserved=True),
               features={k:v.shape[1] for k,v in blocks.items()},source_sha256=sha256_file(path),channel_order=list(ORDER),
               omitted_columns=['protein_not_included','clinical_outcomes_not_features'],
               warning='Explicit cBioPortal TCGA-BRCA source, not repaired Kaggle row identities; WSI intersection is still required.')
    return blocks,audit


In [25]:
def read_cbio_prepared_frame(path):
    blocks,_=load_cbio_prepared_npz(path,{'rna':'^rs_','cnv':'^cn_','mutation':'^mu_'})
    return pd.concat(list(blocks.values()),axis=1).reset_index()


In [26]:
def prepare_selected_molecular_source(cfg, deadline_unix=None):
    """EXPLICIT source switch, never a fallback triggered by an invalid Kaggle CSV."""
    cfg=copy.deepcopy(cfg); mode=cfg.get('molecular_source_mode','attached')
    if mode=='attached':
        print('SOURCE: user-attached molecular table. No alternative is substituted on failure.')
        return prepare_omics_input(cfg)
    if mode!='cbioportal_brca':
        raise ValueError('MOLECULAR_SOURCE must be attached or cbioportal_brca.')
    if cfg.get('omics_path') or cfg.get('id_mapping_csv') or cfg.get('id_column'):
        raise ValueError('In cbioportal_brca mode leave OMICS_CSV/ID_COLUMN/ID_MAPPING_CSV empty. Source IDs are supplied by the study.')
    print('EXPLICIT SOURCE CHANGE: TCGA-BRCA PanCancer Atlas via cBioPortal; attached Kaggle CSVs are NOT used.')
    print('This may change the cohort, features, and endpoint release. It does NOT recover missing Kaggle row IDs.')
    work=Path(cfg['work_dir']); source=work/'sources'/'cbioportal_brca'; source.mkdir(parents=True,exist_ok=True)
    scfg=cfg['cbioportal']; lock=source/'SOURCE_LOCK.json'
    gene_policy=scfg.get('duplicate_gene_policy','exclude_conflicting')
    if gene_policy not in ('exclude_conflicting','error'):
        raise ValueError('Unknown cBioPortal duplicate_gene_policy.')
    source_contract={k:scfg.get(k) for k in ('study_id','url','local_archive','local_dir','duplicate_patient_policy')}
    source_contract.update(duplicate_gene_policy=gene_policy,array_preparation_version='gene_qc_v7_1')
    report_path=work/'OMICS_PREFLIGHT.json'
    if scfg.get('study_id')!=CBIO_STUDY:
        raise OmicsInputError('This adapter implements only the declared TCGA-BRCA study.')
    try:
        if lock.exists():
            recorded=json.loads(lock.read_text())
            if recorded['source_contract']!=source_contract:
                raise ValueError('cBioPortal source selection changed. Use a NEW RUN_NAME.')
            for rel,digest in recorded['output_hashes'].items():
                file=source/rel
                if not file.is_file() or sha256_file(file)!=digest:
                    raise IOError(f'Missing/modified prepared cBioPortal source: {rel}')
            omics=source/'omics_cbioportal_brca.npz'; clinical=source/'clinical_cbioportal_brca.csv'
            report=json.loads((source/'CBIOPORTAL_SOURCE_REPORT.json').read_text())
        else:
            archive=None; owned=False
            if scfg.get('local_dir'):
                if scfg.get('local_archive'):
                    raise ValueError('Choose a local study folder OR local archive, not both.')
                folder=Path(scfg['local_dir']).expanduser().resolve(); cbio_validate_study(folder)
            else:
                if scfg.get('local_archive'):
                    archive=Path(scfg['local_archive']).expanduser().resolve()
                    if not archive.is_file():
                        raise FileNotFoundError(archive)
                else:
                    if not scfg.get('allow_download'):
                        raise PermissionError('Approve the explicit cBioPortal source download or provide its local archive/folder.')
                    scratch=Path(cfg['kaggle']['scratch_dir']).parent/'molecular_scratch';scratch.mkdir(parents=True,exist_ok=True)
                    archive=cbio_fetch_archive(scfg['url'],scratch/(CBIO_STUDY+'.tar.gz'),
                          int(scfg['max_download_gib']*2**30),deadline_unix);owned=True
                folder=source/'raw'
                cbio_extract_required(archive,folder,int(scfg['max_extracted_gib']*2**30),deadline_unix)
                cbio_validate_study(folder)
            check_deadline(deadline_unix)
            omics,clinical,report=cbio_prepare_arrays(folder,source,scfg['duplicate_patient_policy'],
                duplicate_gene_policy=gene_policy)
            outputs=['omics_cbioportal_brca.npz','clinical_cbioportal_brca.csv','CBIOPORTAL_SOURCE_REPORT.json',
                     'source_sample_exclusions.csv','source_survival_exclusions.csv','source_sample_patient_map.csv','source_gene_annotations.csv',
                     'source_rna_gene_qc.json','source_cnv_gene_qc.json','source_gene_qc_summary.csv',
                     'source_rna_duplicate_rows.tsv.gz','source_cnv_duplicate_rows.tsv.gz']
            atomic_json(lock,dict(source_contract=source_contract,downloaded_at_unix=time.time(),
                archive_sha256=sha256_file(archive) if archive else None,
                output_hashes={rel:sha256_file(source/rel) for rel in outputs},
                checksum_note='Archive digest is recorded after HTTPS download, not claimed as an independent publisher checksum.'))
            # Delete only raw files created by this adapter, after committed verified outputs.
            if not scfg.get('local_dir') and not scfg.get('keep_raw_tables',False):
                shutil.rmtree(source/'raw')
            if owned and archive is not None:
                archive.unlink()
                record=archive.with_name(archive.name+'.download.json')
                if record.exists(): record.unlink()
        cfg['omics_path']=str(omics.resolve());cfg['id_column']='patient_id';cfg['id_mapping_csv']=None
        cfg['omics_source']='cbioportal_brca';cfg['feature_patterns']={'rna':'^rs_','cnv':'^cn_','mutation':'^mu_'}
        # User-supplied clinical endpoint remains explicit and is never overwritten.
        if not cfg.get('clinical_csv'):
            cfg['clinical_csv']=str(clinical.resolve())
            cfg['clinical_columns']={'patient':'patient_id','time':'time_days','event':'event','time_unit':'days'}
        blocks,audit=load_cbio_prepared_npz(omics,cfg['feature_patterns'])
        atomic_json(report_path,dict(status='passed_identity_and_modalities',source_mode=mode,
                    source_change='cBioPortal TCGA-BRCA; NOT the original unidentified Kaggle rows',
                    selected_path=str(omics),selected_audit=audit,source_report=report,
                    real_WSI_and_training_not_yet_validated=True))
        print('Publisher-identified molecular matrices:',{k:v.shape for k,v in blocks.items()})
        return cfg
    except Exception as error:
        atomic_json(report_path,dict(status='failed',source_mode=mode,error_type=type(error).__name__,error=str(error),
                    no_fallback_to_other_data=True))
        raise


## 5. Match public TCGA-BRCA clinical records and raw diagnostic SVS files
Downloads use open-access GDC file IDs and checksums. Clinical outcomes are positive overall-survival times
with known vital status. Unknown status is excluded rather than assumed alive. Network failures remain errors.
The source-derived normalizer uses patient-level matching; it does not claim spatial transcriptomics or identical aliquots.

In [27]:
def session():
    s = requests.Session()
    retry = Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET', 'POST'])
    s.mount('https://', HTTPAdapter(max_retries=retry))
    s.headers['User-Agent'] = 'CURE-BRCA-research-pipeline/0.1'
    return s


def query_all(endpoint, filters, fields, cache_path, refresh=False):
    cache_path = Path(cache_path)
    signature = {'endpoint': endpoint, 'filters': filters, 'fields': fields}
    if cache_path.exists() and (not refresh):
        cached = json.loads(cache_path.read_text())
        if cached['query'] != signature:
            raise ValueError(f'Cached query differs: {cache_path}. Use refresh=True or a new work directory.')
        return cached['hits']
    s, hits, offset = (session(), [], 0)
    while True:
        r = s.get(f'{GDC}/{endpoint}', params=dict(filters=json.dumps(filters), fields=fields, size=500, **{'from': offset}, format='JSON'), timeout=(30, 180))
        r.raise_for_status()
        obj = r.json()
        if 'data' not in obj or 'hits' not in obj['data']:
            raise RuntimeError(f'Unexpected GDC response from {endpoint}: {str(obj)[:500]}')
        batch = obj['data']['hits']
        hits.extend(batch)
        total = int(obj['data']['pagination']['total'])
        offset += len(batch)
        if offset >= total:
            break
        if not batch:
            raise RuntimeError('GDC returned an empty page before the reported total.')
    atomic_json(cache_path, dict(query=signature, retrieved_unix=time.time(), hits=hits))
    return hits

In [28]:
def load_clinical_csv(path, columns):
    df = read_table(path)
    p, t, e = (columns[k] for k in ('patient', 'time', 'event'))
    if not {p, t, e} <= set(df):
        raise ValueError(f'Clinical file needs {p},{t},{e}; available: {list(df.columns)}')
    unit = columns.get('time_unit', 'days')
    factors = {'days': 1.0, 'months': 365.25 / 12, 'years': 365.25}
    if unit not in factors:
        raise ValueError('time_unit must be days, months or years.')

    def event_value(x):
        s = str(x).strip().upper()
        if s in ('1', '1.0', 'DEAD', 'DECEASED', '1:DECEASED'):
            return 1
        if s in ('0', '0.0', 'ALIVE', 'LIVING', '0:LIVING', '0:ALIVE'):
            return 0
        return None
    records, excluded = ([], [])
    for _, row in df.iterrows():
        pid = patient_id(row[p])
        duration, event = (numeric_time(row[t]), event_value(row[e]))
        if duration is None or event is None:
            excluded.append(dict(patient_id=pid, reason='invalid_or_missing_OS_duration_or_status'))
        else:
            records.append(dict(patient_id=pid, time_days=duration * factors[unit], event=event))
    return (validate_clinical(pd.DataFrame(records)), excluded)


def validate_clinical(df):
    if df.empty or not {'patient_id', 'time_days', 'event'} <= set(df):
        raise ValueError('No valid survival records with patient_id,time_days,event.')
    if not np.isfinite(df.time_days).all() or (df.time_days <= 0).any() or (not df.event.isin([0, 1]).all()):
        raise ValueError('Survival requires positive finite durations and event=0/1 (1=death).')
    for pid, g in df.groupby('patient_id'):
        if g.event.nunique() != 1 or g.time_days.nunique() != 1:
            raise ValueError(f'Conflicting survival outcomes for {pid}; resolve source records explicitly.')
    return df.drop_duplicates('patient_id').sort_values('patient_id').reset_index(drop=True)


def gdc_clinical(cache_dir):
    filters = {'op': 'in', 'content': {'field': 'project.project_id', 'value': ['TCGA-BRCA']}}
    fields = ','.join(['case_id', 'submitter_id', 'demographic.vital_status', 'demographic.days_to_death', 'diagnoses.days_to_last_follow_up', 'follow_ups.days_to_follow_up', 'index_date'])
    hits = query_all('cases', filters, fields, Path(cache_dir) / 'gdc_cases.json')
    records, excluded = ([], [])
    for case in hits:
        pid = patient_id(case['submitter_id'])
        dem = case.get('demographic') or {}
        if isinstance(dem, list):
            if len(dem) != 1:
                excluded.append(dict(patient_id=pid, reason='ambiguous_demographic_record'))
                continue
            dem = dem[0]
        status = str(dem.get('vital_status', '')).lower()
        if status == 'dead':
            duration = numeric_time(dem.get('days_to_death'))
            event = 1
        elif status == 'alive':
            follow = []
            for diag in case.get('diagnoses') or []:
                follow.append(numeric_time(diag.get('days_to_last_follow_up')))
            for visit in case.get('follow_ups') or []:
                follow.append(numeric_time(visit.get('days_to_follow_up')))
            follow = [x for x in follow if x is not None]
            duration = max(follow) if follow else None
            event = 0
        else:
            excluded.append(dict(patient_id=pid, reason='unknown_vital_status'))
            continue
        if duration is None:
            excluded.append(dict(patient_id=pid, reason='missing_positive_survival_time'))
            continue
        records.append(dict(patient_id=pid, time_days=duration, event=event, time_origin=str(case.get('index_date', 'not_reported'))))
    return (validate_clinical(pd.DataFrame(records)), excluded)

In [29]:
def gdc_slides(cache_dir, diagnostic_only=True):
    filters = {'op': 'and', 'content': [{'op': 'in', 'content': {'field': 'cases.project.project_id', 'value': ['TCGA-BRCA']}}, {'op': 'in', 'content': {'field': 'data_type', 'value': ['Slide Image']}}, {'op': 'in', 'content': {'field': 'data_format', 'value': ['SVS']}}, {'op': 'in', 'content': {'field': 'access', 'value': ['open']}}]}
    fields = 'file_id,file_name,file_size,md5sum,cases.submitter_id,cases.case_id'
    hits = query_all('files', filters, fields, Path(cache_dir) / 'gdc_slides.json')
    records = []
    for h in hits:
        name = h['file_name']
        if diagnostic_only and '-DX' not in name.upper():
            continue
        cases = h.get('cases', [])
        if len(cases) != 1:
            raise ValueError(f'Slide {name} has {len(cases)} associated cases, not one.')
        pid = patient_id(cases[0]['submitter_id'])
        match = re.search('TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}', name.upper())
        if match and patient_id(match.group()) != pid:
            raise ValueError(f'Slide barcode/case mismatch: {name} vs {pid}')
        records.append(dict(patient_id=pid, file_id=h['file_id'], file_name=name, file_size=int(h['file_size']), md5sum=h['md5sum']))
    if not records:
        raise RuntimeError('No eligible TCGA-BRCA diagnostic SVS files returned by GDC.')
    return pd.DataFrame(records).sort_values(['patient_id', 'file_name']).reset_index(drop=True)


def locate_omics(cfg):
    if cfg.get('omics_path'):
        return Path(cfg['omics_path']).expanduser().resolve()
    work = Path(cfg['work_dir'])
    if cfg['omics_source'] == 'kaggle':
        try:
            import kagglehub
        except ImportError as ex:
            raise ImportError('Install kagglehub or set omics_path to the downloaded CSV.') from ex
        root = Path(kagglehub.dataset_download('samdemharter/brca-multiomics-tcga'))
        candidates = sorted(root.rglob('data.csv'))
        if len(candidates) != 1:
            raise ValueError(f'Expected one data.csv, found {len(candidates)} in {root}. Set omics_path explicitly.')
        return candidates[0]
    if cfg['omics_source'] == 'healnet':
        p = work / 'downloads' / 'tcga_brca_all_clean.csv.zip'
        p.parent.mkdir(parents=True, exist_ok=True)
        if not p.exists():
            url = 'https://media.githubusercontent.com/media/konst-int-i/healnet/main/data/tcga/omic/tcga_brca_all_clean.csv.zip'
            r = session().get(url, timeout=(30, 300))
            r.raise_for_status()
            if not r.content.startswith(b'PK'):
                raise ValueError('HEALNet download is not a ZIP (possibly a Git-LFS pointer). Obtain the real Git-LFS archive and set omics_path.')
            tmp = p.with_suffix('.part')
            tmp.write_bytes(r.content)
            os.replace(tmp, p)
        return p.resolve()
    raise ValueError('omics_source must be kaggle or healnet, or provide a local omics_path.')


def download_slide(row, out_dir, token=None, deadline_unix=None):
    check_deadline(deadline_unix)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir / str(row.file_id) / Path(row.file_name).name
    dest.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(row.file_size)

    def md5(path):
        h = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(2 ** 20), b''):
                h.update(chunk)
        return h.hexdigest()
    if dest.exists():
        if dest.stat().st_size == expected_size and md5(dest) == row.md5sum:
            return str(dest.resolve())
        raise IOError(f'Existing slide failed size/MD5 check: {dest}; inspect/remove it explicitly.')
    part = dest.with_suffix(dest.suffix + '.part')
    start = part.stat().st_size if part.exists() else 0
    if start > expected_size:
        raise IOError(f'Oversized partial download: {part}')
    if start < expected_size:
        headers = {'Range': f'bytes={start}-'} if start else {}
        if token:
            headers['X-Auth-Token'] = token
        with session().get(f'{GDC}/data/{row.file_id}', headers=headers, stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            if start and r.status_code == 206:
                if not r.headers.get('Content-Range', '').startswith(f'bytes {start}-'):
                    raise IOError('Unexpected HTTP Content-Range during resume.')
                mode = 'ab'
            else:
                mode = 'wb'
            with open(part, mode) as f:
                for chunk in r.iter_content(2 ** 20):
                    check_deadline(deadline_unix)
                    if chunk:
                        f.write(chunk)
    if part.stat().st_size != expected_size or md5(part) != row.md5sum:
        raise IOError(f'Download integrity failure for {row.file_name}; partial retained for inspection.')
    os.replace(part, dest)
    return str(dest.resolve())

In [30]:
def prepare_metadata(cfg):
    work = Path(cfg['work_dir'])
    work.mkdir(parents=True, exist_ok=True)
    omics_path = locate_omics(cfg)
    blocks, omics_audit = load_omics(omics_path, cfg['feature_patterns'], cfg.get('id_column'), cfg.get('id_mapping_csv'))
    cfg = dict(cfg, omics_path=str(omics_path))
    if cfg.get('clinical_csv'):
        clinical, exclusions = load_clinical_csv(cfg['clinical_csv'], cfg['clinical_columns'])
    elif cfg['omics_source'] == 'healnet':
        raw = read_table(omics_path)
        need = {'case_id', 'survival_months', 'censorship'}
        if not need <= set(raw):
            raise ValueError('HEALNet table lacks case_id,survival_months,censorship. Supply clinical_csv.')
        records, exclusions = ([], [])
        for _, row in raw.iterrows():
            pid, t = (patient_id(row.case_id), numeric_time(row.survival_months))
            c = pd.to_numeric(row.censorship, errors='coerce')
            if t is None or c not in (0, 1):
                exclusions.append(dict(patient_id=pid, reason='invalid_HEALNet_OS_record'))
                continue
            records.append(dict(patient_id=pid, time_days=t * 365.25 / 12, event=int(1 - c)))
        clinical = validate_clinical(pd.DataFrame(records))
    else:
        clinical, exclusions = gdc_clinical(work / 'metadata')
    if cfg.get('local_slides_manifest'):
        slides = pd.read_csv(cfg['local_slides_manifest'], dtype={'patient_id': str})
        if not {'patient_id', 'path'} <= set(slides):
            raise ValueError('Local slide manifest requires patient_id,path columns (one row per SVS).')
        slides.patient_id = slides.patient_id.map(patient_id)
        slides['path'] = slides.path.map(lambda p: str(Path(p).expanduser().resolve()))
        for row in slides.itertuples():
            p = row.path
            if not Path(p).is_file():
                raise FileNotFoundError(p)
            match = re.search('TCGA[-_.][A-Z0-9]{2}[-_.][A-Z0-9]{4}', Path(p).name.upper())
            if match and patient_id(match.group()) != row.patient_id:
                raise ValueError(f'Local filename/patient mismatch: {p} vs {row.patient_id}')
            if Path(p).suffix.lower() == '.pt':
                raise ValueError('.pt feature vectors are not RGB WSI images and cannot be inverted by reshaping.')
    else:
        slides = gdc_slides(work / 'metadata', cfg['diagnostic_slides_only'])
        slides['path'] = [str((work / 'slides' / r.file_id / r.file_name).resolve()) for r in slides.itertuples()]
    omic_ids = set(blocks['rna'].index)
    all_ids = omic_ids | set(clinical.patient_id) | set(slides.patient_id)
    paired = omic_ids & set(clinical.patient_id) & set(slides.patient_id)
    for pid in sorted(all_ids - paired):
        missing = [name for name, ids in [('omics', omic_ids), ('survival', set(clinical.patient_id)), ('WSI', set(slides.patient_id))] if pid not in ids]
        exclusions.append(dict(patient_id=pid, reason='missing_' + '_'.join(missing)))
    if not paired:
        raise ValueError('No patients have all three: molecular data, survival labels, and eligible WSI.')
    cohort = clinical[clinical.patient_id.isin(paired)].sort_values('patient_id').reset_index(drop=True)
    slides = slides[slides.patient_id.isin(paired)].sort_values(['patient_id', 'path']).reset_index(drop=True)
    cohort.to_csv(work / 'cohort_before_patch_qc.csv', index=False)
    slides.to_csv(work / 'slides.csv', index=False)
    pd.DataFrame(exclusions, columns=['patient_id', 'reason']).to_csv(work / 'exclusions_metadata.csv', index=False)
    atomic_json(work / 'omics_audit.json', omics_audit)
    atomic_json(work / 'resolved_config.json', cfg)
    print(f'Metadata match: {len(cohort)} patients, {len(slides)} slides; {int(cohort.event.sum())} observed deaths. This is not yet the post-QC cohort.')
    return (cfg, cohort, slides)

## 6. CLAM tissue segmentation → coordinate sampling → real RGB patches

The actual CLAM `WholeSlideImage.segmentTissue` function and `isInContourV3_Hard` / hole tests are used.
The upstream TCGA preset is read from its source CSV. A serial global-grid adapter replaces only CLAM's
multiprocessing coordinate enumerator, avoiding forked workers in an initialized TensorFlow notebook.
No CLAM attention model is trained and no pretrained feature vectors are used.

For every slide, save **coordinates**, native dimensions, physical crop geometry, and a tissue-contour preview.
Read randomized coordinate candidates until at most 1,024 pass the additional tissue-fraction check.
After all slides of a patient have been processed, select 1,024 total patches using balanced, capacity-aware
slide quotas; prune the unselected temporary images. Sampling is without replacement. No tens-of-thousands-of-images
intermediate dataset is required, although multiple slides and QC may require reading more than the final 1,024.


In [31]:
def tissue_mask(rgb, cfg):
    a = np.asarray(rgb, dtype=np.uint8)
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV).astype('float32')
    return (hsv[..., 1] / 255 >= cfg['min_saturation']) & (hsv[..., 2] / 255 <= cfg['max_brightness'])

def _mpp(slide, cfg):
    props = slide.properties
    try:
        x = float(props['openslide.mpp-x'])
        y = float(props['openslide.mpp-y'])
    except (KeyError, ValueError, TypeError):
        override = cfg.get('unknown_mpp_override')
        if override is None:
            raise SlideGeometryError('Slide lacks microns-per-pixel metadata. Supply a documented unknown_mpp_override; magnification is not silently guessed.')
        x = y = float(override)
    if not (np.isfinite(x) and np.isfinite(y) and (x > 0) and (y > 0)):
        raise SlideGeometryError('Invalid WSI microns-per-pixel metadata.')
    return (x, y)

def read_patch(slide, x, y, cfg, mpp):
    mx, my = mpp
    desired_downsample = cfg['field_size_um'] / (2 * cfg['input_size'] * max(mx, my))
    level = slide.get_best_level_for_downsample(max(1.0, desired_downsample))
    ds = float(slide.level_downsamples[level])
    rw = max(1, int(math.ceil(cfg['field_size_um'] / mx / ds)))
    rh = max(1, int(math.ceil(cfg['field_size_um'] / my / ds)))
    rgba = slide.read_region((int(x), int(y)), level, (rw, rh)).convert('RGBA')
    white = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
    rgb = Image.alpha_composite(white, rgba).convert('RGB')
    return (np.asarray(rgb.resize((cfg['input_size'], cfg['input_size']), Image.Resampling.LANCZOS)), level)

### 6.1 Official CLAM source and TCGA preset

In [32]:
def setup_clam_source(work_dir, ref='master', local_source=''):
    """Resolve CLAM once, snapshot its preprocessing source, and verify it on resume.

    The first run resolves ref to a Git commit; subsequent sessions reuse the saved
    snapshot and do not track a moving branch. No unknown remote pickle is loaded.
    """
    import importlib
    work = Path(work_dir).resolve()
    root = work / 'vendor' / 'CLAM'
    marker = root / 'SOURCE_LOCK.json'
    required = ['wsi_core/WholeSlideImage.py', 'wsi_core/wsi_utils.py',
                'wsi_core/util_classes.py', 'utils/file_utils.py',
                'presets/tcga.csv', 'LICENSE.md']
    if marker.is_file():
        info = json.loads(marker.read_text())
        if info['requested_ref'] != str(ref):
            raise ValueError('CLAM reference changed. Start a new RUN_NAME.')
        for rel, digest in info['files'].items():
            path = root / rel
            if not path.is_file() or sha256_file(path) != digest:
                raise IOError(f'CLAM source snapshot changed or is missing: {path}')
    else:
        if root.exists():
            raise FileExistsError(f'Incomplete CLAM snapshot: {root}. Inspect/remove it before retrying.')
        root.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='cure_clam_') as tmp:
            tmp = Path(tmp)
            if local_source:
                source = Path(local_source).resolve()
                if not source.is_dir():
                    raise FileNotFoundError(source)
                try:
                    revision = subprocess.check_output(
                        ['git','-C',str(source),'rev-parse','HEAD'], text=True,
                        stderr=subprocess.STDOUT, timeout=30).strip()
                except (subprocess.CalledProcessError, FileNotFoundError):
                    revision = 'local-source-content-hashes'
            else:
                source = tmp / 'upstream'
                env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1', GIT_TERMINAL_PROMPT='0')
                subprocess.run(['git','init','--quiet',str(source)], check=True, env=env, timeout=30)
                subprocess.run(['git','-C',str(source),'remote','add','origin',
                                'https://github.com/mahmoodlab/CLAM.git'],check=True,env=env,timeout=30)
                subprocess.run(['git','-C',str(source),'fetch','--depth','1','origin',str(ref)],
                               check=True,env=env,timeout=300)
                subprocess.run(['git','-C',str(source),'checkout','--detach','FETCH_HEAD'],
                               check=True,env=env,timeout=120)
                revision = subprocess.check_output(['git','-C',str(source),'rev-parse','HEAD'],
                                                    text=True,env=env,timeout=30).strip()
            for rel in required:
                if not (source / rel).is_file():
                    raise FileNotFoundError(f'Official CLAM preprocessing file missing: {rel}')
            stage = root.with_name('CLAM.building')
            if stage.exists():
                shutil.rmtree(stage)
            stage.mkdir()
            for rel in ('wsi_core','presets'):
                shutil.copytree(source / rel, stage / rel,
                                ignore=shutil.ignore_patterns('__pycache__','*.pyc'))
            (stage / 'utils').mkdir()
            shutil.copy2(source / 'utils/file_utils.py', stage / 'utils/file_utils.py')
            for rel in ('LICENSE.md','README.md'):
                if (source / rel).is_file():
                    shutil.copy2(source / rel,stage / rel)
            # Package markers isolate these modules from similarly named Kaggle imports.
            for rel in ('wsi_core/__init__.py','utils/__init__.py'):
                if not (stage / rel).exists():
                    (stage / rel).write_text('')
            hashes = {str(p.relative_to(stage)):sha256_file(p)
                      for p in sorted(stage.rglob('*')) if p.is_file()}
            info = dict(repository='https://github.com/mahmoodlab/CLAM',
                        requested_ref=str(ref), resolved_revision=revision, files=hashes,
                        usage='upstream segmentTissue and contour tests; custom serial grid and sampling',
                        fingerprint=fingerprint(hashes))
            atomic_json(stage / 'SOURCE_LOCK.json',info)
            os.replace(stage,root)
    for prefix in ('wsi_core','utils'):
        for name, module in list(sys.modules.items()):
            if name == prefix or name.startswith(prefix + '.'):
                file = getattr(module,'__file__',None)
                if file is None or root not in Path(file).resolve().parents:
                    raise RuntimeError(f'{name} was imported from another package. Restart and run top-to-bottom.')
    if str(root) not in sys.path:
        sys.path.insert(0,str(root))
    importlib.invalidate_caches()
    from wsi_core.WholeSlideImage import WholeSlideImage as ClamWSI
    from wsi_core.util_classes import isInContourV3_Hard
    if not callable(getattr(ClamWSI,'segmentTissue',None)):
        raise RuntimeError('Incompatible CLAM preprocessing API.')
    print('CLAM revision:',info['resolved_revision'], '| source fingerprint:',info['fingerprint'])
    return dict(info, source_dir=str(root))

def clam_preset_parameters(source_dir, overrides):
    """Use the upstream TCGA preset, not guessed preset contents."""
    preset = pd.read_csv(Path(source_dir) / 'presets/tcga.csv').iloc[0].to_dict()
    names = ('seg_level','sthresh','mthresh','close','use_otsu','a_t','a_h','max_n_holes')
    absent = set(names) - set(preset)
    if absent:
        raise ValueError(f'CLAM tcga.csv lacks required columns: {sorted(absent)}')
    unknown = set(overrides) - set(names)
    if unknown:
        raise ValueError(f'Unknown CLAM segmentation overrides: {sorted(unknown)}')
    preset.update(overrides)
    boolean = str(preset['use_otsu']).strip().lower()
    if boolean not in ('true','false','1','0','1.0','0.0'):
        raise ValueError('CLAM use_otsu must be a boolean.')
    return dict(seg_level=int(preset['seg_level']),sthresh=int(preset['sthresh']),
                mthresh=int(preset['mthresh']),close=int(preset['close']),
                use_otsu=boolean in ('true','1','1.0'),
                a_t=float(preset['a_t']),a_h=float(preset['a_h']),
                max_n_holes=int(preset['max_n_holes']))

### 6.2 CLAM coordinate extraction and geometry

In [33]:
def clam_read_geometry(slide, pc):
    """Same physical field-of-view rule as v4; record integer-read rounding.

    Conservative square grid spacing encloses the actual reading rectangle,
    including pyramid rounding. There is no direct reshape of a whole slide.
    """
    mx,my = _mpp(slide,pc)
    desired = pc['field_size_um'] / (2 * pc['input_size'] * max(mx,my))
    level = int(slide.get_best_level_for_downsample(max(1.,desired)))
    ds = float(slide.level_downsamples[level])
    rw = max(1,int(math.ceil(pc['field_size_um'] / mx / ds)))
    rh = max(1,int(math.ceil(pc['field_size_um'] / my / ds)))
    span = max(int(math.ceil(rw*ds)),int(math.ceil(rh*ds)))
    return dict(read_level=level,read_width=rw,read_height=rh,grid_span_level0=span,
                downsample=ds,mpp_x=mx,mpp_y=my,
                actual_output_mpp_x=rw*ds*mx/pc['input_size'],
                actual_output_mpp_y=rh*ds*my/pc['input_size'])

def enumerate_clam_coordinates(wsi_object, footprint, max_grid_checks, deadline_unix=None):
    """Serial coordinate-only enumeration using CLAM's actual contour predicates.

    A single level-0 lattice makes coordinates unique and non-overlapping, even
    between contours. CLAM's multiprocessing process_contours is intentionally
    not invoked inside an already initialized TensorFlow notebook.
    """
    from wsi_core.WholeSlideImage import WholeSlideImage as ClamWSI
    from wsi_core.util_classes import isInContourV3_Hard
    span = int(footprint)
    width,height = map(int,wsi_object.wsi.dimensions)
    if span <= 0 or max_grid_checks < 1:
        raise ValueError('Positive grid span and check budget required.')
    holes = getattr(wsi_object,'holes_tissue',[])
    contours = wsi_object.contours_tissue
    if len(contours) != len(holes):
        raise ValueError('CLAM tissue/hole contour lengths disagree.')
    accepted = set()
    checked = 0
    for contour, contour_holes in zip(contours,holes):
        bx,by,bw,bh = cv2.boundingRect(contour)
        first_x = max(0,((bx-span)//span)*span)
        first_y = max(0,((by-span)//span)*span)
        stop_x = min(bx+bw,width-span)+1
        stop_y = min(by+bh,height-span)+1
        predicate = isInContourV3_Hard(contour=contour,patch_size=span,center_shift=0.5)
        for y in range(first_y,stop_y,span):
            for x in range(first_x,stop_x,span):
                checked += 1
                if checked > max_grid_checks:
                    raise RuntimeError('CLAM coordinate-check budget exceeded. No silent grid truncation; '
                                       'review physical scale and increase the explicit guard in a new run.')
                if checked % 4096 == 0:
                    check_deadline(deadline_unix)
                if ClamWSI.isInContours(predicate,(int(x),int(y)),contour_holes,span):
                    accepted.add((int(x),int(y)))
    coords = np.asarray(sorted(accepted,key=lambda z:(z[1],z[0])),dtype=np.int64).reshape(-1,2)
    return coords, checked

def save_clam_coordinate_h5(path, coords, meta):
    import h5py
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_suffix('.tmp.h5')
    coords = np.asarray(coords,dtype=np.int64).reshape(-1,2)
    with h5py.File(temp,'w') as f:
        data = f.create_dataset('coords',data=coords)
        data.attrs['coordinate_space']='level0_top_left_xy'
        data.attrs['patch_level']=int(meta['geometry']['read_level'])
        data.attrs['patch_size_xy']=[meta['geometry']['read_width'],meta['geometry']['read_height']]
        data.attrs['output_shape']=[128,128,3]
        data.attrs['metadata_json']=json.dumps(meta,sort_keys=True,allow_nan=False)
    os.replace(temp,path)

def clam_coordinate_cache(source_path, folder, pc, source_info, deadline_unix=None):
    """Run upstream CLAM tissue segmentation, save all eligible coordinates only.

    Never uses CLAM attention scores, a pretrained ResNet, or feature vectors.
    The caller closes the returned native slide handle in all paths.
    """
    from wsi_core.WholeSlideImage import WholeSlideImage as ClamWSI
    folder = Path(folder); folder.mkdir(parents=True,exist_ok=True)
    obj = ClamWSI(str(source_path))
    try:
        geom = clam_read_geometry(obj.wsi,pc)
        cp = pc['clam']; params=clam_preset_parameters(source_info['source_dir'],cp['segmentation_overrides'])
        seg_level=params.pop('seg_level')
        if seg_level < 0:
            seg_level=int(obj.wsi.get_best_level_for_downsample(float(cp['segmentation_downsample'])))
        if not 0 <= seg_level < obj.wsi.level_count:
            raise SlideGeometryError('CLAM segmentation level is not present in this slide.')
        seg_w,seg_h=obj.wsi.level_dimensions[seg_level]
        if seg_w*seg_h > cp['max_segmentation_pixels']:
            raise SlideGeometryError('Chosen CLAM segmentation image exceeds the pixel guard. '
                                     'Supply a coarser valid level; no thumbnail masquerading as WSI.')
        check_deadline(deadline_unix)
        # Upstream segmentTissue has no empty-hierarchy branch. Detect ONLY an
        # empty threshold mask using the same thresholding operations beforehand.
        preview_rgba=np.asarray(obj.wsi.read_region((0,0),seg_level,(seg_w,seg_h)))
        saturation=cv2.cvtColor(preview_rgba[:,:,:3],cv2.COLOR_RGB2HSV)[:,:,1]
        med=cv2.medianBlur(saturation,params['mthresh'])
        threshold_flag=cv2.THRESH_BINARY | (cv2.THRESH_OTSU if params['use_otsu'] else 0)
        _,binary=cv2.threshold(med,0 if params['use_otsu'] else params['sthresh'],255,threshold_flag)
        if params['close']>0:
            binary=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,
                                   np.ones((params['close'],params['close']),np.uint8))
        contours,_=cv2.findContours(binary,cv2.RETR_CCOMP,cv2.CHAIN_APPROX_NONE)
        del preview_rgba,saturation,med,binary
        if len(contours)==0:
            obj.contours_tissue=[]; obj.holes_tissue=[]
        else:
            obj.segmentTissue(seg_level=seg_level,sthresh=params['sthresh'],
                              mthresh=params['mthresh'],close=params['close'],
                              use_otsu=params['use_otsu'],
                              filter_params={k:params[k] for k in ('a_t','a_h','max_n_holes')})
        coords,checks=enumerate_clam_coordinates(obj,geom['grid_span_level0'],cp['max_grid_checks'],deadline_unix)
        width,height=map(int,obj.wsi.dimensions)
        meta=dict(slide_path_name=Path(source_path).name,width=width,height=height,
                  level_dimensions=[list(map(int,d)) for d in obj.wsi.level_dimensions],
                  geometry=geom,segmentation_level=seg_level,parameters=params,
                  source_fingerprint=source_info['fingerprint'],
                  source_revision=source_info['resolved_revision'],
                  tissue_contours=len(obj.contours_tissue),eligible_coordinates=int(len(coords)),
                  checked_grid_positions=int(checks),
                  policy='CLAM segmentation and four_pt_hard/hole tests; custom serial level-0 global grid')
        save_clam_coordinate_h5(folder/'coordinates.h5',coords,meta)
        atomic_json(folder/'coordinate_metadata.json',meta)
        thumb=obj.wsi.get_thumbnail((1536,1536)).convert('RGB')
        thumb.save(folder/'slide_thumbnail.jpg',quality=90)
        outline=np.asarray(thumb).copy(); sx=thumb.width/width; sy=thumb.height/height
        for contour in obj.contours_tissue:
            c=np.rint(contour*np.array([sx,sy])).astype(np.int32)
            cv2.drawContours(outline,[c],-1,(0,160,0),1)
        for hole_set in obj.holes_tissue:
            for hole in hole_set:
                h=np.rint(hole*np.array([sx,sy])).astype(np.int32)
                cv2.drawContours(outline,[h],-1,(200,0,0),1)
        Image.fromarray(outline).save(folder/'tissue_contours.png')
        return obj,coords,meta
    except BaseException:
        obj.wsi.close()
        raise

### 6.3 Patient-level sampling and selection previews

In [34]:
def balanced_patch_quotas(capacities, target):
    """Equal-slide quotas with capacity-aware redistribution; sums to min(total,target)."""
    capacities=np.asarray(capacities,dtype=np.int64)
    if capacities.ndim!=1 or np.any(capacities<0) or int(target)<0:
        raise ValueError('Nonnegative capacities and target required.')
    quotas=np.zeros_like(capacities); remaining=min(int(target),int(capacities.sum()))
    while remaining:
        active=np.flatnonzero(quotas<capacities)
        share=max(1,remaining//len(active))
        for i in active:
            add=min(share,int(capacities[i]-quotas[i]),remaining)
            quotas[i]+=add;remaining-=add
            if not remaining:break
    return quotas.tolist()

def select_patient_patch_records(slide_records, target, seed):
    """Select up to target unique images across ALL slides, without replacement.

    This is slide-balanced random selection, NOT learned CLAM top-attention selection.
    Each slide contributes a randomized, QC-passing pool of at most target patches.
    """
    keys=sorted(slide_records)
    rng=np.random.default_rng(seed)
    keys=[keys[i] for i in rng.permutation(len(keys))]
    quotas=balanced_patch_quotas([len(slide_records[k]) for k in keys],target)
    out=[]
    for key,quota in zip(keys,quotas):
        records=slide_records[key]
        if quota:
            ix=rng.choice(len(records),quota,replace=False)
            out.extend(records[i] for i in sorted(ix))
    identities=[(r['slide_id'],int(r['x_level0']),int(r['y_level0'])) for r in out]
    if len(set(identities))!=len(identities):
        raise ValueError('Duplicate tissue coordinates in selected patient bag.')
    return sorted(out,key=lambda r:(r['slide_id'],r['y_level0'],r['x_level0']))

def prune_unselected_patient_images(patient_folder, selected):
    """Delete only intermediate images created by this workflow, after atomic commit."""
    keep={str(Path(r['path']).resolve()) for r in selected}
    for pattern in ('*/patch_*.jpg','*/patch_*.png'):
        for p in Path(patient_folder).glob(pattern):
            if str(p.resolve()) not in keep:
                p.unlink()

def save_selected_coordinate_previews(patient_folder, selected):
    from PIL import ImageDraw
    for folder in sorted(Path(patient_folder).iterdir()):
        if not folder.is_dir() or not (folder/'coordinate_metadata.json').exists():
            continue
        meta=json.loads((folder/'coordinate_metadata.json').read_text())
        with Image.open(folder/'slide_thumbnail.jpg') as base:
            image=base.convert('RGB')
        sx=image.width/meta['width'];sy=image.height/meta['height']
        span=meta['geometry']['grid_span_level0'];draw=ImageDraw.Draw(image)
        owned=[r for r in selected if r['slide_id']==folder.name]
        for r in owned:
            x,y=r['x_level0'],r['y_level0']
            draw.rectangle((x*sx,y*sy,(x+span)*sx,(y+span)*sy),outline=(220,0,0),width=1)
        image.save(folder/'selected_locations.jpg',quality=90)

In [35]:
def validate_config(cfg):
    if cfg['patch']['input_size'] != 128:
        raise ValueError('The preserved CURE input contract is 128 x 128 x 3.')
    pc, tc = (cfg['patch'], cfg['training'])
    for key in ('pool_per_patient', 'min_patches_per_patient'):
        if not isinstance(pc[key], int) or pc[key] < 1:
            raise ValueError(f'patch.{key} must be a positive integer.')
    if pc['min_patches_per_patient'] > pc['pool_per_patient']:
        raise ValueError('min_patches_per_patient exceeds pool_per_patient.')
    for key in ('folds', 'epochs', 'patient_batch_size', 'patches_per_patient_step', 'evaluation_chunk_size', 'survival_intervals', 'patience'):
        if not isinstance(tc[key], int) or tc[key] < 1:
            raise ValueError(f'training.{key} must be a positive integer.')
    if tc['folds'] < 2 or not 0 < tc['validation_fraction'] < 1:
        raise ValueError('Require >=2 folds and 0 < validation_fraction < 1.')
    if pc['field_size_um'] <= 0 or not 0 < pc['tissue_fraction_min'] <= 1:
        raise ValueError('Invalid physical patch size/tissue threshold.')
    if cfg['kaggle']['max_output_gib'] <= cfg['kaggle']['model_reserve_gib']:
        raise ValueError('Output budget must exceed model reserve.')
    if not 0 < cfg['kaggle']['session_hours'] <= 10.5:
        raise ValueError("Choose a session budget between 0 and 10.5 hours, below Kaggle's hard limit.")
    if tc.get('encoder_microbatch_size',4) < 1:
        raise ValueError('encoder_microbatch_size must be positive.')
    if tc.get('patch_sampling','random') not in ('random','coverage_cycle'):
        raise ValueError('Unknown patch sampling rule.')
    patch_cache_suffix(pc)
    if not 1 <= int(pc.get('jpeg_quality',95)) <= 100:
        raise ValueError('jpeg_quality must be in [1,100].')
    cp=pc['clam']
    if cp['short_bag_policy'] not in ('exclude','keep'):
        raise ValueError('SHORT_BAG_POLICY must be exclude or keep.')
    if cp['max_grid_checks']<1 or cp['max_segmentation_pixels']<1:
        raise ValueError('Positive CLAM processing guards required.')
    if not cp.get('source_info',{}).get('fingerprint'):
        raise ValueError('Resolve and verify the CLAM source before binding this run.')
    return cfg


def _science_contract(cfg):
    keys = ('omics_source', 'id_column', 'feature_patterns', 'diagnostic_slides_only', 'patch', 'training', 'clinical_columns')
    out = {k: copy.deepcopy(cfg[k]) for k in keys}
    out['training'].pop('resume', None)
    out['training'].pop('checkpoint_every_steps', None)
    out['training'].pop('log_every_steps', None)
    for key in ('id_mapping_csv', 'clinical_csv', 'split_manifest', 'local_slides_manifest'):
        p = cfg.get(key)
        out[key] = sha256_file(p) if p else None
    out['molecular_source_mode'] = cfg.get('molecular_source_mode', 'attached')
    out['cbioportal'] = copy.deepcopy(cfg.get('cbioportal'))
    out['source_code'] = implementation_fingerprint()
    return out


def bind_run(cfg):
    """Lock data/model settings and keep small source tables inside saved outputs."""
    cfg = copy.deepcopy(validate_config(cfg))
    work = Path(cfg['work_dir']).resolve()
    work.mkdir(parents=True, exist_ok=True)
    cfg['work_dir'] = str(work)
    lock = work / 'RUN_CONTRACT.json'
    for key in ('id_mapping_csv', 'clinical_csv', 'split_manifest', 'local_slides_manifest'):
        p = cfg.get(key)
        if not p:
            saved_manifests = sorted((work / 'sources').glob(key + '.*'))
            if len(saved_manifests) == 1:
                p = str(saved_manifests[0])
                cfg[key] = p
        if p:
            source = Path(p)
            target = work / 'sources' / (key + source.suffix)
            target.parent.mkdir(exist_ok=True)
            if source.resolve() != target.resolve():
                if target.exists() and sha256_file(target) != sha256_file(source):
                    raise ValueError(f'{key} content changed; use a new experiment directory.')
                if not target.exists():
                    shutil.copy2(source, target)
            cfg[key] = str(target.resolve())
    if cfg.get('molecular_source_mode', 'attached') == 'attached':
        cfg = prepare_omics_input(cfg)
    else:
        # Source adapter already selected and validated the publisher-identified matrix.
        load_cbio_prepared_npz(cfg['omics_path'], cfg['feature_patterns'], cfg.get('id_column'))
    source = Path(cfg['omics_path'])
    if not source.is_file():
        raise FileNotFoundError(source)
    target = work / 'sources' / ('omics' + ''.join(source.suffixes))
    target.parent.mkdir(exist_ok=True)
    if source.resolve() != target.resolve():
        if target.exists() and sha256_file(target) != sha256_file(source):
            raise ValueError('Molecular table changed; use a new experiment.')
        if not target.exists():
            shutil.copy2(source, target)
    cfg['omics_path'] = str(target.resolve())
    blocks, audit = load_omics(target, cfg['feature_patterns'], cfg.get('id_column'), cfg.get('id_mapping_csv'))
    print('Actual molecular-table audit:', json.dumps({k: audit[k] for k in ('unique_patients', 'features', 'id_column', 'channel_order')}, indent=2), flush=True)
    details = _science_contract(cfg)
    details['omics_sha256'] = sha256_file(target)
    digest = fingerprint(details)
    if lock.exists():
        old = json.loads(lock.read_text())
        if old['signature'] != digest or old['work_dir'] != str(work):
            raise ValueError('Science/data/code contract differs from the saved run. Restore the same configuration or use a new work directory.')
    else:
        atomic_json(lock, dict(signature=digest, work_dir=str(work), details=details))
    atomic_json(work / 'kaggle_config.json', cfg)
    return cfg


def _metadata(cfg):
    work = Path(cfg['work_dir'])
    stamp = work / 'METADATA_COMPLETE.json'
    names = ('cohort_before_patch_qc.csv', 'slides.csv', 'omics_audit.json')
    if stamp.exists():
        recorded = json.loads(stamp.read_text())
        for name in names:
            if not (work / name).is_file() or sha256_file(work / name) != recorded[name]:
                raise ValueError(f'Incomplete/modified metadata cache: {name}')
        return (pd.read_csv(work / names[0]), pd.read_csv(work / names[1]))
    cfg, cohort, slides = prepare_metadata(cfg)
    atomic_json(stamp, {name: sha256_file(work / name) for name in names})
    return (cohort, slides)


def _source_key(row):
    if 'file_id' in row and pd.notna(row['file_id']):
        return str(row['file_id'])
    p = Path(row['path'])
    return fingerprint(dict(path=str(p.resolve()), size=p.stat().st_size, mtime=p.stat().st_mtime_ns))[:32]


def _cached_slide(manifest, expected_signature):
    saved = json.loads(Path(manifest).read_text())
    if saved['signature'] != expected_signature:
        raise ValueError(f'Patch/source settings changed: {manifest}')
    for record in saved['patches']:
        p = Path(record['path'])
        if not p.is_file() or sha256_file(p) != record['sha256']:
            raise IOError(f'Missing or modified cached RGB patch: {p}')
    return saved

In [36]:
def prepare_streaming(cfg, deadline_unix=None):
    """CLAM coordinates -> per-slide candidate pools -> 1,024 selected patches/patient.

    Raw SVS is read sequentially. Intermediate slide pools are pruned ONLY after
    the final patient manifest is atomically committed. This allows quota
    redistribution when one slide is short, without silently duplicating images.
    """
    import openslide
    from types import SimpleNamespace
    work=Path(cfg['work_dir']);pc=cfg['patch'];kc=cfg['kaggle'];cap=int(pc['pool_per_patient'])
    cohort,slides=_metadata(cfg)
    source_info=cfg['patch']['clam']['source_info']
    estimate=len(cohort)*cap*(128*128*3+2048)+kc['model_reserve_gib']*2**30
    atomic_json(work/'STORAGE_PLAN.json',dict(
        patients_before_patch_qc=len(cohort),selected_patches_per_patient=cap,
        conservative_final_patch_plus_model_gib=estimate/2**30,
        max_output_gib=kc['max_output_gib'],storage_format=pc['storage_format'],
        raw_transfer_gib=float(slides.file_size.sum()/2**30) if 'file_size' in slides else None,
        note='JPEG compression is content-dependent; provisional pools across one patient add temporary disk overhead.'))
    if estimate>kc['max_output_gib']*2**30:
        print('WARNING: uncompressed upper bound exceeds saved-output budget. Actual stored bytes are checked; '
              'JPEG is lossy and completion within this budget is not guaranteed.',flush=True)
    remote='file_id' in slides
    if remote and not cfg['approve_slide_downloads']:
        raise PermissionError('Review STORAGE_PLAN.json and authorize GDC downloads explicitly.')
    if remote and slides.file_size.sum()/2**30>cfg['max_download_gib']:
        raise ValueError('Total requested GDC transfer exceeds the declared budget; no cohort truncation.')
    scratch=Path(kc['scratch_dir']).resolve();scratch.mkdir(parents=True,exist_ok=True)
    if scratch==work or work in scratch.parents or str(scratch).startswith('/kaggle/input'):
        raise ValueError('Raw-slide scratch must be separate from saved output and read-only inputs.')
    budget=kc['max_output_gib']*2**30
    reserve=kc['model_reserve_gib']*2**30
    all_records=[];reports=[];patient_reports=[];completed=0
    output_used=tree_bytes(work)
    for pid in cohort.patient_id:
        check_deadline(deadline_unix)
        rows=slides.loc[slides.patient_id==pid].to_dict('records')
        source_keys=[_source_key(row) for row in rows]
        if len(source_keys)!=len(set(source_keys)):
            raise ValueError(f'Duplicate source slides for patient {pid}.')
        patient_folder=work/'patches'/pid;patient_folder.mkdir(parents=True,exist_ok=True)
        patient_base_bytes=output_used-tree_bytes(patient_folder)
        patient_manifest=patient_folder/'patient_manifest.json'
        signature=fingerprint(dict(protocol='clam_spatial_cure_v5',patient=pid,
            sources=[dict(key=k,md5=r.get('md5sum')) for k,r in zip(source_keys,rows)],patch=pc))
        if patient_manifest.exists():
            saved=_cached_slide(patient_manifest,signature)
            # Protect coordinate/mask provenance, not just selected RGB bytes.
            for artifact in saved.get('artifacts',[]):
                path=Path(artifact['path'])
                if not path.is_file() or sha256_file(path)!=artifact['sha256']:
                    raise IOError(f'Missing/changed CLAM artifact: {path}')
            selected=saved['patches']
            prune_unselected_patient_images(patient_folder,selected)
            reports.extend(saved['reports']);patient_reports.append(saved['patient_report'])
        else:
            pools={};notes=[];slide_audits=[];artifacts=[]
            for key,row in zip(source_keys,rows):
                check_deadline(deadline_unix)
                folder=patient_folder/key;folder.mkdir(exist_ok=True)
                manifest=folder/'candidate_pool_manifest.json'
                sig=fingerprint(dict(patient=pid,source=key,md5=row.get('md5sum'),
                                     patch=pc,protocol='clam_candidate_pool_v5'))
                if manifest.exists():
                    item=_cached_slide(manifest,sig)
                    for artifact in item.get('artifacts',[]):
                        path=Path(artifact['path'])
                        if not path.is_file() or sha256_file(path)!=artifact['sha256']:
                            raise IOError(f'Missing/modified CLAM coordinate artifact: {path}')
                    pools[key]=item['patches'];notes.extend(item['reports'])
                    slide_audits.append(item['slide_audit']);artifacts.extend(item.get('artifacts',[]))
                    continue
                # An interrupted slide is regenerated, not mixed with a prior partial result.
                for pattern in ('patch_*.jpg','patch_*.png','*.tmp.h5'):
                    for p in folder.glob(pattern):p.unlink()
                owned_raw=None;obj=None;current=[];slide_notes=[];slide_artifacts=[]
                if remote:
                    needed=int(row['file_size'])
                    if shutil.disk_usage(scratch).free<needed*1.05+2**29:
                        raise OSError('Insufficient scratch for the next actual SVS.')
                    owned_raw=Path(download_slide(SimpleNamespace(**row),scratch,
                        os.environ.get(cfg.get('download_token_env','GDC_TOKEN')),deadline_unix))
                    source_path=owned_raw
                else:
                    source_path=Path(row['path'])
                    if not source_path.is_file():raise FileNotFoundError(source_path)
                audit=dict(patient_id=pid,slide_id=key,slide_name=row.get('file_name',source_path.name),
                           eligible_coordinates=0,images_decoded=0,accepted_in_candidate_pool=0,status='pending')
                try:
                    obj,coords,meta=clam_coordinate_cache(source_path,folder,pc,source_info,deadline_unix)
                    audit.update(width=meta['width'],height=meta['height'],
                                 eligible_coordinates=len(coords),status='completed',
                                 checked_grid_positions=meta['checked_grid_positions'],
                                 segmentation_level=meta['segmentation_level'])
                    # Randomized coordinate order, not saliency/attention ranking.
                    rng=np.random.default_rng(stable_seed(pc['seed'],pid,key,'clam_candidate_order'))
                    order=rng.permutation(len(coords));g=meta['geometry']
                    used=patient_base_bytes+tree_bytes(patient_folder)
                    for ci in order:
                        if len(current)>=cap:break
                        check_deadline(deadline_unix)
                        x,y=map(int,coords[ci])
                        # Read only inspected selected candidates; never every possible tile.
                        rgb,level=read_patch(obj.wsi,x,y,pc,(g['mpp_x'],g['mpp_y']))
                        audit['images_decoded']+=1
                        fraction=float(tissue_mask(rgb,pc).mean())
                        if fraction<pc['tissue_fraction_min']:continue
                        path=folder/(f'patch_{len(current):06d}'+patch_cache_suffix(pc))
                        if shutil.disk_usage(work).free<2**29:
                            raise OSError('Less than the 512 MiB safety margin remains on disk.')
                        save_rgb_patch(rgb,path,pc);used+=path.stat().st_size
                        if used+reserve>budget:
                            raise OSError('CLAM candidate/selected image cache exceeds the actual-byte '
                                          'budget. Save on larger persistent storage; no hidden reduction.')
                        current.append(dict(patient_id=pid,path=str(path.resolve()),slide_id=key,
                            slide_name=audit['slide_name'],x_level0=x,y_level0=y,read_level=level,
                            read_width=g['read_width'],read_height=g['read_height'],
                            grid_span_level0=g['grid_span_level0'],tissue_fraction=fraction,
                            field_size_um=pc['field_size_um'],output_mpp=pc['field_size_um']/128,
                            actual_output_mpp_x=g['actual_output_mpp_x'],
                            actual_output_mpp_y=g['actual_output_mpp_y'],
                            selection='CLAM-tissue/slide-balanced-random',sha256=sha256_file(path)))
                    audit['accepted_in_candidate_pool']=len(current)
                    for name in ('coordinates.h5','coordinate_metadata.json','slide_thumbnail.jpg','tissue_contours.png'):
                        p=folder/name
                        slide_artifacts.append(dict(path=str(p.resolve()),sha256=sha256_file(p)))
                except (openslide.OpenSlideError,SlideGeometryError) as error:
                    # Native file/scale QC is recorded; unexpected programming errors propagate.
                    # Do not keep only a readable prefix of an explicitly failed slide.
                    for record in current:Path(record['path']).unlink(missing_ok=True)
                    current=[];audit['status']='excluded_slide_QC';audit['accepted_in_candidate_pool']=0
                    slide_notes.append(dict(patient_id=pid,slide_id=key,reason=type(error).__name__+': '+str(error)))
                finally:
                    if obj is not None:obj.wsi.close()
                slide_notes.append(dict(patient_id=pid,slide_id=key,
                    reason=f"CLAM_coords={audit['eligible_coordinates']}; decoded={audit['images_decoded']}; candidate_pool={len(current)}"))
                atomic_json(manifest,dict(signature=sig,patches=current,reports=slide_notes,
                                         slide_audit=audit,artifacts=slide_artifacts))
                # Only remove a raw file downloaded by THIS pipeline, after slide commit.
                if owned_raw is not None:
                    owned_raw.unlink(missing_ok=True)
                    try:owned_raw.parent.rmdir()
                    except OSError:pass
                pools[key]=current;notes.extend(slide_notes);slide_audits.append(audit);artifacts.extend(slide_artifacts)
            selected=select_patient_patch_records(pools,cap,stable_seed(pc['seed'],pid,'patient_selection'))
            available=len(selected)
            policy=pc['clam']['short_bag_policy']
            if available<cap:
                notes.append(dict(patient_id=pid,slide_id='',reason=f'short_bag_{available}_of_{cap}; policy={policy}'))
                if policy=='exclude':selected=[]
            if len(selected)<pc['min_patches_per_patient']:selected=[]
            patient_report=dict(patient_id=pid,slides=len(rows),
                eligible_clam_coordinates=sum(int(a['eligible_coordinates']) for a in slide_audits),
                images_decoded=sum(int(a['images_decoded']) for a in slide_audits),
                candidate_images=sum(len(x) for x in pools.values()),
                selected_patches=len(selected),target_patches=cap,
                retained=bool(selected),short_bag_policy=policy,
                reason='retained' if selected else 'excluded_insufficient_unique_QC_patches')
            atomic_json(patient_folder/'selection_summary.json',dict(patient=patient_report,slides=slide_audits))
            summary=patient_folder/'selection_summary.json'
            artifacts.append(dict(path=str(summary.resolve()),sha256=sha256_file(summary)))
            atomic_json(patient_manifest,dict(signature=signature,patches=selected,reports=notes,
                                             patient_report=patient_report,artifacts=artifacts))
            # The final atomic manifest is now sufficient for resuming after pruning.
            prune_unselected_patient_images(patient_folder,selected)
            reports.extend(notes);patient_reports.append(patient_report)
        save_selected_coordinate_previews(patient_folder,selected)
        output_used=patient_base_bytes+tree_bytes(patient_folder)
        if output_used+reserve>budget:
            raise OSError('Actual selected-patch output exceeds saved-output budget.')
        all_records.extend(selected);completed+=1
        print(f'Patient {completed}/{len(cohort)} {pid}: selected={len(selected)}; '
              f'total selected RGB images={len(all_records)}',flush=True)
        status(work,'clam_preparation','running',processed_patients=completed,
               total_patients=len(cohort),selected_patch_count=len(all_records))
    atomic_csv(pd.DataFrame(reports),work/'patch_qc_report.csv')
    atomic_csv(pd.DataFrame(patient_reports),work/'clam_patient_selection.csv')
    if not all_records:
        raise ValueError('No paired patients retained. Inspect CLAM masks and clam_patient_selection.csv; '
                         'do not duplicate patches to force a 1,024 count.')
    columns=['patient_id','path','slide_id','slide_name','x_level0','y_level0','read_level',
             'read_width','read_height','grid_span_level0','tissue_fraction','field_size_um','output_mpp',
             'actual_output_mpp_x','actual_output_mpp_y','selection','sha256']
    patches=pd.DataFrame(all_records,columns=columns)
    counts=patches.groupby('patient_id').size()
    final=cohort.loc[cohort.patient_id.isin(counts.index)].copy()
    final['patch_count']=final.patient_id.map(counts).astype(int)
    if pc['clam']['short_bag_policy']=='exclude' and not (final.patch_count==cap).all():
        raise AssertionError('Strict 1,024-patch patient contract was violated.')
    if patches.duplicated(['patient_id','slide_id','x_level0','y_level0']).any():
        raise AssertionError('Selected manifest contains duplicate locations.')
    atomic_csv(patches,work/'patches.csv');atomic_csv(final,work/'cohort.csv')
    atomic_json(work/'DATA_COMPLETE.json',dict(status='complete',processed_patients=completed,
        metadata_patients=len(cohort),paired_patients=len(final),patches=len(patches),
        cohort_sha256=sha256_file(work/'cohort.csv'),patches_sha256=sha256_file(work/'patches.csv'),
        clam_selection_sha256=sha256_file(work/'clam_patient_selection.csv'),
        clam_source_fingerprint=source_info['fingerprint']))
    assert_storage(work,budget,reserve)
    print('Measured final paired cohort:',len(final),'patients;',len(patches),'selected RGB patches.')
    print('Complete WSI indexed shape:',(len(patches),128,128,3))
    print('Complete paired omics indexed shape:',(len(patches),128,128,3))
    return final,patches


def require_complete_data(work, deadline_unix=None):
    work = Path(work)
    marker = work / 'DATA_COMPLETE.json'
    if not marker.is_file():
        raise RuntimeError('Real-data preparation is incomplete; training on a temporary subset is not allowed.')
    info = json.loads(marker.read_text())
    if info['processed_patients'] != info['metadata_patients']:
        raise RuntimeError('Incomplete cohort coverage in data marker.')
    for name in ('cohort', 'patches'):
        if sha256_file(work / f'{name}.csv') != info[f'{name}_sha256']:
            raise IOError(f'Modified data manifest: {name}.csv')
    frame = pd.read_csv(work / 'patches.csv')
    for index, row in enumerate(frame.itertuples()):
        if index % 1000 == 0:
            check_deadline(deadline_unix)
        path = Path(row.path)
        if not path.is_file():
            raise FileNotFoundError(f'RGB patch missing: {path}')
        if sha256_file(path) != row.sha256:
            raise IOError(f'RGB patch content differs from committed cache: {path}')
    return info

## 7. Patient-level splits, augmentation and patch–omics pairing
The important operation is `omap = self.maps[pid]` inside that patient's patch loop.
Every patch therefore receives the **same patient's** molecular map. The owner index is kept for survival aggregation.
Four-view augmentation produces four successive variant batches, not four new patients or 256 distinct omics observations.

The revised default coverage cycle visits all cached patch locations without replacement before starting
a new shuffled cycle. Four augmentation variants revisit the same selected coordinates, not new tissue.

In [37]:
def validate_folds(splits, patient_ids, n_folds):
    required = {'patient_id', 'fold', 'split'}
    if not required <= set(splits):
        raise ValueError(f'Split manifest needs {required}.')
    expected = set(patient_ids)
    all_tests = []
    if set(splits.fold) != set(range(n_folds)):
        raise ValueError('Fold IDs must be 0..folds-1.')
    for fold in range(n_folds):
        part = splits[splits.fold == fold]
        if part.patient_id.duplicated().any():
            raise ValueError(f'Duplicate patient in fold {fold}.')
        if set(part.patient_id) != expected:
            raise ValueError(f'Fold {fold} must cover exactly the post-QC cohort.')
        if set(part.split) != {'train', 'validation', 'test'}:
            raise ValueError('Every fold needs three nonempty splits.')
        all_tests.extend(part.loc[part.split == 'test', 'patient_id'])
    if len(all_tests) != len(expected) or set(all_tests) != expected:
        raise ValueError('Each patient must appear in exactly one outer test fold.')
    return splits


def prepare_splits(cfg):
    work = Path(cfg['work_dir'])
    cohort = pd.read_csv(work / 'cohort.csv')
    tc = cfg['training']
    target = work / 'splits.csv'
    signature = fingerprint({'cohort': cohort.to_dict('list'), 'folds': tc['folds'], 'seed': tc['seed'], 'validation_fraction': tc['validation_fraction'], 'split_manifest': cfg.get('split_manifest'), 'split_manifest_sha256': sha256_file(cfg['split_manifest']) if cfg.get('split_manifest') else None})
    stamp = work / 'splits_signature.json'
    if target.exists():
        if not stamp.exists() or json.loads(stamp.read_text())['signature'] != signature:
            raise ValueError('Split settings/cohort changed; do not overwrite existing folds. Use a new work_dir.')
        return validate_folds(pd.read_csv(target), cohort.patient_id, tc['folds'])
    if cfg.get('split_manifest'):
        result = validate_folds(pd.read_csv(cfg['split_manifest']), cohort.patient_id, tc['folds'])
    else:
        if cohort.event.value_counts().min() < tc['folds'] or cohort.event.nunique() != 2:
            raise ValueError('Not enough observed deaths/censored patients for the requested stratified folds.')
        cv = StratifiedKFold(tc['folds'], shuffle=True, random_state=tc['seed'])
        records = []
        for f, (development, test) in enumerate(cv.split(cohort.patient_id, cohort.event)):
            train, val = train_test_split(development, test_size=tc['validation_fraction'], random_state=tc['seed'] + f, stratify=cohort.iloc[development].event)
            for split, ix in [('train', train), ('validation', val), ('test', test)]:
                records.extend((dict(patient_id=pid, fold=f, split=split) for pid in cohort.iloc[ix].patient_id))
        result = validate_folds(pd.DataFrame(records), cohort.patient_id, tc['folds'])
    result.to_csv(target, index=False)
    atomic_json(stamp, {'signature': signature})
    return result

In [38]:
def coverage_patch_indices(n, k, epoch, seed, pid, sampling='coverage_cycle'):
    """A new set of tissue locations each epoch; augmentation does not advance coverage.

    In coverage_cycle mode every retained location appears once per ceil(n/k)
    epochs. The final bag of a cycle can be smaller than k; no padding/duplication.
    """
    if min(n, k) < 1 or epoch < 0:
        raise ValueError('n and k must be positive and epoch nonnegative.')
    if sampling == 'random':
        return np.random.default_rng(stable_seed(seed, epoch, pid)).choice(n, min(k, n), replace=False)
    if sampling != 'coverage_cycle':
        raise ValueError('patch_sampling must be coverage_cycle or random.')
    steps = math.ceil(n / k)
    cycle, offset = divmod(epoch, steps)
    order = np.random.default_rng(stable_seed(seed, pid, cycle, 'patch_coverage_v1')).permutation(n)
    return order[offset*k:min((offset+1)*k, n)]


def replay_state_variables(model):
    """Mutable Keras state, including BatchNorm statistics and Dropout RNG counters.

    The source CURE DCT matrix is a fixed nontrainable weight, not mutable state.
    Excluding it avoids copying identical large matrices for every microbatch.
    No random TensorFlow operation with untracked state may be added to the encoder.
    """
    return [v for v in model.non_trainable_variables
            if str(getattr(v, 'path', v.name)).split('/')[-1] != 'orthonormal_dct_ii']


def snapshot_replay_state(model):
    return [np.array(ops.convert_to_numpy(v), copy=True) for v in replay_state_variables(model)]


def restore_replay_state(model, state):
    variables = replay_state_variables(model)
    if len(variables) != len(state):
        raise RuntimeError('Layer state changed after building the replay plan.')
    for variable, value in zip(variables, state):
        if tuple(variable.shape) != tuple(value.shape):
            raise RuntimeError('Replay-state shape mismatch.')
        variable.assign(value)


def large_bag_plan(n_rows, n_patients, microbatch_size):
    if min(n_rows, n_patients, microbatch_size) < 1:
        raise ValueError('Counts must be positive.')
    return dict(patch_pairs_per_update=int(n_rows), independent_patients=int(n_patients),
                host_wsi_shape=[int(n_rows),128,128,3], host_omics_shape=[int(n_rows),128,128,3],
                maximum_encoder_chunk_shape=[int(min(microbatch_size,n_rows)),128,128,3],
                chunks_per_encoder_pass=int(math.ceil(n_rows/microbatch_size)),
                encoder_passes_per_update=2, optimizer_updates_per_bag=1,
                survival_target_shape=[int(n_patients),2])


def patch_cache_suffix(cfg):
    fmt = cfg.get('storage_format', 'png')
    if fmt not in ('png', 'jpeg'):
        raise ValueError('patch.storage_format must be png or jpeg.')
    return '.jpg' if fmt == 'jpeg' else '.png'


def save_rgb_patch(rgb, path, cfg):
    """JPEG is explicit lossy storage, not feature extraction; PNG is lossless."""
    rgb = np.asarray(rgb)
    if rgb.shape != (128,128,3) or rgb.dtype != np.uint8:
        raise ValueError('Expected an actual RGB uint8 tissue patch.')
    path = Path(path)
    fmt = cfg.get('storage_format', 'png')
    expected = patch_cache_suffix(cfg)
    if path.suffix != expected:
        raise ValueError('Patch filename and requested storage format differ.')
    tmp = path.with_name(path.stem + '.tmp' + expected)
    if fmt == 'jpeg':
        quality = int(cfg.get('jpeg_quality',95))
        if not 1 <= quality <= 100:
            raise ValueError('jpeg_quality must be in [1,100].')
        Image.fromarray(rgb).save(tmp, format='JPEG', quality=quality, subsampling=0)
    else:
        Image.fromarray(rgb).save(tmp, format='PNG', compress_level=3)
    os.replace(tmp,path)


In [39]:
def four_view(image, variant):
    """Reproduce source's original/20degree/(5,5)translation/3x3blur set.
    On omics this is experimental representation perturbation, not known label invariance.
    """
    image = np.asarray(image, dtype='float32')
    h, w = image.shape[:2]
    if variant == 0:
        return image.copy()
    if variant == 1:
        mat = cv2.getRotationMatrix2D((w / 2, h / 2), 20.0, 1.0)
        return cv2.warpAffine(image, mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if variant == 2:
        return cv2.warpAffine(image, np.float32([[1, 0, 5], [0, 1, 5]]), (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if variant == 3:
        return cv2.GaussianBlur(image, (3, 3), 0)
    raise ValueError('variant must be 0,1,2,3')


def load_rgb(path):
    with Image.open(path) as im:
        x = np.asarray(im.convert('RGB'), dtype='float32') / 255.0
    if x.shape != (128, 128, 3):
        raise ValueError(f'Patch has shape {x.shape}, expected (128,128,3): {path}')
    return x


class PatientBatches:

    def __init__(self, cohort, patches, maps, ids, config):
        self.cohort = cohort.set_index('patient_id')
        self.ids = list(ids)
        self.config = config
        self.maps = {pid: maps[j] for j, pid in enumerate(ids)}
        self.paths = {pid: sorted(patches.loc[patches.patient_id == pid, 'path'].tolist()) for pid in ids}
        if any((not self.paths[pid] for pid in ids)):
            raise ValueError('A patient has no patches.')
        if len(ids) != len(set(ids)):
            raise ValueError('Duplicate patients in a split.')
        if maps.shape != (len(ids), 128, 128, 3):
            raise ValueError(f'Unexpected omics map shape: {maps.shape}')

    def iter_epoch(self, epoch, start_batch=0):
        if start_batch < 0:
            raise ValueError('start_batch must be nonnegative.')
        cfg = self.config
        rng = np.random.default_rng(stable_seed(cfg['seed'], epoch, 'patient_order'))
        ordered = np.asarray(self.ids)[rng.permutation(len(self.ids))]
        b = cfg['patient_batch_size']
        k = cfg['patches_per_patient_step']
        if b < 1 or k < 1:
            raise ValueError('Patient and patch batch sizes must be positive.')
        views = 4 if cfg['wsi_augmentation'] == 'four_views' or cfg['omics_augmentation'] == 'legacy_four_views' else 1
        if cfg['wsi_augmentation'] not in ('none', 'four_views'):
            raise ValueError('wsi_augmentation must be none or four_views.')
        if cfg['omics_augmentation'] not in ('none', 'legacy_four_views'):
            raise ValueError('omics_augmentation must be none or legacy_four_views.')
        for start in range(0, len(ordered), b):
            if (start // b + 1) * views <= start_batch:
                continue
            group = ordered[start:start + b].tolist()
            selected = {}
            for pid in group:
                prng = np.random.default_rng(stable_seed(cfg['seed'], epoch, pid))
                paths = self.paths[pid]
                ix = coverage_patch_indices(len(paths), k, epoch, cfg['seed'], pid, cfg.get('patch_sampling','random'))
                selected[pid] = [paths[i] for i in ix]
            for variant in range(views):
                if start // b * views + variant < start_batch:
                    continue
                wsi, omics, segments, owners = ([], [], [], [])
                for patient_index, pid in enumerate(group):
                    for path in selected[pid]:
                        wsi_variant = variant if cfg['wsi_augmentation'] == 'four_views' else 0
                        wsi.append(four_view(load_rgb(path), wsi_variant))
                        omap = self.maps[pid]
                        if cfg['omics_augmentation'] == 'legacy_four_views':
                            omap = four_view(omap, variant)
                        omics.append(omap)
                        segments.append(patient_index)
                        owners.append(pid)
                labels = self.cohort.loc[group, ['time_days', 'event']].to_numpy(dtype='float64')
                yield (np.stack(wsi), np.stack(omics), np.asarray(segments, 'int32'), labels, group, owners)

## 8. Your CURE-derived model — full visible implementation
The external inputs are WSI `(nb,128,128,3)` and omics `(nb,128,128,3)`.
Internally the second input is split into RNA, CNV and mutation streams, giving your backbone four same-patient inputs.
The 64→128→256→512 progression and EMRC/RGSA, PCMFA/MACFusion, EHF, SIR and LF paths are retained.

Engineering adjustments: serializable layers instead of Lambda closures, smooth norm floors, bounded exponentiation,
real arithmetic for complex squared magnitudes, and normal training/inference dropout.
The source's nonstandard Poincare/Lorentz formulas are retained as legacy attention operations;
this notebook does not assert their mathematical manifold interpretation is correct.

In [40]:
@register(package='CURE')
class ChannelSlice(L.Layer):

    def __init__(self, channel, **kwargs):
        super().__init__(**kwargs)
        self.channel = int(channel)

    def call(self, x):
        return x[..., self.channel:self.channel + 1]

    def get_config(self):
        return dict(super().get_config(), channel=self.channel)


@register(package='CURE')
class ChannelShuffle(L.Layer):

    def __init__(self, groups=8, **kwargs):
        super().__init__(**kwargs)
        self.groups = int(groups)

    def build(self, shape):
        if shape[-1] is None or int(shape[-1]) % self.groups:
            raise ValueError(f'Channel shuffle needs channels divisible by {self.groups}: {shape}')
        self.channels = int(shape[-1])
        super().build(shape)

    def call(self, x):
        s = ops.shape(x)
        y = ops.reshape(x, (-1, s[1], s[2], self.groups, self.channels // self.groups))
        return ops.reshape(ops.transpose(y, (0, 1, 2, 4, 3)), (-1, s[1], s[2], self.channels))

    def compute_output_shape(self, s):
        return s

    def get_config(self):
        return dict(super().get_config(), groups=self.groups)

### 8.1 MACFusion / PCMFA attention core

In [41]:
@register(package='CURE')
class MACFusion(L.Layer):

    def __init__(self, units, fractal_dim=1.8, learnable_curvature=True, epsilon=1e-06, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.fractal_dim = float(fractal_dim)
        self.learnable_curvature = bool(learnable_curvature)
        self.epsilon = float(epsilon)

    def build(self, shapes):
        a, b = shapes
        if len(a) != 4 or len(b) != 4 or a[-1] != b[-1] or (a[-1] != self.units) or (a[1:3] != b[1:3]):
            raise ValueError(f'MACFusion requires matching 4D maps with channels=units; got {shapes}.')
        c = int(a[-1])
        self.feature_dim = c
        self.psi_real = self.add_weight(name='psi_real', shape=(c, self.units), initializer='glorot_uniform')
        self.psi_imag = self.add_weight(name='psi_imag', shape=(c, self.units), initializer='glorot_uniform')
        self.S = self.add_weight(name='fractal_scaling', shape=(self.units,), initializer=keras.initializers.Constant(self.fractal_dim), constraint=keras.constraints.MinMaxNorm(1.0, 2.0))
        self.curvature_log = self.add_weight(name='curvature_log', shape=(1,), initializer='he_normal', trainable=self.learnable_curvature)
        self.lorentz_alpha = self.add_weight(name='lorentz_scale', shape=(1,), initializer='he_normal')
        self.hyper_beta = self.add_weight(name='beta_scale', shape=(c,), initializer='he_normal')
        self.hyper_gate = self.add_weight(name='hyper_gate', shape=(1,), initializer='ones')
        self.hyper_gate1 = self.add_weight(name='hyper_gate1', shape=(1,), initializer='ones')
        self.cross_gate = self.add_weight(name='cross_gate', shape=(c,), initializer='ones')
        self.output_gate = self.add_weight(name='output_gate', shape=(c,), initializer='ones')
        i = np.arange(c, dtype=float)[:, None]
        k = np.arange(c, dtype=float)[None, :]
        dct = np.cos(np.pi / c * (i + 0.5) * k) * np.sqrt(2 / c)
        dct[:, 0] /= np.sqrt(2)
        self.dct = self.add_weight(name='orthonormal_dct_ii', shape=(c, c), trainable=False, initializer=keras.initializers.Constant(dct.astype('float32')))
        super().build(shapes)

    def _norm(self, x):
        return ops.sqrt(ops.sum(ops.square(x), axis=-1, keepdims=True) + self.epsilon ** 2)

    def _lorentz(self, a, b, c):
        ta = ops.sqrt(1 + c * ops.sum(a * a, axis=-1, keepdims=True)) / c
        tb = ops.sqrt(1 + c * ops.sum(b * b, axis=-1, keepdims=True)) / c
        return (ops.concatenate((tb, a), axis=-1), ops.concatenate((ta, b), axis=-1))

    def _poincare_legacy(self, a, c):
        n = self._norm(a)
        scale = ops.tanh(ops.sqrt(c) * n) * (a / (n + self.epsilon))
        return a * scale

    def call(self, inputs):
        a, b = inputs
        fa = ops.matmul(ops.mean(a, axis=(1, 2)), self.dct)
        fb = ops.matmul(ops.mean(b, axis=(1, 2)), self.dct)
        c = ops.clip(ops.exp(ops.clip(self.curvature_log, -20.0, 20.0)), 0.1, 10.0)
        adj_s = self.S * ops.sigmoid(c)
        adj_c = c * ops.mean(ops.sigmoid(self.S))
        ar, ai = (ops.matmul(fa, self.psi_real), ops.matmul(fa, self.psi_imag))
        br, bi = (ops.matmul(fb, self.psi_real), ops.matmul(fb, self.psi_imag))
        qa = ops.softmax((ar * ar + ai * ai) * ops.power(self._norm(ar), adj_s - 2), axis=-1)
        qb = ops.softmax((br * br + bi * bi) * ops.power(self._norm(br), adj_s - 2), axis=-1)
        ha, hb = self._lorentz(fa, fb, adj_c)
        ba = ops.matmul(qa, ops.transpose(self.psi_real))
        bb = ops.matmul(qb, ops.transpose(self.psi_real))
        la = self.lorentz_alpha * (-ha[:, :1] ** 2 + ops.sum((ha[:, 1:] + ba) ** 2 * self.hyper_beta, axis=-1, keepdims=True))
        lb = self.lorentz_alpha * (-hb[:, :1] ** 2 + ops.sum((hb[:, 1:] + bb) ** 2 * self.hyper_beta, axis=-1, keepdims=True))
        pa = ops.sum(self._poincare_legacy(fa, adj_c) * fa, axis=-1, keepdims=True)
        pb = ops.sum(self._poincare_legacy(fb, adj_c) * fb, axis=-1, keepdims=True)
        ga = ops.sigmoid(self.hyper_gate * la + self.hyper_gate1 * pa)
        gb = ops.sigmoid(self.hyper_gate * lb + self.hyper_gate1 * pb)
        qa = qa[:, None, None, :]
        qb = qb[:, None, None, :]
        ga = ga[:, None, None, :]
        gb = gb[:, None, None, :]
        gate = (self.cross_gate * self.output_gate)[None, None, None, :]
        return (a * gate * (qb + gb), b * gate * (qa + ga))

    def compute_output_shape(self, shapes):
        return shapes

    def get_config(self):
        return dict(super().get_config(), units=self.units, fractal_dim=self.fractal_dim, learnable_curvature=self.learnable_curvature, epsilon=self.epsilon)

### 8.2 Multikernel residual convolution and learned fusion

In [42]:
@register(package='CURE')
class LF(L.Layer):

    def build(self, shapes):
        self.scale1 = self.add_weight(name='global_scale1', shape=(1, 1, 1, 1), initializer='ones')
        self.scale2 = self.add_weight(name='global_scale2', shape=(1, 1, 1, 1), initializer='ones')
        super().build(shapes)

    def call(self, inputs):
        return ops.concatenate((inputs[0] * self.scale1, inputs[1] * self.scale2), axis=-1)

    def compute_output_shape(self, shapes):
        return tuple(shapes[0][:-1]) + (shapes[0][-1] + shapes[1][-1],)


def multi_kernel_groupwise_conv1(x, filters, groups=8, strides=2):
    paths = [L.Conv2D(filters // 4, 1, groups=groups, padding='same')(x), L.DepthwiseConv2D(3, padding='same')(x), L.DepthwiseConv2D(5, padding='same')(x), L.DepthwiseConv2D(3, dilation_rate=2, padding='same')(x)]
    z = ChannelShuffle(groups)(L.Concatenate()(paths))
    z = L.DepthwiseConv2D(3, padding='same')(z)
    z = L.Conv2D(filters, 1, groups=groups, padding='same')(z)
    z = L.DepthwiseConv2D(3, strides=strides, padding='same')(z)
    shortcut = L.Conv2D(filters, 1, groups=groups, padding='same')(x)
    shortcut = L.DepthwiseConv2D(3, strides=strides, padding='same')(shortcut)
    return L.Activation('gelu')(L.Add()([z, shortcut]))


def multi_kernel_groupwise_conv3(x, filters, groups=8, strides=2):
    p1 = L.BatchNormalization()(L.Conv2D(filters, 1, strides=strides, groups=groups, padding='same', use_bias=False)(x))
    p3 = L.BatchNormalization()(L.DepthwiseConv2D(3, strides=strides, padding='same', use_bias=False)(x))
    p5 = L.BatchNormalization()(L.DepthwiseConv2D(5, strides=strides, padding='same', use_bias=False)(x))
    z = ChannelShuffle(groups)(L.Concatenate()([p1, p3, p5]))
    z = L.BatchNormalization()(L.Conv2D(filters, 1, groups=groups, padding='same', use_bias=False)(z))
    shortcut = L.BatchNormalization()(L.Conv2D(filters, 1, strides=strides, groups=groups, padding='same', use_bias=False)(x))
    return L.Activation('gelu')(L.Add()([z, shortcut]))


def RGSA(x, filters, strides=1, use_projection=True):
    shortcut = x
    z = multi_kernel_groupwise_conv3(x, filters, strides=strides)
    z = L.Activation('relu')(L.BatchNormalization()(z))
    z = multi_kernel_groupwise_conv3(z, filters, strides=1)
    z = L.BatchNormalization()(L.Conv2D(filters, 1, padding='same')(z))
    stride_tuple = (strides, strides) if isinstance(strides, int) else tuple(strides)
    if use_projection or stride_tuple != (1, 1) or x.shape[-1] != filters:
        shortcut = L.BatchNormalization()(L.Conv2D(filters, 1, strides=strides, groups=8, padding='same')(shortcut))
    return L.Activation('relu')(L.Add()([z, shortcut]))


def EMRC(a, b, filters, strides):
    return (RGSA(a, filters, strides), RGSA(b, filters, strides))

### 8.3 EHF and SIR cross-modal paths

In [43]:
def PCMFA(a, b, f):
    a, b = MACFusion(f)([a, b])
    return MACFusion(f)([a, b])


def EHF1(a, b, f, s):
    a, b = EMRC(a, b, f, s)
    return PCMFA(a, b, f)


def SIR(a, b, f):
    if f not in (64, 128, 256, 512):
        raise ValueError(f'Unsupported SIR width {f}')
    widths = {64: [128, 256, 512], 128: [256, 512], 256: [512], 512: [512]}[f]
    for width in widths:
        s = 1 if f == 512 else 2
        a = multi_kernel_groupwise_conv1(a, width, strides=s)
        b = multi_kernel_groupwise_conv1(b, width, strides=s)
    return (a, b)


def EHF2(a, b, f, s):
    a, b = EMRC(a, b, f, s)
    a, b = PCMFA(L.BatchNormalization()(a), L.BatchNormalization()(b), f)
    sa, sb = SIR(a, b, f)
    merged = L.Conv2D(f, 1, groups=8, padding='same')(LF()([a, b]))
    return (merged, sa, sb)


def EHF(a, b, filters, strides1=1, strides2=2):
    if filters == 64:
        a, b = EHF1(a, b, 64, strides2)
        return EHF1(a, b, 64, strides2)
    a, b = EHF1(a, b, filters, strides1)
    return EHF2(a, b, filters, strides2)


def _stem(x):
    x = L.Conv2D(64, 7, strides=2, padding='same')(x)
    x = L.Activation('relu')(L.BatchNormalization()(x))
    return L.MaxPooling2D(3, strides=2, padding='same')(x)

### 8.4 Assemble the two external image inputs into the four-stream CURE model

In [44]:
def build_patch_encoder(freeze_batch_norm=False):
    """Both external inputs are (nb,128,128,3); all CURE fusion occurs BEFORE pooling."""
    wsi = L.Input((128, 128, 3), name='wsi_rgb')
    omics = L.Input((128, 128, 3), name='omics_rna_cnv_mut')
    rna = ChannelSlice(0, name='rna_channel')(omics)
    cnv = ChannelSlice(1, name='cnv_channel')(omics)
    mutation = ChannelSlice(2, name='mutation_channel')(omics)
    a, b = EHF(_stem(wsi), _stem(rna), 64, strides1=1, strides2=1)
    a, s11, s12 = EHF(a, b, 128)
    b = multi_kernel_groupwise_conv3(_stem(cnv), 256, strides=2)
    a, s21, s22 = EHF(a, b, 256)
    b = multi_kernel_groupwise_conv3(_stem(mutation), 512, strides=2)
    b = multi_kernel_groupwise_conv3(b, 512, strides=2)
    a, s31, s32 = EHF(a, b, 512)
    fused = L.Concatenate(name='cure_final_spatial_fusion')([a, s11, s21, s31, s12, s22, s32])
    fused = L.Dropout(0.25, seed=2026, name='fusion_dropout')(fused)
    embedding = L.GlobalAveragePooling2D(name='post_fusion_patch_embedding')(fused)
    model = keras.Model([wsi, omics], embedding, name='CURE_BRCA_two_input_four_stream_encoder')
    if freeze_batch_norm:
        for layer in model.layers:
            if isinstance(layer, L.BatchNormalization):
                layer.trainable = False
    return model


def build_patient_heads(embedding_dim, n_intervals):
    h = L.Input((embedding_dim,), name='fused_patch_embedding')
    score = L.Dense(1, name='attention_score')(L.Dense(128, activation='tanh')(h))
    attention = keras.Model(h, score, name='post_fusion_patient_attention')
    p = L.Input((embedding_dim,), name='patient_embedding')
    z = L.Dense(128, activation='relu')(p)
    z = L.Dropout(0.25, seed=2027)(z)
    logits = L.Dense(n_intervals, bias_initializer=keras.initializers.Constant(-3.0), name='log_rate_parameters')(z)
    head = keras.Model(p, logits, name='piecewise_exponential_survival_head')
    return (attention, head)

## 9. Patient-level survival objective and training code
Fusion is performed on spatial maps **before** spatial pooling and patient aggregation.
The survival head is the explicitly documented adaptation: continuous-time piecewise-exponential likelihood,
not the original minibatch cumulative-sum Cox loss. `event=1` is an observed death; `event=0` is right censoring.
Each patient contributes one survival loss per bag, not one independently labelled survival subject per patch.

All fold selection uses validation NLL. Held-out tests are evaluated only after checkpoint selection.
A five-year cumulative-hazard risk score is a configured ranking score; calibration and clinical utility are not established.

For large bags the active trainer is `make_large_bag_train_step` below. `make_train_step` remains a small-batch
reference implementation, not the active full-cohort trainer. Do not average independent four-patch survival losses.

In [45]:
def fit_intervals(time_days, event, n_intervals=4):
    t = np.asarray(time_days, float) / 365.25
    e = np.asarray(event, float)
    if len(t) != len(e) or not np.isfinite(t).all() or np.any(t <= 0) or (not np.isin(e, [0, 1]).all()):
        raise ValueError('Invalid survival targets.')
    failures = t[e == 1]
    if len(failures) < 2:
        raise ValueError('At least two observed training events are required.')
    if int(n_intervals) < 1:
        raise ValueError('n_intervals must be positive.')
    internal = np.quantile(failures, np.arange(1, n_intervals) / n_intervals)
    starts = np.unique(np.r_[0.0, internal])
    return starts


def exposures(time_years, starts):
    t = np.asarray(time_years, float).reshape(-1, 1)
    starts = np.asarray(starts, float)
    if len(starts) == 0 or starts[0] != 0 or np.any(np.diff(starts) <= 0):
        raise ValueError('Intervals must start at zero and increase strictly.')
    if not np.isfinite(t).all() or np.any(t < 0):
        raise ValueError('Invalid time.')
    ends = np.r_[starts[1:], np.inf]
    return np.maximum(0, np.minimum(t, ends) - starts)


def pe_nll_numpy(time_years, event, rates, starts):
    rates = np.asarray(rates, float)
    t = np.asarray(time_years, float)
    e = np.asarray(event, float)
    if rates.shape != (len(t), len(starts)) or np.any(rates <= 0) or (not np.isfinite(rates).all()):
        raise ValueError('Rates must be finite positive and shape (patients,intervals).')
    idx = np.searchsorted(starts, t, side='right') - 1
    return (exposures(t, starts) * rates).sum(1) - e * np.log(rates[np.arange(len(t)), idx])


def survival_probabilities(rates, time_grid_years, starts):
    return np.exp(-np.asarray(rates) @ exposures(time_grid_years, starts).T)


def harrell_c_index(time, event, risk):
    """Higher risk = earlier death. Same-time event/censor pair is comparable;
    two events at exactly the same time are not. Tied risks score one half.
    Returns NaN if there are no comparable pairs, never a fabricated 0.5.
    """
    t, e, r = (np.asarray(time, float), np.asarray(event, float), np.asarray(risk, float))
    if not len(t) == len(e) == len(r) or not np.isfinite(t).all() or (not np.isfinite(r).all()):
        raise ValueError('Invalid arrays for C-index.')
    if np.any(t <= 0) or not np.isin(e, [0, 1]).all():
        raise ValueError('Invalid survival labels.')
    concordant, comparable = (0.0, 0)
    for i in range(len(t)):
        if not e[i]:
            continue
        mask = (t > t[i]) | (t == t[i]) & (e == 0)
        d = r[i] - r[mask]
        comparable += len(d)
        concordant += float((d > 0).sum() + 0.5 * (d == 0).sum())
    return concordant / comparable if comparable else float('nan')


def c_index_bootstrap(time, event, risk, repeats=1000, seed=2026):
    t, e, r = map(np.asarray, (time, event, risk))
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(repeats):
        ix = rng.integers(0, len(t), len(t))
        v = harrell_c_index(t[ix], e[ix], r[ix])
        if np.isfinite(v):
            vals.append(v)
    return dict(lower=float(np.quantile(vals, 0.025)) if vals else None, upper=float(np.quantile(vals, 0.975)) if vals else None, valid_replicates=len(vals), requested_replicates=repeats, note='Patient bootstrap conditional on the fitted model; not retraining uncertainty.')


class StreamingAttention:
    """Exact, numerically stable softmax pooling of post-fusion embeddings in chunks."""

    def __init__(self):
        self.m = -np.inf
        self.denom = 0.0
        self.numer = None

    def update(self, embeddings, scores):
        h = np.asarray(embeddings, np.float64)
        s = np.asarray(scores, np.float64).reshape(-1)
        if h.ndim != 2 or len(h) != len(s) or (not len(h)):
            raise ValueError('Invalid pooling chunk.')
        if not np.isfinite(h).all() or not np.isfinite(s).all():
            raise FloatingPointError('Non-finite embeddings.')
        m = max(self.m, float(s.max()))
        old = np.exp(self.m - m) if np.isfinite(self.m) else 0.0
        weights = np.exp(s - m)
        numerator = (weights[:, None] * h).sum(0)
        self.numer = numerator if self.numer is None else self.numer * old + numerator
        self.denom = self.denom * old + weights.sum()
        self.m = m

    def value(self):
        if self.numer is None or self.denom <= 0:
            raise ValueError('No patches to pool.')
        return (self.numer / self.denom).astype('float32')

In [46]:
def initialize(seed, deterministic=True):
    keras.utils.set_random_seed(seed)
    if deterministic:
        tf.config.experimental.enable_op_determinism()
    for gpu in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    keras.mixed_precision.set_global_policy('float32')


def segment_attention(embeddings, scores, segments, n_patients):
    scores = tf.reshape(scores, [-1])
    count = tf.math.unsorted_segment_sum(tf.ones_like(scores), segments, n_patients)
    tf.debugging.assert_positive(count, message='A patient has no patches in this batch.')
    max_s = tf.math.unsorted_segment_max(scores, segments, n_patients)
    weights = tf.exp(scores - tf.gather(max_s, segments))
    denominator = tf.math.unsorted_segment_sum(weights, segments, n_patients)
    numerator = tf.math.unsorted_segment_sum(embeddings * weights[:, None], segments, n_patients)
    return numerator / denominator[:, None]


def pe_nll_tf(labels, logits, starts):
    labels = tf.cast(labels, tf.float64)
    times = labels[:, 0] / 365.25
    events = tf.cast(labels[:, 1], tf.float32)
    starts = tf.cast(starts, tf.float64)
    ends = tf.concat([starts[1:], tf.constant([np.inf], tf.float64)], axis=0)
    exposure = tf.maximum(tf.constant(0.0, tf.float64), tf.minimum(times[:, None], ends[None, :]) - starts[None, :])
    rates = tf.nn.softplus(tf.cast(logits, tf.float32)) + 1e-06
    bins = tf.searchsorted(starts, times, side='right') - 1
    chosen = tf.gather(rates, bins, axis=1, batch_dims=1)
    return tf.reduce_sum(tf.cast(exposure, tf.float32) * rates, axis=1) - events * tf.math.log(chosen)


def make_train_step(encoder, attention, head, optimizer, starts, clip_norm=1.0):
    variables = encoder.trainable_variables + attention.trainable_variables + head.trainable_variables
    if not optimizer.built:
        optimizer.build(variables)
    starts = tf.constant(starts, dtype=tf.float64)

    @tf.function(input_signature=[tf.TensorSpec([None, 128, 128, 3], tf.float32), tf.TensorSpec([None, 128, 128, 3], tf.float32), tf.TensorSpec([None], tf.int32), tf.TensorSpec([None, 2], tf.float64)])
    def step(wsi, omics, segments, labels):
        tf.debugging.assert_equal(tf.shape(wsi)[0], tf.shape(omics)[0])
        tf.debugging.assert_equal(tf.shape(segments)[0], tf.shape(wsi)[0])
        tf.debugging.assert_all_finite(wsi, 'Non-finite WSI input')
        tf.debugging.assert_all_finite(omics, 'Non-finite molecular input')
        tf.debugging.assert_positive(labels[:, 0])
        tf.debugging.assert_greater_equal(labels[:, 1], tf.constant(0.0, tf.float64))
        tf.debugging.assert_less_equal(labels[:, 1], tf.constant(1.0, tf.float64))
        tf.debugging.assert_equal(labels[:, 1], tf.round(labels[:, 1]))
        with tf.GradientTape() as tape:
            z = encoder([wsi, omics], training=True)
            scores = attention(z, training=True)
            patient_z = segment_attention(z, scores, segments, tf.shape(labels)[0])
            logits = head(patient_z, training=True)
            loss = tf.reduce_mean(pe_nll_tf(labels, logits, starts))
            extra = encoder.losses + attention.losses + head.losses
            if extra:
                loss += tf.add_n(extra)
        tf.debugging.assert_all_finite(loss, 'Non-finite loss; do not continue training.')
        gradients = tape.gradient(loss, variables)
        if any((g is None for g in gradients)):
            absent = [v.name for v, g in zip(variables, gradients) if g is None]
            raise RuntimeError(f'Disconnected trainable weights: {absent}')
        for g in gradients:
            tf.debugging.assert_all_finite(g, 'Non-finite gradient; training stopped.')
        gradients, norm = tf.clip_by_global_norm(gradients, clip_norm)
        optimizer.apply_gradients(zip(gradients, variables))
        return (loss, norm)
    return step

In [47]:
def patient_predictions(encoder, attention, head, loader, ids, starts, horizon, chunk_size=8, max_patches=None, deadline_unix=None, cache_dir=None):
    if chunk_size < 1:
        raise ValueError('evaluation_chunk_size must be positive.')
    rows = []
    rates_all = []
    cache_dir = Path(cache_dir) if cache_dir is not None else None
    if cache_dir is not None:
        cache_dir.mkdir(parents=True, exist_ok=True)
    for patient_number, pid in enumerate(ids):
        check_deadline(deadline_unix)
        cached = cache_dir / f'{pid}.json' if cache_dir is not None else None
        if cached is not None and cached.exists():
            item = json.loads(cached.read_text())
            rows.append(item['row'])
            rates_all.append(np.asarray(item['rates'], dtype='float32'))
            continue
        paths = loader.paths[pid]
        if max_patches and len(paths) > max_patches:
            ix = np.random.default_rng(stable_seed('validation', pid)).choice(len(paths), max_patches, replace=False)
            paths = [paths[i] for i in sorted(ix)]
        pool = StreamingAttention()
        for start in range(0, len(paths), chunk_size):
            check_deadline(deadline_unix)
            xx = np.stack([load_rgb(p) for p in paths[start:start + chunk_size]])
            oo = np.repeat(loader.maps[pid][None, ...], len(xx), axis=0)
            zz = encoder([xx, oo], training=False)
            scores = attention(zz, training=False)
            pool.update(zz.numpy(), scores.numpy())
        logits = head(pool.value()[None, :], training=False)
        rates = (tf.nn.softplus(logits) + 1e-06).numpy()[0]
        if not np.isfinite(rates).all():
            raise FloatingPointError(f'Invalid survival rates for {pid}')
        risk = float(np.dot(rates, exposures([horizon], starts)[0]))
        label = loader.cohort.loc[pid]
        rows.append(dict(patient_id=pid, time_days=float(label.time_days), event=int(label.event), risk=risk, survival_probability_at_horizon=float(np.exp(-risk)), risk_horizon_years=horizon, patch_count=len(paths)))
        rates_all.append(rates)
        if cached is not None:
            atomic_json(cached, dict(row=rows[-1], rates=rates.tolist()))
        if (patient_number + 1) % 10 == 0 or patient_number + 1 == len(ids):
            print(f'Patient evaluation: {patient_number + 1}/{len(ids)}', flush=True)
    return (pd.DataFrame(rows), np.stack(rates_all))


def _safe_float(x):
    return float(x) if np.isfinite(x) else None


def _save_components(directory, encoder, attention, head):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for name, model in [('patch_encoder', encoder), ('attention', attention), ('survival_head', head)]:
        temp = directory / f'{name}.tmp.keras'
        model.save(temp)
        os.replace(temp, directory / f'{name}.keras')

### 9.1 Memory-bounded whole-patient gradient replay
1. Encode all selected pairs in small chunks and cache only their fused embeddings.
2. Calculate one survival loss for the complete patient bags.
3. Replay each encoder chunk with its embedding gradient, preserving RNG/BatchNorm state.
4. Sum the encoder gradients and apply ONE optimizer update.

This is **not** independent patch loss averaging. All selected patches affect the patient attention.
It trades extra forward computation for a smaller activation-memory requirement. Encoder BatchNorm
uses the actual chunk size, not the total bag size; changing chunk size changes this normalization context.


In [48]:
def make_large_bag_train_step(encoder, attention, head, optimizer, starts,
                              microbatch_size=4, clip_norm=1.0, deadline_unix=None):
    """One patient-bag objective/update with all selected patches, bounded encoder tape.

    Pass 1 caches detached post-fusion embeddings. Compute the patient survival
    loss ONCE on the whole set, then replay each encoder chunk with dL/dz.
    BatchNorm state and Keras Dropout RNG are restored before every replay;
    final state is committed only once. Trainable encoder weights are NOT frozen.

    Equivalence target: sequential training-mode microbatch encoder calls followed
    by one global patient aggregation/loss. NOT a monolithic large-BatchNorm call.
    Keras-managed RNG only; replay mismatches cause failure, never silent training.
    """
    if keras.backend.backend() != 'tensorflow':
        raise RuntimeError('The production large-bag trainer requires TensorFlow. No backend fallback.')
    if not isinstance(microbatch_size,int) or microbatch_size < 1:
        raise ValueError('microbatch_size must be a positive integer.')
    if clip_norm <= 0:
        raise ValueError('clip_norm must be positive.')
    if any(isinstance(layer,L.Dropout) and layer.seed is None for layer in encoder.layers):
        raise ValueError('Use explicit Keras Dropout seeds for replayable masks.')
    enc_vars = list(encoder.trainable_variables)
    top_vars = list(attention.trainable_variables) + list(head.trainable_variables)
    variables = enc_vars + top_vars
    if not variables or len({id(v) for v in variables}) != len(variables):
        raise ValueError('Build separate encoder/attention/head models first.')
    if not optimizer.built:
        optimizer.build(variables)
    starts_tensor = tf.constant(starts, dtype=tf.float64)

    @tf.function(input_signature=[tf.TensorSpec([None,128,128,3],tf.float32),
                                  tf.TensorSpec([None,128,128,3],tf.float32)])
    def cache_forward(w, o):
        return encoder([w,o], training=True)

    @tf.function(input_signature=[tf.TensorSpec([None,128,128,3],tf.float32),
                                  tf.TensorSpec([None,128,128,3],tf.float32),
                                  tf.TensorSpec([None,encoder.output_shape[-1]],tf.float32)])
    def replay_vjp(w, o, upstream):
        with tf.GradientTape() as tape:
            zz = encoder([w,o], training=True)
        grads = tape.gradient(zz, enc_vars, output_gradients=upstream)
        if any(g is None for g in grads):
            raise RuntimeError('A CURE encoder weight is disconnected during replay.')
        return zz, grads

    def step(wsi, omics, segments, labels):
        # Remain on the host: do not convert the entire RGB bag to GPU tensors.
        wsi, omics = np.asarray(wsi,dtype='float32'), np.asarray(omics,dtype='float32')
        segments = np.asarray(segments)
        labels = np.asarray(labels,dtype='float64')
        n = len(wsi)
        if n < 1 or wsi.shape != omics.shape or wsi.shape[1:] != (128,128,3):
            raise ValueError('Paired WSI/omics bags must both have shape (Q,128,128,3).')
        if labels.ndim != 2 or labels.shape[1] != 2 or len(labels) < 1:
            raise ValueError('Patient labels must have shape (B,2).')
        if segments.shape != (n,) or segments.dtype.kind not in 'iu':
            raise ValueError('One integer owner index is required per patch.')
        if np.any(segments<0) or np.any(segments>=len(labels)):
            raise ValueError('Patient owner index out of range.')
        if not np.all(np.bincount(segments,minlength=len(labels))>0):
            raise ValueError('Every patient needs at least one patch.')
        if not (np.isfinite(wsi).all() and np.isfinite(omics).all() and np.isfinite(labels).all()):
            raise FloatingPointError('Nonfinite actual input or label.')
        if np.any(labels[:,0]<=0) or not np.isin(labels[:,1],[0,1]).all():
            raise ValueError('Labels require positive observed time and event in {0,1}.')
        # CURE has no encoder activity/weight regularizer. Do not silently omit one.
        if encoder.losses:
            raise ValueError('Encoder regularizers need an explicitly derived replay term; unsupported here.')
        initial = [snapshot_replay_state(m) for m in (encoder,attention,head)]
        states, embeddings = [], []
        ranges = [(s,min(s+microbatch_size,n)) for s in range(0,n,microbatch_size)]
        committed = False
        try:
            for s,e in ranges:
                check_deadline(deadline_unix)
                states.append(snapshot_replay_state(encoder))
                z = cache_forward(wsi[s:e],omics[s:e])
                tf.debugging.assert_all_finite(z,'Nonfinite cached fused embedding.')
                embeddings.append(z.numpy())
                if encoder.losses:
                    raise ValueError('Activity regularization detected; not supported by this replay rule.')
            final_encoder_state = snapshot_replay_state(encoder)
            z_host = np.concatenate(embeddings,axis=0)
            z_full = tf.convert_to_tensor(z_host,dtype=tf.float32)
            seg = tf.convert_to_tensor(segments,dtype=tf.int32)
            lab = tf.convert_to_tensor(labels,dtype=tf.float64)
            with tf.GradientTape() as tape:
                tape.watch(z_full)
                patient_z = segment_attention(z_full,attention(z_full,training=True),seg,len(labels))
                logits = head(patient_z,training=True)
                loss = tf.reduce_mean(pe_nll_tf(lab,logits,starts_tensor))
                extra = attention.losses + head.losses
                if extra:
                    loss += tf.add_n(extra)
            derivatives = tape.gradient(loss,[z_full]+top_vars)
            if any(g is None for g in derivatives):
                raise RuntimeError('Disconnected patient-level derivative in the whole-bag loss.')
            tf.debugging.assert_all_finite(loss,'Nonfinite patient-level survival loss.')
            upstream = derivatives[0].numpy()
            accumulated = [tf.zeros_like(v) for v in enc_vars]
            max_replay_difference = 0.0
            for index,(s,e) in enumerate(ranges):
                check_deadline(deadline_unix)
                restore_replay_state(encoder,states[index])
                replayed,grads = replay_vjp(wsi[s:e],omics[s:e],upstream[s:e])
                actual = replayed.numpy()
                difference = float(np.max(np.abs(actual-z_host[s:e])))
                max_replay_difference = max(max_replay_difference,difference)
                np.testing.assert_allclose(actual,z_host[s:e],rtol=2e-5,atol=2e-5,
                    err_msg='Encoder replay differs: unsupported randomness or nondeterministic operation.')
                for j,g in enumerate(grads):
                    tf.debugging.assert_all_finite(g,'Nonfinite CURE replay gradient.')
                    accumulated[j] = accumulated[j] + g
            # No division by chunk count: dL/dz already carries the patient-mean factor.
            restore_replay_state(encoder,final_encoder_state)
            gradients = accumulated + list(derivatives[1:])
            for g in gradients:
                tf.debugging.assert_all_finite(g,'Nonfinite whole-bag gradient.')
            gradients,norm = tf.clip_by_global_norm(gradients,clip_norm)
            check_deadline(deadline_unix)
            optimizer.apply_gradients(zip(gradients,variables))
            committed = True
            step.last_report = dict(**large_bag_plan(n,len(labels),microbatch_size),
                                    maximum_replay_absolute_difference=max_replay_difference,
                                    gradient_rule='whole_patient_bag_then_encoder_VJP_replay',
                                    batch_norm_scope='encoder_microbatch',
                                    loss=float(loss.numpy()))
            return loss,norm
        finally:
            if not committed:
                for model,state in zip((encoder,attention,head),initial):
                    restore_replay_state(model,state)
    step.last_report = None
    return step


### 9.1 Training loop, optimizer checkpoints and held-out evaluation

In [49]:
def train_fold(cfg, fold, deadline_unix=None):
    """Complete a fold, or checkpoint and raise SessionPause; never score a partial-training run."""
    check_deadline(deadline_unix)
    work = Path(cfg['work_dir'])
    tc = cfg['training']
    splits = prepare_splits(cfg)
    cohort = pd.read_csv(work / 'cohort.csv')
    patches = pd.read_csv(work / 'patches.csv')
    part = splits[splits.fold == fold]
    if part.empty:
        raise ValueError(f'Unknown fold: {fold}')
    ids = {s: sorted(part.loc[part.split == s, 'patient_id'].tolist()) for s in ['train', 'validation', 'test']}
    directory = work / 'folds' / f'fold_{fold}'
    directory.mkdir(parents=True, exist_ok=True)
    execution_only = ('resume', 'checkpoint_every_steps', 'log_every_steps')
    locked_training = {k: v for k, v in tc.items() if k not in execution_only}
    signature = fingerprint(dict(training=locked_training, split=ids, patterns=cfg['feature_patterns'], cohort=sha256_file(work / 'cohort.csv'), patches=sha256_file(work / 'patches.csv'), omics=sha256_file(cfg['omics_path']), source_code=implementation_fingerprint(), id_column=cfg.get('id_column'), id_mapping=sha256_file(cfg['id_mapping_csv']) if cfg.get('id_mapping_csv') else None))
    lock = directory / 'run_signature.json'
    if lock.exists() and json.loads(lock.read_text())['signature'] != signature:
        raise ValueError('Resume contract changed: data/model/split/settings differ. Use a new experiment.')
    if (directory / 'last_checkpoint').exists() and (not tc['resume']):
        raise FileExistsError('Checkpoints exist. Set training.resume=true or use a new work directory.')
    atomic_json(lock, dict(signature=signature))
    if (directory / 'test_metrics.json').exists():
        return json.loads((directory / 'test_metrics.json').read_text())
    blocks, audit = load_omics(cfg['omics_path'], cfg['feature_patterns'], cfg.get('id_column'), cfg.get('id_mapping_csv'))
    transform_path = directory / 'omics_spatializer.json'
    if transform_path.exists():
        mapper = OmicsSpatializer.load(transform_path)
        if mapper.train_ids != ids['train']:
            raise ValueError('Preprocessor training IDs differ.')
    else:
        mapper = OmicsSpatializer(128, tc['max_features_per_modality'], tc['omics_interpolation']).fit(blocks, ids['train'])
        mapper.save(transform_path)
    loaders = {s: PatientBatches(cohort, patches, mapper.transform(blocks, ids[s]), ids[s], tc) for s in ids}
    inspection_batch = next(loaders['train'].iter_epoch(0))
    inspect_training_batch(inspection_batch, cfg, directory / 'input_inspection')
    del inspection_batch
    labels = cohort.set_index('patient_id').loc[ids['train']]
    starts = fit_intervals(labels.time_days, labels.event, tc['survival_intervals'])
    horizon = float(tc['risk_horizon_years'])
    if horizon <= 0:
        raise ValueError('risk_horizon_years must be positive.')
    atomic_json(directory / 'survival_spec.json', dict(starts_years=starts.tolist(), final_interval='unbounded', risk_horizon_years=horizon, time_conversion_days_per_year=365.25, horizon_exceeds_max_training_followup=bool(horizon > labels.time_days.max() / 365.25), objective='continuous-time piecewise-exponential patient likelihood', event_definition='1=observed all-cause death; 0=right censored'))
    atomic_json(directory / 'split_ids.json', ids)
    keras.backend.clear_session()
    initialize(tc['seed'] + fold, tc['deterministic_ops'])
    encoder = build_patch_encoder(tc['freeze_batch_norm'])
    attention, head = build_patient_heads(encoder.output_shape[-1], len(starts))
    optimizer = keras.optimizers.Adam(tc['learning_rate'])
    train_step = make_large_bag_train_step(encoder, attention, head, optimizer, starts,
        microbatch_size=tc.get('encoder_microbatch_size',4), clip_norm=tc['gradient_clip_norm'],
        deadline_unix=deadline_unix)
    epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    batch_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    loss_sum = tf.Variable(0.0, dtype=tf.float64, trainable=False)
    observed_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    norm_var = tf.Variable(0.0, dtype=tf.float64, trainable=False)
    best_var = tf.Variable(np.inf, dtype=tf.float64, trainable=False)
    wait_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    checkpoint = tf.train.Checkpoint(encoder=encoder, attention=attention, head=head, optimizer=optimizer, epoch=epoch_var, batch=batch_var, loss_sum=loss_sum, observed=observed_var, gradient_norm=norm_var, best=best_var, wait=wait_var)
    manager = tf.train.CheckpointManager(checkpoint, str(directory / 'last_checkpoint'), max_to_keep=2)
    if tc['resume'] and manager.latest_checkpoint:
        checkpoint.restore(manager.latest_checkpoint).assert_existing_objects_matched()
        print(f'Resumed fold {fold}: {int(epoch_var.numpy())} completed epochs; next batch {int(batch_var.numpy())}.', flush=True)
    atomic_json(directory / 'environment.json', dict(python=platform.python_version(), tensorflow=tf.__version__, keras=keras.__version__, numpy=np.__version__, gpus=[d.name for d in tf.config.list_physical_devices('GPU')], parameters=encoder.count_params() + attention.count_params() + head.count_params(), precision='float32', distribution='one visible GPU; not multi-GPU data parallel'))
    history_path = directory / 'history.csv'
    history = pd.read_csv(history_path).to_dict('records') if history_path.exists() else []
    history = [r for r in history if r['epoch'] <= int(epoch_var.numpy())]
    every = max(1, int(tc.get('checkpoint_every_steps', 50)))
    log_every = max(1, int(tc.get('log_every_steps', 25)))
    try:
        for epoch in range(int(epoch_var.numpy()), tc['epochs']):
            if int(wait_var.numpy()) >= tc['patience']:
                break
            check_deadline(deadline_unix)
            tic = time.monotonic()
            resume_batch = int(batch_var.numpy())
            for batch_index, batch in enumerate(loaders['train'].iter_epoch(epoch, start_batch=resume_batch), resume_batch):
                check_deadline(deadline_unix)
                w, o, segments, y, group, owners = batch
                if any((owners[j] != group[int(segments[j])] for j in range(len(owners)))):
                    raise AssertionError('Patch/omics patient ownership mismatch.')
                loss, norm = train_step(w, o, segments, y)
                if batch_index == 0 or (batch_index+1) % log_every == 0:
                    atomic_json(directory / 'last_large_bag_step.json', train_step.last_report)
                loss_sum.assign_add(float(loss.numpy()) * len(group))
                observed_var.assign_add(len(group))
                norm_var.assign(max(float(norm_var.numpy()), float(norm.numpy())))
                batch_var.assign(batch_index + 1)
                if (batch_index + 1) % every == 0:
                    manager.save()
                if (batch_index + 1) % log_every == 0:
                    print(f'Fold {fold} epoch {epoch + 1} batch {batch_index + 1}: NLL={float(loss.numpy()):.5f} | paired patches={len(w)} | encoder chunk={tc.get("encoder_microbatch_size",4)}', flush=True)
            manager.save()
            val_cache = directory / 'evaluation_cache' / f'validation_epoch_{epoch + 1}'
            val, vrates = patient_predictions(encoder, attention, head, loaders['validation'], ids['validation'], starts, horizon, tc['evaluation_chunk_size'], tc.get('validation_patches_per_patient', 64), deadline_unix=deadline_unix, cache_dir=val_cache)
            val_loss = float(pe_nll_numpy(val.time_days.to_numpy() / 365.25, val.event.to_numpy(), vrates, starts).mean())
            val_ci = harrell_c_index(val.time_days, val.event, val.risk)
            if not np.isfinite(val_loss):
                raise FloatingPointError('Non-finite validation NLL.')
            improved = val_loss < float(best_var.numpy())
            if improved:
                best_var.assign(val_loss)
                wait_var.assign(0)
                _save_components(directory / 'best', encoder, attention, head)
                atomic_json(directory / 'best_epoch.json', dict(epoch=epoch + 1, validation_nll=val_loss))
            else:
                wait_var.assign_add(1)
            if int(observed_var.numpy()) == 0:
                raise RuntimeError('No patient batches in the training epoch.')
            row = dict(epoch=epoch + 1, train_nll=float(loss_sum.numpy()) / int(observed_var.numpy()), validation_nll=val_loss, validation_c_index=val_ci, max_preclip_gradient_norm=float(norm_var.numpy()), seconds_this_session=time.monotonic() - tic)
            history.append(row)
            atomic_csv(pd.DataFrame(history), history_path)
            epoch_var.assign(epoch + 1)
            batch_var.assign(0)
            loss_sum.assign(0.0)
            observed_var.assign(0)
            norm_var.assign(0.0)
            manager.save()
            atomic_json(directory / 'TRAINING_PROGRESS.json', dict(status='training', completed_epochs=epoch + 1, best_validation_nll=float(best_var.numpy()), patience_used=int(wait_var.numpy())))
            shutil.rmtree(val_cache)
            print(f"Fold {fold} epoch {epoch + 1}: train_NLL={row['train_nll']:.4f} val_NLL={val_loss:.4f} val_C={val_ci:.4f}", flush=True)
            if int(wait_var.numpy()) >= tc['patience']:
                break
    except SessionPause:
        manager.save()
        atomic_json(directory / 'TRAINING_PROGRESS.json', dict(status='paused', completed_epochs=int(epoch_var.numpy()), next_batch=int(batch_var.numpy()), test_evaluated=False))
        raise
    atomic_json(directory / 'TRAINING_PROGRESS.json', dict(status='training_complete_evaluation_pending', completed_epochs=int(epoch_var.numpy()), stopped_by_patience=int(wait_var.numpy()) >= tc['patience']))
    check_deadline(deadline_unix)
    for name in ('patch_encoder', 'attention', 'survival_head'):
        if not (directory / 'best' / f'{name}.keras').is_file():
            raise RuntimeError('No complete best model: random-weight evaluation is prohibited.')
    del train_step, optimizer, manager, checkpoint, encoder, attention, head
    import gc
    gc.collect()
    keras.backend.clear_session()
    encoder = keras.models.load_model(directory / 'best' / 'patch_encoder.keras', compile=False)
    attention = keras.models.load_model(directory / 'best' / 'attention.keras', compile=False)
    head = keras.models.load_model(directory / 'best' / 'survival_head.keras', compile=False)
    test, rates = patient_predictions(encoder, attention, head, loaders['test'], ids['test'], starts, horizon, tc['evaluation_chunk_size'], max_patches=None, deadline_unix=deadline_unix, cache_dir=directory / 'evaluation_cache' / 'held_out_test')
    test['fold'] = fold
    atomic_csv(test, directory / 'test_predictions.csv')
    grid = np.linspace(0, max(horizon, float(labels.time_days.max() / 365.25)), 101)
    probs = survival_probabilities(rates, grid, starts)
    np.savez_compressed(directory / 'test_survival_curves.npz', patient_id=test.patient_id.to_numpy(dtype=str), time_years=grid, survival_probability=probs, interval_rates=rates)
    ci = harrell_c_index(test.time_days, test.event, test.risk)
    result = dict(fold=fold, patients=len(test), events=int(test.event.sum()), c_index=_safe_float(ci), patient_nll=float(pe_nll_numpy(test.time_days.to_numpy() / 365.25, test.event, rates, starts).mean()), risk_horizon_years=horizon, bootstrap=c_index_bootstrap(test.time_days, test.event, test.risk, tc['bootstrap_replicates'], tc['seed'] + fold), status='complete', selected_epoch=json.loads((directory / 'best_epoch.json').read_text())['epoch'])
    atomic_json(directory / 'test_metrics.json', result)
    atomic_json(directory / 'TRAINING_PROGRESS.json', dict(status='complete', completed_epochs=int(epoch_var.numpy()), test_evaluated=True))
    return result


def predict_from_export(fold_dir, omics_blocks, patient_patches, patient_ids, chunk_size=8, time_grid_years=None):
    """Inference on genuine NEW patient data; no outcome labels are required.
    omics_blocks: same named gene columns as training. patient_patches: DataFrame(patient_id,path).
    Returns one risk row and one survival curve per patient.
    """
    directory = Path(fold_dir)
    mapper = OmicsSpatializer.load(directory / 'omics_spatializer.json')
    spec = json.loads((directory / 'survival_spec.json').read_text())
    starts = np.asarray(spec['starts_years'])
    maps = mapper.transform(omics_blocks, patient_ids)
    encoder = keras.models.load_model(directory / 'best' / 'patch_encoder.keras', compile=False)
    attention = keras.models.load_model(directory / 'best' / 'attention.keras', compile=False)
    head = keras.models.load_model(directory / 'best' / 'survival_head.keras', compile=False)
    rows = []
    all_rates = []
    for j, pid in enumerate(patient_ids):
        paths = sorted(patient_patches.loc[patient_patches.patient_id == pid, 'path'])
        if not paths:
            raise ValueError(f'No actual RGB patches for {pid}')
        pool = StreamingAttention()
        for start in range(0, len(paths), chunk_size):
            x = np.stack([load_rgb(p) for p in paths[start:start + chunk_size]])
            o = np.repeat(maps[j:j + 1], len(x), axis=0)
            z = encoder([x, o], training=False)
            pool.update(z.numpy(), attention(z, training=False).numpy())
        rates = (tf.nn.softplus(head(pool.value()[None, :], training=False)) + 1e-06).numpy()[0]
        risk = float(rates @ exposures([spec['risk_horizon_years']], starts)[0])
        rows.append(dict(patient_id=pid, risk=risk, survival_probability_at_horizon=float(np.exp(-risk)), patch_count=len(paths), risk_horizon_years=spec['risk_horizon_years']))
        all_rates.append(rates)
    grid = np.asarray(time_grid_years if time_grid_years is not None else np.linspace(0, 5, 61))
    return (pd.DataFrame(rows), grid, survival_probabilities(np.stack(all_rates), grid, starts))

In [50]:
def summarize_folds(cfg):
    paths = sorted((Path(cfg['work_dir']) / 'folds').glob('fold_*/test_metrics.json'))
    if not paths:
        raise FileNotFoundError('No completed fold metrics.')
    rows = [json.loads(p.read_text()) for p in paths]
    vals = [r['c_index'] for r in rows if r['c_index'] is not None]
    result = dict(completed_folds=len(rows), requested_folds=cfg['training']['folds'], c_index_mean=float(np.mean(vals)) if vals else None, c_index_sd=float(np.std(vals, ddof=1)) if len(vals) > 1 else None, complete=len(rows) == cfg['training']['folds'], folds=rows, warning='Different cohorts/preprocessing/survival heads are not a controlled HEALNet comparison. Do not pool cross-fold raw risks as if one common model produced them.')
    atomic_json(Path(cfg['work_dir']) / 'cross_validation_summary.json', result)
    return result

### 9.2 Actual input-shape inspection helpers

In [51]:
def validate_pairing_tables(cohort, patches, blocks):
    required = {'patient_id', 'time_days', 'event'}
    if not required.issubset(cohort.columns):
        raise ValueError(f'Cohort is missing {required - set(cohort.columns)}')
    if cohort.empty or cohort.patient_id.duplicated().any():
        raise ValueError('The cohort must contain one nonempty, unique row per patient.')
    if not {'patient_id', 'path'}.issubset(patches.columns) or patches.empty:
        raise ValueError('No patch manifest with patient_id,path.')
    if patches.path.duplicated().any():
        raise ValueError('The same patch path appears twice; resolve the manifest rather than double-counting.')
    ids = set(cohort.patient_id)
    if set(patches.patient_id) != ids:
        raise ValueError('Every cohort patient must own patches, and every patch must belong to this cohort.')
    for kind in ORDER:
        if blocks[kind].index.duplicated().any():
            raise ValueError(f'Duplicate {kind} patient records.')
        if not ids <= set(blocks[kind].index):
            raise ValueError(f'Some patients lack {kind}.')
    counts = patches.groupby('patient_id').size().reindex(cohort.patient_id)
    if 'patch_count' in cohort and (not np.array_equal(counts.to_numpy(), cohort.patch_count.to_numpy())):
        raise ValueError('Cached cohort patch_count disagrees with the actual manifest.')
    return counts


class AlignedImageView:
    """Lazy, patient-ID-indexed view of REAL patches or their actual omics maps.

    `.shape` is the size of the on-disk indexed dataset, NOT an allocated NumPy
    array. `view[:8]` returns a real float32 NumPy array of eight input images.
    Materializing the entire cohort is deliberately disabled to protect memory.
    """

    def __init__(self, patch_frame, kind, patient_maps=None, patient_ids=None, max_rows_per_read=256):
        if kind not in ('wsi', 'omics'):
            raise ValueError('kind must be wsi or omics')
        if not {'patient_id', 'path'}.issubset(patch_frame.columns) or patch_frame.empty:
            raise ValueError('An actual, nonempty patch manifest is required.')
        if patch_frame.path.duplicated().any():
            raise ValueError('Duplicate paths would double-count patches.')
        self.frame = patch_frame.reset_index(drop=True).copy()
        self.kind = kind
        self.max_rows_per_read = int(max_rows_per_read)
        if self.max_rows_per_read < 1:
            raise ValueError('max_rows_per_read must be positive')
        self.shape = (len(self.frame), 128, 128, 3)
        self.dtype = np.dtype('float32')
        self.patient_maps = patient_maps
        self.map_row = None
        if kind == 'omics':
            if patient_maps is None or patient_ids is None:
                raise ValueError('Supply original patient_maps and their patient_ids.')
            patient_ids = list(patient_ids)
            if len(set(patient_ids)) != len(patient_ids):
                raise ValueError('Omics patient IDs must be unique.')
            if patient_maps.shape != (len(patient_ids), 128, 128, 3):
                raise ValueError('Patient map shape/ID count mismatch.')
            if not np.isfinite(patient_maps).all():
                raise ValueError('Patient maps contain nonfinite values.')
            lookup = pd.Series(np.arange(len(patient_ids)), index=patient_ids)
            index = self.frame.patient_id.map(lookup)
            if index.isna().any():
                raise ValueError('A WSI patch has no matching molecular patient record.')
            self.map_row = index.to_numpy(dtype=np.int64)

    def __len__(self):
        return self.shape[0]

    def __repr__(self):
        return f'AlignedImageView(kind={self.kind!r}, shape={self.shape}, storage="lazy; use [:8] for real NumPy input arrays")'

    def __array__(self, dtype=None, copy=None):
        raise MemoryError('Do not call np.asarray on the entire cohort. Read a small slice, e.g. view[:8].')

    def __getitem__(self, key):
        scalar = isinstance(key, (int, np.integer)) and (not isinstance(key, bool))
        if scalar:
            i = int(key)
            if i < 0:
                i += len(self)
            if not 0 <= i < len(self):
                raise IndexError(i)
            indices = np.asarray([i], dtype=np.int64)
        elif isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            number = len(range(start, stop, step))
            if number > self.max_rows_per_read:
                raise MemoryError('Requested slice is too large. Read a small batch.')
            indices = np.arange(start, stop, step, dtype=np.int64)
        else:
            indices = np.asarray(key)
            if indices.ndim != 1 or indices.dtype.kind not in 'iu':
                raise TypeError('Use one integer, a slice, or a one-dimensional integer index list.')
            indices = indices.astype(np.int64)
            indices = np.where(indices < 0, indices + len(self), indices)
            if np.any(indices < 0) or np.any(indices >= len(self)):
                raise IndexError('An index is outside the patch dataset.')
        if len(indices) > self.max_rows_per_read:
            raise MemoryError('Requested too many rows. Use several small batches.')
        if len(indices) == 0:
            return np.empty((0, 128, 128, 3), dtype=np.float32)
        if self.kind == 'wsi':
            result = np.stack([load_rgb(self.frame.iloc[int(i)].path) for i in indices])
        else:
            result = np.asarray(self.patient_maps[self.map_row[indices]], dtype=np.float32)
        return result[0] if scalar else result

In [52]:
def get_fold_inputs(cfg, fold, all_blocks=None):
    work = Path(cfg['work_dir'])
    cohort = pd.read_csv(work / 'cohort.csv')
    patches = pd.read_csv(work / 'patches.csv').sort_values(['patient_id', 'path']).reset_index(drop=True)
    blocks = all_blocks
    if blocks is None:
        blocks, _ = load_omics(cfg['omics_path'], cfg['feature_patterns'], cfg.get('id_column'), cfg.get('id_mapping_csv'))
    validate_pairing_tables(cohort, patches, blocks)
    splits = prepare_splits(cfg)
    part = splits[splits.fold == fold]
    if part.empty:
        raise ValueError(f'Fold {fold} does not exist.')
    ids = {s: sorted(part.loc[part.split == s, 'patient_id'].tolist()) for s in ('train', 'validation', 'test')}
    mapper = OmicsSpatializer(128, cfg['training']['max_features_per_modality'], cfg['training']['omics_interpolation']).fit(blocks, ids['train'])
    all_ids = sorted(cohort.patient_id.tolist())
    maps = mapper.transform(blocks, all_ids)
    wsi = AlignedImageView(patches, 'wsi')
    omics = AlignedImageView(patches, 'omics', maps, all_ids)
    return (cohort, patches, blocks, ids, mapper, all_ids, maps, wsi, omics)


def input_shape_report(cfg, fold, cohort, patches, blocks, ids, mapper, maps):
    counts = validate_pairing_tables(cohort, patches, blocks)
    total = int(len(patches))
    maximum = int(cfg['patch']['pool_per_patient'])
    rows = []
    for split, pids in ids.items():
        selected = cohort[cohort.patient_id.isin(pids)]
        n = len(pids)
        split_count = int(counts.reindex(pids).sum())
        k = cfg['training']['patches_per_patient_step']
        instances = sum(len(coverage_patch_indices(int(counts[pid]),k,0,cfg['training']['seed'],pid,cfg['training'].get('patch_sampling','random'))) for pid in pids)
        variants = 4 if cfg['training']['wsi_augmentation'] == 'four_views' or cfg['training']['omics_augmentation'] == 'legacy_four_views' else 1
        rows.append(dict(split=split, patients=n, deaths=int(selected.event.sum()), retained_wsi_patches=split_count, original_patient_omics_shape=str((n, 128, 128, 3)), aligned_wsi_shape=str((split_count, 128, 128, 3)), aligned_omics_shape=str((split_count, 128, 128, 3)), training_pair_presentations_epoch0=instances * variants if split == 'train' else None))
    report = dict(fold=int(fold), paired_patients=len(cohort), actual_total_patches=total, configured_patch_cap=maximum, configured_upper_bound=len(cohort) * maximum, every_patient_reached_cap=bool((counts == maximum).all()), patches_per_patient_min=int(counts.min()), patches_per_patient_median=float(counts.median()), patches_per_patient_max=int(counts.max()), source_feature_counts={m: int(blocks[m].shape[1]) for m in ORDER}, train_selected_feature_counts={m: len(mapper.state[m]['columns']) for m in ORDER}, original_patient_omics_shape=list(maps.shape), indexed_wsi_shape=[total, 128, 128, 3], indexed_aligned_omics_shape=[total, 128, 128, 3], allocated_entire_patch_dataset=False, independent_patients=len(cohort), note="All counts derive from actual manifests. N*p is exact only when all patients have p patches. Omics maps are fitted using this fold's training patients only. Augmented/repeated rows are not independent patients.", split_counts=rows)
    directory = Path(cfg['work_dir']) / 'input_inspection' / f'fold_{fold}'
    directory.mkdir(parents=True, exist_ok=True)
    atomic_json(directory / 'input_shapes.json', report)
    atomic_csv(pd.DataFrame(rows), directory / 'split_shapes.csv')
    atomic_csv(counts.rename('actual_patch_count').reset_index(), directory / 'per_patient_patch_counts.csv')
    return (report, pd.DataFrame(rows))


def inspect_training_batch(batch, cfg, folder=None):
    x_wsi, x_omics, segments, labels, group, owners = batch
    if not x_wsi.shape == x_omics.shape == (len(owners), 128, 128, 3):
        raise AssertionError('WSI/omics batch shapes do not match.')
    if segments.shape != (len(owners),) or labels.shape != (len(group), 2):
        raise AssertionError('Patch owners or patient outcome shapes are inconsistent.')
    if segments.dtype.kind not in 'iu' or np.any(segments < 0) or np.any(segments >= len(group)):
        raise AssertionError('Patch ownership indices must be nonnegative integers within the patient batch.')
    if len(set(group)) != len(group) or not np.isfinite(labels).all():
        raise AssertionError('Patient IDs must be unique and survival labels finite.')
    if np.any(labels[:, 0] <= 0) or not np.isin(labels[:, 1], [0, 1]).all():
        raise AssertionError('Survival labels require a positive duration and binary event indicator.')
    for row, owner in enumerate(owners):
        if owner != group[int(segments[row])]:
            raise AssertionError('Cross-patient pairing detected.')
    if not np.isfinite(x_wsi).all() or not np.isfinite(x_omics).all():
        raise FloatingPointError('Nonfinite input data.')
    preview = pd.DataFrame(dict(input_row=np.arange(len(owners)), wsi_patient_id=owners, omics_patient_id=[group[int(s)] for s in segments], segment_id=segments))
    print('ACTUAL NumPy arrays in one training step:')
    print('X_wsi_batch.shape =', x_wsi.shape, '| dtype =', x_wsi.dtype)
    print('X_omics_batch.shape =', x_omics.shape, '| dtype =', x_omics.dtype)
    print('patient_survival_y.shape =', labels.shape, '(one [time_days,event] row per patient)')
    print('patch_to_patient_index.shape =', segments.shape)
    print('Number of independent patients in this step =', len(group))
    print(preview.head(16).to_string(index=False))
    if len(preview)>16:
        print(f'... {len(preview)-16} further correct pairs (saved to CSV).')
    plan = large_bag_plan(len(owners),len(group),cfg.get('training',{}).get('encoder_microbatch_size',4))
    print('All these paired patches contribute to ONE patient-bag loss/update.')
    print('Encoder chunk shape:', tuple(plan['maximum_encoder_chunk_shape']))
    print('Chunks per pass:',plan['chunks_per_encoder_pass'], '| passes:',plan['encoder_passes_per_update'])
    report = dict(wsi_shape=list(x_wsi.shape), omics_shape=list(x_omics.shape), patient_labels_shape=list(labels.shape), independent_patients=len(group), all_patient_pairs_match=True, wsi_range=[float(x_wsi.min()), float(x_wsi.max())], omics_range=[float(x_omics.min()), float(x_omics.max())])
    report['large_bag_execution'] = plan
    if folder is not None:
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        atomic_json(folder / 'actual_batch_shapes.json', report)
        atomic_csv(preview, folder / 'actual_batch_patient_pairs.csv')
    return (report, preview)

In [53]:
def implementation_fingerprint():
    """Fingerprint Python bytecode without transient IPython filenames/line numbers.

    This also works when definitions live directly in notebook cells. Unknown
    closure/constant types are rejected, not silently replaced by a fake hash.
    """
    import types

    def constant(value):
        if isinstance(value, types.CodeType):
            return code(value)
        if isinstance(value, tuple):
            return [constant(v) for v in value]
        if isinstance(value, dict):
            return {str(k): constant(v) for k, v in sorted(value.items(), key=lambda z: repr(z[0]))}
        if isinstance(value, list):
            return [constant(v) for v in value]
        if isinstance(value, frozenset):
            return sorted([constant(v) for v in value], key=repr)
        if isinstance(value, bytes):
            return {'bytes': value.hex()}
        if value is Ellipsis:
            return {'ellipsis': True}
        if value is None or isinstance(value, (str, int, float, bool, complex)):
            return repr(value)
        raise TypeError(f'Unsupported code constant {type(value)}')

    def code(c):
        return dict(bytecode=c.co_code.hex(), names=c.co_names, constants=[constant(v) for v in c.co_consts], arguments=(c.co_argcount, c.co_posonlyargcount, c.co_kwonlyargcount), freevars=c.co_freevars, cellvars=c.co_cellvars, flags=c.co_flags, exceptions=getattr(c, 'co_exceptiontable', b'').hex())

    def function(f):
        return dict(code=code(f.__code__), defaults=constant(f.__defaults__), keyword_defaults=constant(f.__kwdefaults__))
    out = {}
    for name in PRODUCTION_DEFINITION_NAMES:
        item = globals().get(name)
        if item is None:
            raise RuntimeError(f'Run the definition cell for {name} before continuing.')
        if isinstance(item, type):
            methods = {}
            for key, member in item.__dict__.items():
                if isinstance(member, (staticmethod, classmethod)):
                    member = member.__func__
                if isinstance(member, property):
                    member = member.fget
                if isinstance(member, types.FunctionType):
                    methods[key] = function(member)
            out[name] = methods
        elif isinstance(item, types.FunctionType):
            out[name] = function(item)
        else:
            raise TypeError(f'Unexpected production definition type: {name}')
    return fingerprint({'code': out, 'molecular_order': ORDER, 'gdc_endpoint': GDC})

In [54]:
PRODUCTION_DEFINITION_NAMES = ['atomic_json', 'atomic_csv', 'sha256_file', 'fingerprint', 'stable_seed', 'patient_id', 'sample_type', 'numeric_time', 'require_files', 'SessionPause', 'check_deadline', 'tree_bytes', 'assert_storage', 'status', 'restore_saved_run', 'SlideGeometryError', 'read_table', '_id_column', 'load_omics', 'OmicsSpatializer', 'session', 'query_all', 'load_clinical_csv', 'validate_clinical', 'gdc_clinical', 'gdc_slides', 'locate_omics', 'download_slide', 'prepare_metadata', 'tissue_mask', '_mpp', 'read_patch', 'validate_config', '_science_contract', 'bind_run', '_metadata', '_source_key', '_cached_slide', 'prepare_streaming', 'require_complete_data', 'validate_folds', 'prepare_splits', 'four_view', 'load_rgb', 'PatientBatches', 'ChannelSlice', 'ChannelShuffle', 'MACFusion', 'LF', 'multi_kernel_groupwise_conv1', 'multi_kernel_groupwise_conv3', 'RGSA', 'EMRC', 'PCMFA', 'EHF1', 'SIR', 'EHF2', 'EHF', '_stem', 'build_patch_encoder', 'build_patient_heads', 'fit_intervals', 'exposures', 'pe_nll_numpy', 'survival_probabilities', 'harrell_c_index', 'c_index_bootstrap', 'StreamingAttention', 'initialize', 'segment_attention', 'pe_nll_tf', 'make_train_step', 'patient_predictions', '_safe_float', '_save_components', 'train_fold', 'predict_from_export', 'summarize_folds', 'validate_pairing_tables', 'AlignedImageView', 'get_fold_inputs', 'input_shape_report', 'inspect_training_batch', 'implementation_fingerprint', 'coverage_patch_indices', 'replay_state_variables', 'snapshot_replay_state', 'restore_replay_state', 'large_bag_plan', 'patch_cache_suffix', 'save_rgb_patch', 'make_large_bag_train_step', 'setup_clam_source', 'clam_preset_parameters', 'clam_read_geometry', 'enumerate_clam_coordinates', 'save_clam_coordinate_h5', 'clam_coordinate_cache', 'balanced_patch_quotas', 'select_patient_patch_records', 'prune_unselected_patient_images', 'save_selected_coordinate_previews', 'OmicsInputError', 'read_omics_frame', 'canonical_tcga_barcode', '_barcode_profile', 'inspect_omics_frame', '_choose_barcode_column', '_apply_authentic_id_map', 'orient_omics_frame', '_omics_candidate_paths', 'molecular_blocks_equal', 'prepare_omics_input']
PRODUCTION_DEFINITION_NAMES += ['cbio_meta', 'cbio_read_tsv', 'cbio_fetch_archive', 'cbio_extract_required', 'cbio_validate_study', 'cbio_sample_map', 'cbio_resolve_sample', 'cbio_gene_key', 'cbio_expression_matrix', 'cbio_sequenced_samples', 'cbio_mutation_matrix', 'cbio_survival', 'cbio_prepare_arrays', 'load_cbio_prepared_npz', 'read_cbio_prepared_frame', 'prepare_selected_molecular_source']


In [55]:
def run_visible_stage(name, function, *args, **kwargs):
    """Pause cleanly so Kaggle can save outputs, but never call a partial run complete."""
    global PIPELINE_ACTIVE
    if not PIPELINE_ACTIVE:
        print(f'{name}: skipped because this session has paused.')
        return None
    try:
        check_deadline(DEADLINE_UNIX)
        return function(*args, **kwargs)
    except SessionPause as error:
        PIPELINE_ACTIVE = False
        status(WORK_DIR, name, 'paused_resume_required', reason=str(error), full_run_complete=False)
        print('Save the output. Attach it to a new notebook; keep RUN_NAME and set RESUME_FROM.')
        return None
    except Exception as error:
        PIPELINE_ACTIVE = False
        atomic_json(WORK_DIR/'FAILURE.json', dict(stage=name, error=str(error), traceback=traceback.format_exc()))
        status(WORK_DIR, name, 'failed', reason=str(error), full_run_complete=False)
        raise

## 10. Acquire and validate the EXPLICIT molecular source before CLAM and slide downloads
In the default mode, download the publisher's TCGA-BRCA study. Validate the study/assay metadata,
join RNA/CNV/MAF at a shared **sample** first, use the published clinical sample-to-patient map,
and keep only unambiguous primary-tumor samples. No anonymous Kaggle row is relabelled.
`CBIOPORTAL_SOURCE_REPORT.json` records original matrix sizes, actual matches, exclusions,
transformations and source hashes. `OMICS_PREFLIGHT.json` records success/failure.

The source conversion preserves published expression values, discrete GISTIC calls and uses mutation-record indicators
on the RNA/CNV shared gene universe. A mutation zero means no record in the source MAF for an assayed
sample; it is not proof of per-gene wild type. All source mutation classifications are included.
This is a declared data representation, not a new measurement or biological pixel registration.
Imputation, variable-feature filtering and scaling still fit ONLY on each training fold.


In [56]:
# The editable settings in section 1 override this complete configuration.
CONFIG = {'work_dir': '/kaggle/working/cure_brca_clam1024_v5',
 'omics_path': '',
 'omics_source': 'cbioportal_brca',
 'id_column': None,
 'feature_patterns': {'rna': '^rs_', 'cnv': '^cn_', 'mutation': '^mu_'},
 'id_mapping_csv': None,
 'clinical_csv': None,
 'clinical_columns': {'patient': 'patient_id', 'time': 'time_days', 'event': 'event', 'time_unit': 'days'},
 'local_slides_manifest': None,
 'diagnostic_slides_only': True,
 'approve_slide_downloads': True,
 'max_download_gib': 1500.0,
 'download_token_env': 'GDC_TOKEN',
 'patch': {'pool_per_patient': 1024,
           'storage_format': 'jpeg',
           'jpeg_quality': 95,
           'input_size': 128,
           'field_size_um': 128.0,
           'unknown_mpp_override': None,
           'tissue_fraction_min': 0.65,
           'min_saturation': 0.08,
           'max_brightness': 0.92,
           'min_patches_per_patient': 8,
           'seed': 2026,
           'clam': {'requested_ref': 'master',
                    'short_bag_policy': 'exclude',
                    'segmentation_overrides': {},
                    'segmentation_downsample': 64.0,
                    'max_segmentation_pixels': 20000000,
                    'max_grid_checks': 2000000,
                    'source_info': {}}},
 'training': {'seed': 2026,
              'folds': 5,
              'validation_fraction': 0.2,
              'patient_batch_size': 2,
              'patches_per_patient_step': 1024,
              'encoder_microbatch_size': 4,
              'patch_sampling': 'coverage_cycle',
              'evaluation_chunk_size': 4,
              'epochs': 100,
              'patience': 15,
              'learning_rate': 0.0001,
              'gradient_clip_norm': 1.0,
              'survival_intervals': 4,
              'risk_horizon_years': 5.0,
              'bootstrap_replicates': 1000,
              'max_features_per_modality': None,
              'omics_interpolation': {'rna': 'linear', 'cnv': 'linear', 'mutation': 'nearest'},
              'wsi_augmentation': 'four_views',
              'omics_augmentation': 'none',
              'freeze_batch_norm': False,
              'resume': True,
              'deterministic_ops': True,
              'validation_patches_per_patient': 1024,
              'checkpoint_every_steps': 1,
              'log_every_steps': 1},
 'split_manifest': None,
 'kaggle': {'session_hours': 10.0,
            'max_output_gib': 16.0,
            'model_reserve_gib': 3.0,
            'scratch_dir': '/tmp/cure_brca_svs'}}
CONFIG['molecular_source_mode'] = MOLECULAR_SOURCE
CONFIG['cbioportal'] = dict(study_id=CBIO_STUDY, url=CBIO_ARCHIVE_URL,
    local_archive=CBIOPORTAL_LOCAL_ARCHIVE, local_dir=CBIOPORTAL_LOCAL_DIR,
    allow_download=ALLOW_MOLECULAR_DOWNLOADS, max_download_gib=MAX_MOLECULAR_DOWNLOAD_GIB,
    max_extracted_gib=MAX_MOLECULAR_EXTRACTED_GIB, duplicate_patient_policy='exclude',
    keep_raw_tables=False, duplicate_gene_policy=CBIO_DUPLICATE_GENE_POLICY)
if MOLECULAR_SOURCE == 'attached':
    CONFIG['omics_source'] = 'kaggle'


In [57]:
CONFIG.update(work_dir=str(WORK_DIR.resolve()), omics_path=OMICS_CSV,
              id_column=ID_COLUMN, id_mapping_csv=ID_MAPPING_CSV or None,
              clinical_csv=CLINICAL_CSV or None, local_slides_manifest=LOCAL_SVS_MANIFEST or None,
              split_manifest=EXTERNAL_SPLIT_CSV or None,
              approve_slide_downloads=ALLOW_GDC_DOWNLOADS, max_download_gib=MAX_TOTAL_DOWNLOAD_GIB)
CONFIG['patch'].update(pool_per_patient=PATCH_POOL_PER_PATIENT, unknown_mpp_override=UNKNOWN_MPP_OVERRIDE, storage_format=PATCH_STORAGE_FORMAT, jpeg_quality=JPEG_QUALITY)
CONFIG['patch']['clam'].update(requested_ref=CLAM_REF, short_bag_policy=SHORT_BAG_POLICY,
                              segmentation_overrides=CLAM_SEGMENTATION_OVERRIDES)
CONFIG['training'].update(patient_batch_size=PATIENT_BATCH_SIZE,
    patches_per_patient_step=PATCHES_PER_PATIENT_STEP, encoder_microbatch_size=ENCODER_MICROBATCH_SIZE, patch_sampling=PATCH_SAMPLING, validation_patches_per_patient=VALIDATION_PATCHES_PER_PATIENT,
    evaluation_chunk_size=EVALUATION_CHUNK_SIZE, epochs=EPOCHS, patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE, wsi_augmentation=WSI_AUGMENTATION, omics_augmentation=OMICS_AUGMENTATION)
CONFIG['kaggle'].update(session_hours=SESSION_HOURS, max_output_gib=MAX_SAVED_OUTPUT_GIB,
                       scratch_dir=str(Path(tempfile.gettempdir())/RUN_NAME/'svs_scratch'))

if RESUME_FROM:
    restore_saved_run(RESUME_FROM, WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
# Validate the ACTUAL molecular source before cloning CLAM or downloading slides.
_omics_config = run_visible_stage('omics_preflight', prepare_selected_molecular_source, CONFIG, deadline_unix=DEADLINE_UNIX)
if PIPELINE_ACTIVE:
    CONFIG = _omics_config
    CLAM_SOURCE_INFO = setup_clam_source(WORK_DIR, CLAM_REF, CLAM_SOURCE_DIR)
    CONFIG['patch']['clam']['source_info'] = CLAM_SOURCE_INFO
    CONFIG = bind_run(CONFIG)
    raw_omics = read_omics_frame(CONFIG['omics_path'])
    omics_blocks, omics_audit = load_omics(CONFIG['omics_path'], CONFIG['feature_patterns'],
                                         CONFIG.get('id_column'), CONFIG.get('id_mapping_csv'))
    print('Source-prepared molecular table .shape =', raw_omics.shape)
    print('Identified patient-ID source:', omics_audit['id_column'])
    print('Validated input orientation:', omics_audit['identity']['layout'])
    for molecular_type in ORDER:
        print(f"{molecular_type}_original_matrix.shape = {omics_blocks[molecular_type].shape}")
    print('Columns are molecular measurements; rows are original patients.')
    display(raw_omics.iloc[:5, :8])
    display(pd.DataFrame([dict(modality=k,patients=v.shape[0],original_features=v.shape[1])
                          for k,v in omics_blocks.items()]))
    atomic_json(WORK_DIR/'source_omics_audit.json', omics_audit)

    gene_qc_summary = WORK_DIR/'sources'/'cbioportal_brca'/'source_gene_qc_summary.csv'
    if gene_qc_summary.is_file():
        print('Audited duplicate-gene handling (not a patient-count increase):')
        display(pd.read_csv(gene_qc_summary))
    status(WORK_DIR,'omics_preflight','passed',full_run_complete=False)
    del raw_omics  # only the numeric modality blocks are needed downstream


EXPLICIT SOURCE CHANGE: TCGA-BRCA PanCancer Atlas via cBioPortal; attached Kaggle CSVs are NOT used.
This may change the cohort, features, and endpoint release. It does NOT recover missing Kaggle row IDs.
[rna gene QC] 20531 source rows -> 20481 unique retained keys; 0 identical extra rows collapsed; 24 conflicting keys found; policy=exclude_conflicting
[cnv gene QC] 25128 source rows -> 24993 unique retained keys; 16 identical extra rows collapsed; 57 conflicting keys found; policy=exclude_conflicting
Publisher-identified molecular matrices: {'rna': (1050, 20242), 'cnv': (1050, 20242), 'mutation': (1050, 20242)}


From https://github.com/mahmoodlab/CLAM
 * branch            53e2409d4a8189c682c173382964a85f114f923c -> FETCH_HEAD
HEAD is now at 53e2409 Update README.md


CLAM revision: 53e2409d4a8189c682c173382964a85f114f923c | source fingerprint: 1276e9149091a6592daca951adb201f627151e4f8c53a6c2929b6de37843226f
Actual molecular-table audit: {
  "unique_patients": 1050,
  "features": {
    "rna": 20242,
    "cnv": 20242,
    "mutation": 20242
  },
  "id_column": "patient_id",
  "channel_order": [
    "rna",
    "cnv",
    "mutation"
  ]
}
Source-prepared molecular table .shape = (1050, 60727)
Identified patient-ID source: patient_id
Validated input orientation: publisher_sample_matched_numeric_arrays
rna_original_matrix.shape = (1050, 20242)
cnv_original_matrix.shape = (1050, 20242)
mutation_original_matrix.shape = (1050, 20242)
Columns are molecular measurements; rows are original patients.


,patient_id,rs_ENTREZ_1,rs_ENTREZ_10,rs_ENTREZ_100,rs_ENTREZ_1000,rs_ENTREZ_10000,rs_ENTREZ_100008586,rs_ENTREZ_100009676
0,TCGA-3C-AAAU,197.089996,10.684400,110.635002,79.960800,205.072006,0.0000,85.437500
1,TCGA-3C-AALI,237.384003,85.372498,152.800003,88.635101,220.772003,2.4089,148.563995
2,TCGA-3C-AALJ,423.237000,0.906600,270.171997,108.793999,266.545990,0.0000,92.003601
3,TCGA-3C-AALK,191.018005,9.102200,113.363998,8330.160156,503.516998,0.0000,66.028099
4,TCGA-4H-AAAK,268.881012,0.851100,111.915001,133.617004,1274.469971,0.0000,68.595703


,modality,patients,original_features
0,rna,1050,20242
1,cnv,1050,20242
2,mutation,1050,20242


Audited duplicate-gene handling (not a patient-count increase):


,modality,policy,source_rows,unique_retained_gene_keys,identical_extra_rows_collapsed,conflicting_keys_excluded,primary_sample_columns,shared_gene_keys
0,rna,exclude_conflicting,20531,20481,0,24,1082,20242
1,cnv,exclude_conflicting,25128,24993,16,57,1070,20242


[omics_preflight] passed


## 11. Match WSI, omics and survival by patient ID
Both row IDs and outcome validity are checked before any large slide download.
The same patient can have several slides. Slides are not counted as additional independent patients.

In [58]:
metadata_result = run_visible_stage('metadata', _metadata, CONFIG)
if PIPELINE_ACTIVE:
    cohort_before_qc, slide_manifest = metadata_result
    print('cohort_before_patch_qc.shape =', cohort_before_qc.shape)
    print('slide_manifest.shape =', slide_manifest.shape)
    print('Paired patients before patch QC =', cohort_before_qc.patient_id.nunique())
    print('Eligible raw WSI files =', len(slide_manifest))
    if 'file_size' in slide_manifest:
        print('Total requested raw-slide transfer (GiB) =', round(slide_manifest.file_size.sum()/2**30,2))
    display(cohort_before_qc.head())
    display(slide_manifest.head())

Metadata match: 1001 patients, 1070 slides; 142 observed deaths. This is not yet the post-QC cohort.
cohort_before_patch_qc.shape = (1001, 3)
slide_manifest.shape = (1070, 6)
Paired patients before patch QC = 1001
Eligible raw WSI files = 1070
Total requested raw-slide transfer (GiB) = 1026.23


,patient_id,time_days,event
0,TCGA-3C-AALI,4007.699230,0
1,TCGA-3C-AALJ,1474.993425,0
2,TCGA-3C-AALK,1448.975902,0
3,TCGA-4H-AAAK,348.234540,0
4,TCGA-5L-AAT0,1477.995447,0


,patient_id,file_id,file_name,file_size,md5sum,path
0,TCGA-3C-AALI,6a7477b1-86f3-473f-9bf1-174a758142e9,TCGA-3C-AALI-01Z-00-DX2.CF4496E0-AB52-4F3E-BDF...,2079652388,e4b36bfd3f9b2c8d8692c0ce1c5f2870,/kaggle/working/cure_brca_cbio_clam1024_v7_1/s...
1,TCGA-3C-AALI,d46167af-6c29-49c7-95cf-3a801181aca4,TCGA-3C-AALI-01Z-00-DX1.F6E9A5DF-D8FB-45CF-B4B...,1706305510,e2ae6d543180fb8004c30a8f13fde42d,/kaggle/working/cure_brca_cbio_clam1024_v7_1/s...
2,TCGA-3C-AALJ,5d4c7a38-a574-4157-88a2-bd55f39e1599,TCGA-3C-AALJ-01Z-00-DX2.62DFE56B-B84C-40F9-962...,1839955836,6cfb23a0d80dd31b3a4b1a203bb8b1fd,/kaggle/working/cure_brca_cbio_clam1024_v7_1/s...
3,TCGA-3C-AALJ,68458986-1efd-4fce-9e65-ce5430ab125c,TCGA-3C-AALJ-01Z-00-DX1.777C0957-255A-42F0-9EE...,1872890822,2931aab977fbe391c1a2ee70235431f9,/kaggle/working/cure_brca_cbio_clam1024_v7_1/s...
4,TCGA-3C-AALK,93b26333-5723-4fa4-a4de-6124c04ab243,TCGA-3C-AALK-01Z-00-DX1.4E6EB156-BB19-410F-878...,1769848096,3d63b3311612d763525b6edb0848b986,/kaggle/working/cure_brca_cbio_clam1024_v7_1/s...


## 12. Select 1,024 actual tissue patches per retained patient using CLAM

`clam_patient_selection.csv` reports candidate coordinates, decoded regions, selected patches and exclusions.
An exact 1,024 total is enforced by the default short-bag policy, not by duplicating patches.
The operation visits all metadata-matched patients before any training begins.


In [ ]:
if PIPELINE_ACTIVE and not (WORK_DIR/'DATA_COMPLETE.json').exists():
    run_visible_stage('clam_patch_selection', prepare_streaming, CONFIG, deadline_unix=DEADLINE_UNIX)
if PIPELINE_ACTIVE:
    run_visible_stage('patch_integrity', require_complete_data, WORK_DIR, deadline_unix=DEADLINE_UNIX)
if PIPELINE_ACTIVE:
    cohort = pd.read_csv(WORK_DIR/'cohort.csv')
    patch_table = pd.read_csv(WORK_DIR/'patches.csv')
    patch_counts = validate_pairing_tables(cohort, patch_table, omics_blocks)
    N = int(len(cohort))
    N_PATCHES = int(len(patch_table))
    print('N = actual paired patients after QC =', N)
    print('N_PATCHES = sum of actual patches over patients =', N_PATCHES)
    print('N * configured patch cap =', N*PATCH_POOL_PER_PATIENT, '(an upper bound, not necessarily actual count)')
    print('patch_table.shape =', patch_table.shape)
    print('Patches/patient: min =', patch_counts.min(), ', median =', patch_counts.median(), ', max =', patch_counts.max())
    display(patch_counts.rename('actual_patches').reset_index().head(20))
    print("CLAM selection summary (measured, not assumed):")
    display(pd.read_csv(WORK_DIR/'clam_patient_selection.csv').head(20))
    if SHORT_BAG_POLICY == "exclude":
        assert (patch_counts == PATCH_POOL_PER_PATIENT).all()
        assert N_PATCHES == N * PATCH_POOL_PER_PATIENT
        print(f"Exact retained cohort: {N} patients × {PATCH_POOL_PER_PATIENT} = {N_PATCHES} patches")


Patient 1/1001 TCGA-3C-AALI: selected=1024; total selected RGB images=1024
[clam_preparation] running
Patient 2/1001 TCGA-3C-AALJ: selected=1024; total selected RGB images=2048
[clam_preparation] running
Patient 3/1001 TCGA-3C-AALK: selected=1024; total selected RGB images=3072
[clam_preparation] running
Patient 4/1001 TCGA-4H-AAAK: selected=1024; total selected RGB images=4096
[clam_preparation] running
Patient 5/1001 TCGA-5L-AAT0: selected=0; total selected RGB images=4096
[clam_preparation] running
Patient 6/1001 TCGA-5T-A9QA: selected=1024; total selected RGB images=5120
[clam_preparation] running
Patient 7/1001 TCGA-A1-A0SB: selected=1024; total selected RGB images=6144
[clam_preparation] running
Patient 8/1001 TCGA-A1-A0SD: selected=1024; total selected RGB images=7168
[clam_preparation] running
Patient 9/1001 TCGA-A1-A0SE: selected=1024; total selected RGB images=8192
[clam_preparation] running
Patient 10/1001 TCGA-A1-A0SF: selected=1024; total selected RGB images=9216
[clam_pre

### 12.1 Inspect real CLAM tissue contours and selected patch locations

In [ ]:
if PIPELINE_ACTIVE and SHOW_REAL_INPUT_IMAGES:
    # Actual masks and selected coordinates, not generated tissue illustrations.
    first_pid = cohort.patient_id.iloc[0]
    slide_dirs = sorted(p for p in (WORK_DIR/'patches'/first_pid).iterdir()
                        if p.is_dir() and (p/'coordinate_metadata.json').exists())
    for folder in slide_dirs[:2]:
        meta = json.loads((folder/'coordinate_metadata.json').read_text())
        print("Patient", first_pid, "| slide", folder.name)
        print("Original WSI (width,height):", (meta['width'],meta['height']))
        print("CLAM candidate coordinates:", meta['eligible_coordinates'])
        print("Saved patch read geometry:", meta['geometry'])
        display(Image.open(folder/'tissue_contours.png'))
        display(Image.open(folder/'selected_locations.jpg'))


## 13. Create patient-level folds, then spatialize the omics
The omics maps below use **training-fold-only** preprocessing for `INSPECT_FOLD`.
The same fitted transformation is applied to its validation and test patients.
The training loop independently repeats this step for each fold. There is no preprocessing fitted to the entire cohort.

In [ ]:
if PIPELINE_ACTIVE:
    inspected = run_visible_stage('fold_inputs', get_fold_inputs, CONFIG, INSPECT_FOLD, omics_blocks)
if PIPELINE_ACTIVE:
    (cohort, patch_table, omics_blocks, fold_ids, omics_mapper, all_patient_ids,
     X_omics_patients, X_wsi_all, X_omics_aligned) = inspected
    actual_report, split_shape_table = input_shape_report(
        CONFIG, INSPECT_FOLD, cohort, patch_table, omics_blocks, fold_ids, omics_mapper, X_omics_patients)
    print('X_omics_patients.shape =', X_omics_patients.shape, '(REAL NumPy array; one map per patient)')
    print('X_wsi_all.shape =', X_wsi_all.shape, '(LAZY indexed dataset, stored as actual selected RGB patch files)')
    print('X_omics_aligned.shape =', X_omics_aligned.shape, '(LAZY patient-ID-aligned view)')
    print('The two large indexed shapes match. Only requested slices are loaded into RAM.')
    display(split_shape_table)
    display(pd.DataFrame([dict(modality=k,original_features=omics_blocks[k].shape[1],
                              retained_training_features=len(omics_mapper.state[k]['columns'])) for k in ORDER]))
    # Show ONE patient's bag separately from the complete dataset.
    sample_patient = fold_ids['train'][0]
    sample_rows = patch_table.loc[patch_table.patient_id == sample_patient].copy()
    X_wsi_one_patient = AlignedImageView(sample_rows, 'wsi')
    X_omics_one_patient = AlignedImageView(sample_rows, 'omics', X_omics_patients, all_patient_ids)
    print("Patient:", sample_patient)
    print("ONE patient WSI selected-patch shape:", X_wsi_one_patient.shape)
    print("ONE patient aligned-omics shape:", X_omics_one_patient.shape)
    print("ONE patient's original omics-map shape:", X_omics_patients[all_patient_ids.index(sample_patient)].shape)
    print("Actual loaded four-patch slice:", X_wsi_one_patient[:4].shape)


## 14. Load REAL images and prove the matching with `.shape`
`X_wsi_all` and `X_omics_aligned` are disk-backed/indexed views, not fabricated arrays or allocations of the whole cohort.
Indexing them loads actual NumPy inputs. The first example uses two training patients and shows their IDs row-by-row.
The omics values repeat within each patient's patches; another patient's map is selected only when the patient ID changes.

In [ ]:
if PIPELINE_ACTIVE:
    example_ids = fold_ids['train'][:2]
    example_rows = []
    for pid in example_ids:
        example_rows.extend(patch_table.index[patch_table.patient_id==pid][:4].tolist())
    X_wsi_example = X_wsi_all[example_rows]
    X_omics_example = X_omics_aligned[example_rows]
    print('type(X_wsi_example) =', type(X_wsi_example))
    print('X_wsi_example.shape =', X_wsi_example.shape)
    print('X_omics_example.shape =', X_omics_example.shape)
    ownership_table = patch_table.loc[example_rows,['patient_id','path']].reset_index(drop=True)
    ownership_table = ownership_table.rename(columns={'patient_id':'wsi_patient_id'})
    ownership_table['omics_patient_id'] = [all_patient_ids[X_omics_aligned.map_row[i]] for i in example_rows]
    ownership_table['correct_pair'] = ownership_table.wsi_patient_id == ownership_table.omics_patient_id
    assert ownership_table.correct_pair.all()
    display(ownership_table)
    for pid in example_ids:
        rows = np.flatnonzero(ownership_table.wsi_patient_id.to_numpy()==pid)
        for j in rows:
            np.testing.assert_array_equal(X_omics_example[rows[0]], X_omics_example[j])
    print('PASS: every unaugmented omics row equals the map of its owning patient.')
    inspection_dir = WORK_DIR/'input_inspection'/f'fold_{INSPECT_FOLD}'
    atomic_csv(ownership_table, inspection_dir/'example_patient_pairs.csv')

### 14.1 Actual WSI patch and separate molecular channels from one training patient

In [ ]:
if PIPELINE_ACTIVE and SHOW_REAL_INPUT_IMAGES:
    import matplotlib.pyplot as plt
    pid = ownership_table.iloc[0].wsi_patient_id
    plt.figure(figsize=(4,4))
    plt.imshow(X_wsi_example[0])
    plt.title(f'Actual WSI tissue patch\n{pid}')
    plt.axis('off')
    plt.show()
    for channel, label in enumerate(('RNA resampled map', 'CNV resampled map', 'Mutation resampled map')):
        plt.figure(figsize=(4,4))
        plt.imshow(X_omics_example[0,:,:,channel])
        plt.title(f'{label} — not a tissue image\n{pid}')
        plt.colorbar(label='Standardized molecular representation')
        plt.axis('off')
        plt.show()

## 15. Inspect all 1,024 patches per patient and the encoder chunk
With the default settings, two retained patients contribute 2 × 1,024 = 2,048 matched image pairs.
Both inputs have shape `(2048,128,128,3)`. The encoder processes at most four pairs at a time;
the patient-level aggregation and loss still use all 1,024 selected patches per patient.
Actual printed shapes take precedence for a smaller final minibatch or an explicit short-bag policy.


In [ ]:
if PIPELINE_ACTIVE:
    train_ids = fold_ids['train']
    train_patient_maps = omics_mapper.transform(omics_blocks, train_ids)
    train_loader = PatientBatches(cohort, patch_table, train_patient_maps, train_ids, CONFIG['training'])
    actual_batch = next(train_loader.iter_epoch(epoch=0))
    X_wsi_batch, X_omics_batch, patch_to_patient, survival_y, batch_patients, patch_owners = actual_batch
    batch_report, batch_pair_table = inspect_training_batch(actual_batch, CONFIG, inspection_dir)
    print('Original patient maps stored in this training split:', train_patient_maps.shape)
    print('Repeated/aligned batch maps actually used by CURE:', X_omics_batch.shape)
    print('Every patch owner is a patient in the training split:', set(patch_owners)<=set(train_ids))

## 16. Verify the full CURE graph on actual data before the long run
This cell does an inference pass, prints intermediate shapes and optionally performs one gradient update
using **actual training patients only**, then tests `.keras` reloads. It is a disposable runtime check;
its temporary weights are deleted, and each training fold starts/resumes its own model afterwards.
A passed runtime check is not a convergence result or a claim of higher C-index.
This check now replays all selected patch chunks during its gradient update. It is still a runtime check,
not trained survival performance; normal training reinitializes a fresh fold model afterward.

In [ ]:
if PIPELINE_ACTIVE:
    def check_actual_model():
        initialize(CONFIG['training']['seed'], CONFIG['training']['deterministic_ops'])
        training_labels = cohort.set_index('patient_id').loc[fold_ids['train']]
        starts = fit_intervals(training_labels.time_days, training_labels.event,
                               CONFIG['training']['survival_intervals'])
        encoder = build_patch_encoder(CONFIG['training']['freeze_batch_norm'])
        attention, survival_head = build_patient_heads(encoder.output_shape[-1], len(starts))
        print('CURE external input shapes:', encoder.input_shape)
        print('CURE fused spatial-map shape:', encoder.get_layer('cure_final_spatial_fusion').output.shape)
        print('CURE parameter count:', encoder.count_params())
        if PRINT_FULL_MODEL_SUMMARY:
            encoder.summary()
        chunk = CONFIG['training']['encoder_microbatch_size']
        z = tf.concat([encoder([X_wsi_batch[s:s+chunk],X_omics_batch[s:s+chunk]],training=False)
                       for s in range(0,len(X_wsi_batch),chunk)],axis=0)
        patient_z = segment_attention(z, attention(z,training=False), patch_to_patient, len(batch_patients))
        logits = survival_head(patient_z, training=False)
        print('Actual fused patch embeddings .shape =', z.shape)
        print('Actual aggregated patient embeddings .shape =', patient_z.shape)
        print('Actual patient survival-rate outputs .shape =', logits.shape)
        tf.debugging.assert_all_finite(logits, 'Nonfinite forward result')
        report = dict(source='actual training patients/patches; no synthetic replacement',
                      input_wsi=list(X_wsi_batch.shape),input_omics=list(X_omics_batch.shape),
                      patch_embeddings=list(z.shape),patient_embeddings=list(patient_z.shape),
                      patient_outputs=list(logits.shape),gpu_devices=[d.name for d in physical_gpus],
                      gradient_update=False,checkpoint_reload=False)
        if RUN_REAL_DATA_GRADIENT_CHECK:
            optimizer = keras.optimizers.Adam(LEARNING_RATE)
            step = make_large_bag_train_step(encoder,attention,survival_head,optimizer,starts,
                microbatch_size=chunk,deadline_unix=DEADLINE_UNIX)
            loss, norm = step(X_wsi_batch,X_omics_batch,patch_to_patient,survival_y)
            report.update(gradient_update=True,training_nll=float(loss.numpy()),gradient_norm=float(norm.numpy()))
            print('Real-data check NLL:', report['training_nll'], '| preclip gradient norm:', report['gradient_norm'])
            report['large_bag_execution'] = step.last_report
            del step, optimizer
        with tempfile.TemporaryDirectory(prefix='cure_actual_reload_') as temporary:
            for name, component, component_input in (
                ('encoder',encoder,[X_wsi_batch[:chunk],X_omics_batch[:chunk]]),
                ('attention',attention,z), ('survival_head',survival_head,patient_z)):
                file = Path(temporary)/f'{name}.keras'
                before = component(component_input,training=False).numpy()
                component.save(file)
                restored = keras.models.load_model(file,compile=False)
                after = restored(component_input,training=False).numpy()
                np.testing.assert_allclose(before,after,rtol=2e-5,atol=2e-5)
                del restored
        report['checkpoint_reload'] = True
        atomic_json(inspection_dir/'actual_data_runtime_check.json',report)
        del encoder, attention, survival_head, z, patient_z, logits
        gc.collect(); keras.backend.clear_session()
        print('Actual-data forward/reload check passed. This is not a trained survival result.')
        return report
    model_check = run_visible_stage('actual_data_model_check', check_actual_model)

## 17. Full patient-level training and held-out evaluation
The complete training loop is defined in section 9.1 and called here on each configured fold.
Checkpoint selection uses validation NLL. A pause saves completed updates and resumes with identical
scientific settings. It never changes the number of patients or patch budget to make the session finish.

Depending on download size and training time, several saved Kaggle sessions may be needed. Only
`RUN_STATUS.json` with `status="complete"` means all five folds finished.

In [ ]:
if PIPELINE_ACTIVE and RUN_TRAINING:
    requested_folds = list(range(CONFIG['training']['folds'])) if FOLDS_TO_RUN is None else list(FOLDS_TO_RUN)
    if len(requested_folds)!=len(set(requested_folds)) or any(f not in range(CONFIG['training']['folds']) for f in requested_folds):
        raise ValueError('Requested fold numbers must be unique and in range.')
    # Free inspection copies; the trainer constructs its own split-safe maps/loaders.
    for temporary_name in ('X_wsi_example','X_omics_example','X_wsi_batch','X_omics_batch','actual_batch',
                           'train_loader','train_patient_maps','X_omics_patients','X_omics_aligned','inspected'):
        globals().pop(temporary_name,None)
    gc.collect()
    for fold in requested_folds:
        if not PIPELINE_ACTIVE:
            break
        status(WORK_DIR,'training','running',fold=fold,full_run_complete=False)
        fold_result = run_visible_stage(f'train_fold_{fold}', train_fold, CONFIG, fold, deadline_unix=DEADLINE_UNIX)
        if fold_result is not None:
            print(json.dumps(fold_result,indent=2))
    if PIPELINE_ACTIVE:
        summary = summarize_folds(CONFIG)
        status(WORK_DIR,'evaluation','complete' if summary['complete'] else 'selected_folds_complete',
               completed_folds=summary['completed_folds'],full_run_complete=summary['complete'])
elif PIPELINE_ACTIVE:
    status(WORK_DIR,'preparation','prepared_not_trained',full_run_complete=False)
    print('RUN_TRAINING=False: data were prepared and inspected; survival training was not requested.')

## 18. Inspect saved shape reports, training curves and actual predictions
Results are read from files produced by this run, never from hardcoded example C-index values.
A plotted survival curve is a model prediction for a held-out patient, not an observed survival trajectory.

In [ ]:
print('Saved output directory:', WORK_DIR)
if (WORK_DIR/'RUN_STATUS.json').exists():
    print((WORK_DIR/'RUN_STATUS.json').read_text())
shape_path = WORK_DIR/'input_inspection'/f'fold_{INSPECT_FOLD}'/'input_shapes.json'
if shape_path.exists():
    print('ACTUAL saved input-count report:')
    print(shape_path.read_text())
metric_paths = sorted((WORK_DIR/'folds').glob('fold_*/test_metrics.json'))
if metric_paths:
    rows = [json.loads(p.read_text()) for p in metric_paths]
    display(pd.DataFrame([{k:r.get(k) for k in ('fold','patients','events','c_index','patient_nll','selected_epoch')} for r in rows]))
else:
    print('No fold has completed held-out evaluation. No test performance is available.')

for f in range(CONFIG['training']['folds']):
    folder = WORK_DIR/'folds'/f'fold_{f}'
    if (folder/'history.csv').exists():
        import matplotlib.pyplot as plt
        history = pd.read_csv(folder/'history.csv')
        display(history.tail())
        plt.figure(figsize=(7,4))
        plt.plot(history.epoch,history.train_nll,label='Training NLL')
        plt.plot(history.epoch,history.validation_nll,label='Validation NLL')
        plt.xlabel('Epoch'); plt.ylabel('Patient-level negative log-likelihood')
        plt.title(f'Fold {f}: actual training history'); plt.legend(); plt.show()
    if (folder/'test_predictions.csv').exists():
        display(pd.read_csv(folder/'test_predictions.csv').head())
    if (folder/'test_survival_curves.npz').exists():
        saved = np.load(folder/'test_survival_curves.npz',allow_pickle=False)
        plt.figure(figsize=(7,4))
        for row in range(min(3,len(saved['patient_id']))):
            plt.plot(saved['time_years'],saved['survival_probability'][row],label=str(saved['patient_id'][row]))
        plt.xlabel('Years from the recorded survival origin'); plt.ylabel('Predicted survival probability')
        plt.ylim(0,1); plt.title(f'Fold {f}: held-out predictions'); plt.legend(); plt.show()